# Network + GPU round-trip spike

Answers one question: once a Java client talks to MatrixDB's GPU engine over a real TCP socket, does the ~12-19x in-process GPU scan win (see FINDINGS.md 3.5) survive the network + serialization hop, or does that hop eat it alive? See docs/superpowers/specs/2026-06-30-network-gpu-spike-design.md for the full design and the PASS/FAIL decision rule.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

## 1. Confirm a GPU is attached

In [ ]:
!nvcc --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Write the source files

In [ ]:
%%writefile types.hpp
#pragma once

#include <cstdint>
#include <cstddef> // ponytail: spec wrote <csize_t> (does not exist); size_t lives in <cstddef>

#if defined(__ARM_ARCH) || defined(__apple_build_version__)
    // Apple Silicon L1 cache lines are typically 128 bytes
    constexpr size_t MATRIX_CACHE_LINE = 128;
#else
    // Standard Intel/AMD x86_64 cache lines are 64 bytes
    constexpr size_t MATRIX_CACHE_LINE = 64;
#endif

/**
 * @brief Represents a single transaction query record.
 * Designed with descending structural member alignment to prevent internal compiler padding.
 * Aligned to target CPU cache line boundaries to eliminate false sharing.
 */
struct alignas(MATRIX_CACHE_LINE) DatabaseQuery {
    uint64_t query_id;
    uint64_t timestamp_us;
    uint64_t transaction_id;
    uint32_t opcode;
    uint32_t payload_size;
    uint8_t payload[32];

    static constexpr size_t data_footprint =
        sizeof(uint64_t) * 3 +
        sizeof(uint32_t) * 2 +
        sizeof(uint8_t) * 32;

    static constexpr size_t padding_needed =
        (MATRIX_CACHE_LINE > data_footprint) ? (MATRIX_CACHE_LINE - data_footprint) : 0;

    // Explicit structural padding to align with target L1 cache line size
    uint8_t padding[padding_needed > 0 ? padding_needed : 1];
};

static_assert((sizeof(DatabaseQuery) % MATRIX_CACHE_LINE) == 0,
    "DatabaseQuery structure violates cache-line alignment constraints.");

// Operational instruction carried in DatabaseQuery::opcode (spec: Read, Write, Scan).
enum MatrixOpcode : uint32_t {
    OP_READ  = 1,
    OP_WRITE = 2,
    OP_SCAN  = 3,
    OP_TXN_WRITE = 4,   // a transactional put: buffered on WAL replay until a commit marker
};

// Aggregate reduction carried by OP_SCAN (payload offset 16). AGG_COUNT==0 is the default, so a
// scan with no agg op set counts matches (the original behavior). SUM/MIN/MAX reduce the values
// matching the predicate (value > threshold).
enum MatrixAggOp : uint32_t {
    AGG_COUNT = 0,
    AGG_SUM   = 1,
    AGG_MIN   = 2,
    AGG_MAX   = 3,
};

// Columnar store + page-ownership layout (shard-per-thread).
// The keyspace (STORE_SLOTS) is split into PAGE_COUNT contiguous pages. Exactly one
// owner (one GPU thread) reads/writes a page's slots, so the same key always routes
// to the same owner: writes to a key serialize by ownership. No cross-thread conflict
// => no store atomics, no OCC, no delta log. Each page has a single owner; pages are
// independent of one another (shared-nothing).
//
// DM-1b fix (2026-07-01): this used to be 4096 while KVStore's own point-op capacity
// (compute_mock.cpp's `KVStore kv_{...}`) was independently hardcoded to 65536 (== BATCH_MAX) --
// two different capacities for what's supposed to be "the same" point-op store across backends.
// GPU's flat, direct-mapped store (compute_cuda.cuh's matrix_page_kernel: `store[key & MASK]`, no
// probing) silently overwrote on any two keys sharing a slot -- guaranteed for the standard
// benchmark workload (BATCH_MAX=65536 unique keys into only 4096 slots). Matching this to BATCH_MAX
// (and KVStore's capacity, which now reads MATRIX_STORE_SLOTS instead of its own literal) makes the
// two backends' capacities the same symbol, permanently, and gives GPU enough slots that the
// standard sequential-key workload (query_id 0..BATCH_MAX-1) is a perfect bijection -- zero
// collisions by construction, not by luck. See PRODUCTION_READINESS.md DM-1b and
// test_gpu_pointop_collision.cu for the hardware-verified proof. This does NOT make the GPU store a
// general-purpose hash table: a batch with more than STORE_SLOTS truly-distinct keys, or an
// adversarial (non-sequential) key distribution, can still collide -- narrower and far less likely
// than the guaranteed-collision case this fix closes, and unlike a hash table's graceful probing,
// still resolves via silent last-writer-wins if it happens. Not solved further: this kernel is
// confirmed unreachable from anything exercised in this repo (see the DM-1b note), so going further
// than "make the actual, demonstrated case correct" was judged disproportionate.
constexpr size_t MATRIX_STORE_SLOTS = 65536;                      // power of two; == BATCH_MAX, == KVStore's capacity
constexpr size_t MATRIX_STORE_MASK  = MATRIX_STORE_SLOTS - 1;
constexpr size_t MATRIX_PAGE_COUNT  = 1024;                       // owner threads; the tuning knob
constexpr size_t MATRIX_PAGE_SIZE   = MATRIX_STORE_SLOTS / MATRIX_PAGE_COUNT; // slots per page
constexpr size_t MATRIX_BATCH_MAX   = 65536;                      // sweep ceiling / scratch buffer size

// Resident analytical column (the GPU-DB's actual data): a uint32 column that lives
// in VRAM/RAM and is scanned in place by OP_SCAN — never shipped per query. 16M values
// = 64MB, well past CPU cache so the GPU bandwidth advantage is real. Filled value[i]=i
// so a threshold T yields a known count (SCAN_SIZE-1-T) for oracle verification.
constexpr size_t MATRIX_SCAN_COLUMN_SIZE = 1u << 24;             // 16,777,216 values (divisible by 4 for uint4)

static_assert(MATRIX_STORE_SLOTS % MATRIX_PAGE_COUNT == 0,
    "store slots must divide evenly into pages so each page owns a contiguous slice");


In [ ]:
%%writefile ring_buffer.hpp
#pragma once

#include "types.hpp"
#include <atomic>
#include <array>
#include <utility>

/**
 * @brief Ultra-low latency, lock-free, zero-allocation SPSC Ring Buffer.
 * Enforces power-of-two capacities to replace expensive modulo operations with bitwise AND masking.
 */
template <typename T, size_t Capacity>
class SPSCRingBuffer {
    static_assert((Capacity & (Capacity - 1)) == 0, "SPSCRingBuffer Capacity must be a power of two.");
    static_assert(Capacity >= 2, "SPSCRingBuffer Capacity must be at least two elements.");

public:
    SPSCRingBuffer()
        : head_(0)
        , tail_(0)
        , cached_head_(0)
        , cached_tail_(0) {}

    ~SPSCRingBuffer() = default;

    SPSCRingBuffer(const SPSCRingBuffer&) = delete;
    SPSCRingBuffer& operator=(const SPSCRingBuffer&) = delete;
    SPSCRingBuffer(SPSCRingBuffer&&) = delete;
    SPSCRingBuffer& operator=(SPSCRingBuffer&&) = delete;

    /**
     * @brief Emplaces a new item at the tail of the buffer. Only the Producer may call this method.
     */
    template <typename... Args>
    bool try_emplace(Args&&... args) {
        const size_t current_tail = tail_.load(std::memory_order_relaxed);

        // Check local cached head before loading the atomic head_ cursor from shared memory
        if (current_tail - cached_head_ >= Capacity) {
            cached_head_ = head_.load(std::memory_order_acquire);
            if (current_tail - cached_head_ >= Capacity) {
                return false; // Queue is genuinely full
            }
        }

        const size_t index = current_tail & MASK;
        buffer_[index] = T{std::forward<Args>(args)...};

        tail_.store(current_tail + 1, std::memory_order_release);
        return true;
    }

    /**
     * @brief Pops an item from the head of the buffer. Only the Consumer may call this method.
     */
    bool try_pop(T& value) {
        const size_t current_head = head_.load(std::memory_order_relaxed);

        // Check local cached tail before loading the atomic tail_ cursor from shared memory
        if (current_head >= cached_tail_) {
            cached_tail_ = tail_.load(std::memory_order_acquire);
            if (current_head >= cached_tail_) {
                return false; // Queue is genuinely empty
            }
        }

        const size_t index = current_head & MASK;
        value = std::move(buffer_[index]);

        head_.store(current_head + 1, std::memory_order_release);
        return true;
    }

    [[nodiscard]] bool empty() const noexcept {
        return head_.load(std::memory_order_relaxed) == tail_.load(std::memory_order_relaxed);
    }

    [[nodiscard]] size_t size() const noexcept {
        const size_t t = tail_.load(std::memory_order_relaxed);
        const size_t h = head_.load(std::memory_order_relaxed);
        return (t >= h) ? (t - h) : 0;
    }

private:
    static constexpr size_t MASK = Capacity - 1;

    // Isolate variables on separate cache lines to eliminate false sharing
    alignas(MATRIX_CACHE_LINE) std::array<T, Capacity> buffer_{};
    alignas(MATRIX_CACHE_LINE) std::atomic<size_t> head_;
    alignas(MATRIX_CACHE_LINE) std::atomic<size_t> tail_;

    // Thread-local shadow indices
    alignas(MATRIX_CACHE_LINE) size_t cached_head_;
    alignas(MATRIX_CACHE_LINE) size_t cached_tail_;
};


In [ ]:
%%writefile compute.hpp
#pragma once

#include "types.hpp"
#include <cstddef>
#include <cstdint>
#include <cstring>
#include <limits>

// memcpy-based bit_cast (the well-defined type-pun): identical to std::bit_cast but works under C++17
// too. The engine packs typed results (double/int64) into a uint64 out vector via this; std::bit_cast
// would force C++20 on every build, but the GPU/CPU pipeline (main.cpp) compiles under nvcc -std=c++17.
template <typename To, typename From>
inline To matrix_bit_cast(const From& src) {
    static_assert(sizeof(To) == sizeof(From), "matrix_bit_cast requires equal sizes");
    To dst;
    std::memcpy(&dst, &src, sizeof(To));
    return dst;
}

/**
 * @brief Pure virtual interface defining the compute engine's entry point.
 * Enables zero-copy, raw pointer processing over batch slices.
 *
 * Two implementations satisfy this contract and are selected at compile time:
 *   - CPUMockEngine (compute_mock.cpp)  — default, runs anywhere, no GPU needed
 *   - CUDAGPUEngine (compute_cuda.cuh)  — built with nvcc when MATRIX_USE_CUDA is set
 *
 * Both use the page-ownership model: a key always routes to one owning page, so the
 * two backends produce byte-identical store contents (verifiable in main.cpp).
 */
class ComputeInterface {
public:
    virtual ~ComputeInterface() = default;
    virtual void execute_batch(DatabaseQuery* batch_array, size_t count) = 0;

    // Run a single OP_SCAN over the resident column (sets q.transaction_id to the count).
    // Separate from execute_batch so scans run on their own thread/queue and stream and
    // don't head-of-line-block point ops. Touches only scan-path state (disjoint from
    // the point-op store/counters), so the two may run concurrently.
    virtual void execute_scan(DatabaseQuery& q) = 0;

    virtual uint64_t reads() const = 0;
    virtual uint64_t writes() const = 0;
    virtual uint64_t commits() const = 0; // writes actually applied to the store

    // OP_SCAN result accounting: how many scan queries ran, and the running sum of their
    // filter-count results. main asserts the sum against a known oracle to prove scans
    // flowed ring -> batcher -> execute_batch -> (GPU) scan kernel correctly.
    virtual uint64_t scans() const = 0;
    virtual uint64_t scan_result_sum() const = 0;

    // Wall time the engine spent inside OP_SCAN work. Lets the pipeline report point-op
    // and scan throughput separately — a single 64MB scan costs ~1000x a point op, so
    // one combined "ops/sec" number is meaningless once scans are mixed in.
    virtual double scan_time_s() const = 0;

    // Pure kernel time for OP_SCAN work (GPU: cudaEvent-measured, excludes launch/sync/
    // copy and host jitter; CPU: same as the compute loop). scan_time_s() - this =
    // per-scan overhead. The split attributes the integrated-vs-standalone gap.
    virtual double scan_kernel_time_s() const = 0;

    // Order-independent fingerprint of the whole store (sum of slot values). Lets
    // main.cpp assert CPU and GPU reach byte-identical state — the real test of the
    // ownership model, not just matching counts.
    virtual uint64_t store_checksum() const = 0;

    // Scan benchmark: allocate n resident values (value[i]=i), then time a single
    // filter-count scan (count of value[i] > threshold) over that resident data.
    // alloc+fill are NOT timed — the point is "data lives here, query scans it".
    // This is the GPU's home turf (bandwidth over data too big for CPU cache),
    // the opposite of the point-op path. Returns seconds; sets out_count.
    // out_count is deterministic (value[i]=i) so it must match across CPU and GPU.
    virtual double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) = 0;

    // Same scan over a uint32 column. Scan is bandwidth-bound, so halving bytes/value
    // should ~double values/sec at the same GB/s — the columnar "narrowest type" win.
    // If GB/s holds vs the uint64 scan, we are at the bandwidth wall (vectorized loads
    // won't help); if it drops, narrower loads underfill the bus and vectorizing is next.
    virtual double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) = 0;

    // Vectorized uint32 scan (4 values per load via uint4). Tests the memory-level
    // parallelism theory: if this beats the scalar u32 GB/s, narrow scalar loads were
    // underfilling the bus (MLP-bound) and wider loads are the fix. If it doesn't, the
    // kernel is ALU-bound and we stop optimizing. CPU may treat this same as scalar.
    virtual double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) = 0;

    // Items-per-thread scan (register blocking, ITEMS independent loads/thread before
    // comparing). CUB's BlockReduce lever. Tests whether deeper memory-level parallelism
    // per thread beats both scalar and uint4. CPU delegates (compiler auto-vectorizes).
    virtual double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) = 0;
};

// Which page owns a query's key. Single source of truth for both engines and the
// kernel: slot = key & STORE_MASK, page = slot / PAGE_SIZE.
inline size_t matrix_page_of(uint64_t key) {
    return (key & MATRIX_STORE_MASK) / MATRIX_PAGE_SIZE;
}

// OP_SCAN carries its filter threshold in the query payload, and may target a specific
// catalog column: threshold at payload[0] (u32), column id at payload[8] (u64). Column
// id 0 == the legacy fixed scan column. payload begins 8-byte aligned (DatabaseQuery
// starts with u64 fields, payload is at offset 32), so the u64 at payload+8 is aligned.
// One codec used by both engines so they decode identically.
inline void matrix_set_scan_target(DatabaseQuery& q, uint32_t threshold, uint64_t column_id) {
    q.opcode = OP_SCAN;
    *reinterpret_cast<uint32_t*>(q.payload) = threshold;
    *reinterpret_cast<uint64_t*>(q.payload + 8) = column_id;
}
inline uint64_t matrix_get_scan_column_id(const DatabaseQuery& q) {
    return *reinterpret_cast<const uint64_t*>(q.payload + 8);
}
inline void matrix_set_scan_threshold(DatabaseQuery& q, uint32_t threshold) {
    matrix_set_scan_target(q, threshold, 0); // legacy: target the fixed column (id 0)
}
inline uint32_t matrix_get_scan_threshold(const DatabaseQuery& q) {
    return *reinterpret_cast<const uint32_t*>(q.payload);
}

// OP_SCAN's aggregate op lives at payload offset 16 (u32). Default 0 == AGG_COUNT (the original
// count semantics), so a query built with only set_scan_target/set_scan_threshold reduces by COUNT.
inline void matrix_set_scan_agg_op(DatabaseQuery& q, MatrixAggOp op) {
    *reinterpret_cast<uint32_t*>(q.payload + 16) = static_cast<uint32_t>(op);
}
inline MatrixAggOp matrix_get_scan_agg_op(const DatabaseQuery& q) {
    return static_cast<MatrixAggOp>(*reinterpret_cast<const uint32_t*>(q.payload + 16));
}

// Filtered reduction over a uint32 column: reduce the values matching the predicate
// (value > threshold) by `op`. One tight loop per op (dispatch OUTSIDE the loop, so the COUNT
// loop stays exactly as fast as the original scan). Empty match-set sentinels: COUNT/SUM -> 0,
// MIN -> UINT64_MAX, MAX -> 0 (matched values are > threshold >= 0, i.e. >= 1, so 0 / UINT64_MAX
// unambiguously signal "no match"). SUM accumulates in u64 (no overflow for the engine's columns).
inline uint64_t matrix_cpu_reduce(const uint32_t* v, size_t n, uint32_t threshold, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) if (v[i] > threshold) s += v[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if (v[i] > threshold && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if (v[i] > threshold && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { uint64_t c = 0; for (size_t i = 0; i < n; ++i) c += (v[i] > threshold); return c; }
    }
}

// Unfiltered scalar reduction (aggregate ALL values; the no-WHERE companion to matrix_cpu_reduce).
// COUNT -> n; SUM -> Σv (u64); MIN/MAX over all (empty sentinels MIN UINT64_MAX / MAX 0 reachable
// only when n==0). Separate from matrix_cpu_reduce (which always filters value>threshold) — clearer
// than threading if constexpr through its four per-op loops.
inline uint64_t matrix_cpu_reduce_all(const uint32_t* v, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) s += v[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if (v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if (v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      return n;
    }
}

// Null-aware unfiltered scalar reduce over a uint32 column (DM-3j): rows where nulls[i] != 0 are SKIPPED
// (SQL NULL semantics — COUNT counts non-null, SUM/MIN/MAX ignore nulls). Empty/all-null sentinels match
// matrix_cpu_reduce (COUNT/SUM 0, MIN UINT64_MAX, MAX 0). `nulls` has one byte per row.
inline uint64_t matrix_cpu_reduce_all_nullable(const uint32_t* v, size_t n, MatrixAggOp op, const uint8_t* nulls) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) if (!nulls[i]) s += v[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { uint64_t c = 0; for (size_t i = 0; i < n; ++i) c += (nulls[i] == 0); return c; }
    }
}

// Per-column element type. U32 == 0 is the default (an untagged column is uint32). int64 columns
// (DM-3a) hold signed 64-bit values — negatives and values beyond UINT32_MAX, which uint32 cannot.
enum class MatrixType : uint32_t { U32 = 0, I64, F64 };   // F64 == 2 (DM-3e)

// Signed unfiltered scalar reduce over an int64 column (the int64 sibling of matrix_cpu_reduce_all).
// COUNT -> n; SUM -> Σv (int64; see the overflow note in the design); MIN/MAX over all (empty
// sentinels MIN INT64_MAX / MAX INT64_MIN, reachable only when n == 0).
inline int64_t matrix_cpu_reduce_all_i64(const int64_t* v, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { int64_t s = 0; for (size_t i = 0; i < n; ++i) s += v[i]; return s; }
        case AGG_MIN: { int64_t m = INT64_MAX; for (size_t i = 0; i < n; ++i) if (v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { int64_t m = INT64_MIN; for (size_t i = 0; i < n; ++i) if (v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      return static_cast<int64_t>(n);
    }
}

// Null-aware int64 reduce (DM-3j across types): skip rows where nulls[i] != 0. Sentinels match
// matrix_cpu_reduce_all_i64 (SUM/COUNT 0, MIN INT64_MAX, MAX INT64_MIN).
inline int64_t matrix_cpu_reduce_all_i64_nullable(const int64_t* v, size_t n, MatrixAggOp op, const uint8_t* nulls) {
    switch (op) {
        case AGG_SUM: { int64_t s = 0; for (size_t i = 0; i < n; ++i) if (!nulls[i]) s += v[i]; return s; }
        case AGG_MIN: { int64_t m = INT64_MAX; for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { int64_t m = INT64_MIN; for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { int64_t c = 0; for (size_t i = 0; i < n; ++i) c += (nulls[i] == 0); return c; }
    }
}

enum class MatrixCmp : uint32_t { GT = 0, GE, LT, LE, EQ, NE, BETWEEN }; // GT == 0 == the default

// A value predicate for WHERE. `a` = primary bound (threshold; inclusive lower for BETWEEN);
// `b` = inclusive upper for BETWEEN (ignored otherwise). uint32 to match the column element type.
struct MatrixPredicate { MatrixCmp cmp = MatrixCmp::GT; uint32_t a = 0; uint32_t b = 0; };

inline bool matrix_pred_match(uint32_t v, const MatrixPredicate& p) {
    switch (p.cmp) {
        case MatrixCmp::GE:      return v >= p.a;
        case MatrixCmp::LT:      return v <  p.a;
        case MatrixCmp::LE:      return v <= p.a;
        case MatrixCmp::EQ:      return v == p.a;
        case MatrixCmp::NE:      return v != p.a;
        case MatrixCmp::BETWEEN: return v >= p.a && v <= p.b;   // inclusive [a, b]
        case MatrixCmp::GT:
        default:                 return v >  p.a;
    }
}

// Predicate-aware filtered scalar reduce — the general sibling of matrix_cpu_reduce. Same empty-set
// sentinels (COUNT/SUM 0, MIN UINT64_MAX, MAX 0; see the MAX caveat in the design). matrix_cpu_reduce
// (the GT fast path) is intentionally left untouched so the id-0 oracle scan is byte-identical.
inline uint64_t matrix_cpu_reduce_pred(const uint32_t* v, size_t n, const MatrixPredicate& p, MatrixAggOp op,
                                       const uint8_t* nulls = nullptr) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match(v[i], p)) s += v[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match(v[i], p) && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match(v[i], p) && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { uint64_t c = 0; for (size_t i = 0; i < n; ++i) c += ((!nulls || !nulls[i]) && matrix_pred_match(v[i], p)); return c; }
    }
}

// int64 value predicate for WHERE (the signed sibling of MatrixPredicate). a = primary bound / BETWEEN
// lower (inclusive); b = BETWEEN upper (inclusive). Signed comparisons so negatives order correctly.
struct MatrixPredicateI64 { MatrixCmp cmp = MatrixCmp::GT; int64_t a = 0; int64_t b = 0; };

inline bool matrix_pred_match_i64(int64_t v, const MatrixPredicateI64& p) {
    switch (p.cmp) {
        case MatrixCmp::GE:      return v >= p.a;
        case MatrixCmp::LT:      return v <  p.a;
        case MatrixCmp::LE:      return v <= p.a;
        case MatrixCmp::EQ:      return v == p.a;
        case MatrixCmp::NE:      return v != p.a;
        case MatrixCmp::BETWEEN: return v >= p.a && v <= p.b;
        case MatrixCmp::GT:
        default:                 return v >  p.a;
    }
}

// Filtered signed scalar reduce — the int64 sibling of matrix_cpu_reduce_pred. Same empty sentinels as
// matrix_cpu_reduce_all_i64 (SUM/COUNT 0, MIN INT64_MAX, MAX INT64_MIN; the MAX sentinel is ambiguous
// only for a column literally containing INT64_MIN — read COUNT).
inline int64_t matrix_cpu_reduce_pred_i64(const int64_t* v, size_t n, const MatrixPredicateI64& p, MatrixAggOp op,
                                          const uint8_t* nulls = nullptr) {
    switch (op) {
        case AGG_SUM: { int64_t s = 0; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_i64(v[i], p)) s += v[i]; return s; }
        case AGG_MIN: { int64_t m = INT64_MAX; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_i64(v[i], p) && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { int64_t m = INT64_MIN; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_i64(v[i], p) && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { int64_t c = 0; for (size_t i = 0; i < n; ++i) c += ((!nulls || !nulls[i]) && matrix_pred_match_i64(v[i], p)); return c; }
    }
}

// Unfiltered scalar reduce over a double column. COUNT (double)n; SUM Σv; MIN/MAX over all (empty
// sentinels MIN +inf, MAX -inf). IEEE: NaN values are skipped by MIN/MAX (NaN compares false) and
// poison SUM (NaN propagates) — documented, not special-cased.
inline double matrix_cpu_reduce_all_f64(const double* v, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { double s = 0.0; for (size_t i = 0; i < n; ++i) s += v[i]; return s; }
        case AGG_MIN: { double m = std::numeric_limits<double>::infinity();  for (size_t i = 0; i < n; ++i) if (v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { double m = -std::numeric_limits<double>::infinity(); for (size_t i = 0; i < n; ++i) if (v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      return static_cast<double>(n);
    }
}

// Null-aware double reduce (DM-3j across types): skip rows where nulls[i] != 0. Sentinels match
// matrix_cpu_reduce_all_f64 (SUM/COUNT 0, MIN +inf, MAX -inf). COUNT returns the non-null count.
inline double matrix_cpu_reduce_all_f64_nullable(const double* v, size_t n, MatrixAggOp op, const uint8_t* nulls) {
    switch (op) {
        case AGG_SUM: { double s = 0.0; for (size_t i = 0; i < n; ++i) if (!nulls[i]) s += v[i]; return s; }
        case AGG_MIN: { double m = std::numeric_limits<double>::infinity();  for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { double m = -std::numeric_limits<double>::infinity(); for (size_t i = 0; i < n; ++i) if (!nulls[i] && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { double c = 0.0; for (size_t i = 0; i < n; ++i) c += (nulls[i] == 0) ? 1.0 : 0.0; return c; }
    }
}

struct MatrixPredicateF64 { MatrixCmp cmp = MatrixCmp::GT; double a = 0.0; double b = 0.0; };
inline bool matrix_pred_match_f64(double v, const MatrixPredicateF64& p) {
    switch (p.cmp) {
        case MatrixCmp::GE:      return v >= p.a;
        case MatrixCmp::LT:      return v <  p.a;
        case MatrixCmp::LE:      return v <= p.a;
        case MatrixCmp::EQ:      return v == p.a;
        case MatrixCmp::NE:      return v != p.a;
        case MatrixCmp::BETWEEN: return v >= p.a && v <= p.b;
        case MatrixCmp::GT:
        default:                 return v >  p.a;
    }
}
inline double matrix_cpu_reduce_pred_f64(const double* v, size_t n, const MatrixPredicateF64& p, MatrixAggOp op,
                                         const uint8_t* nulls = nullptr) {
    switch (op) {
        case AGG_SUM: { double s = 0.0; for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_f64(v[i], p)) s += v[i]; return s; }
        case AGG_MIN: { double m = std::numeric_limits<double>::infinity();  for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_f64(v[i], p) && v[i] < m) m = v[i]; return m; }
        case AGG_MAX: { double m = -std::numeric_limits<double>::infinity(); for (size_t i = 0; i < n; ++i) if ((!nulls || !nulls[i]) && matrix_pred_match_f64(v[i], p) && v[i] > m) m = v[i]; return m; }
        case AGG_COUNT:
        default:      { double c = 0.0; for (size_t i = 0; i < n; ++i) c += ((!nulls || !nulls[i]) && matrix_pred_match_f64(v[i], p)) ? 1.0 : 0.0; return c; }
    }
}

// A structured analytical query over catalog columns (value_col / key_col > 0). has_filter applies
// WHERE value <cmp> threshold (BETWEEN uses [threshold, upper]); grouped applies GROUP BY key_col into num_groups dense buckets.
struct MatrixQuery {
    uint64_t    value_col  = 0;
    MatrixAggOp agg        = AGG_COUNT;
    bool        has_filter = false;
    uint32_t    threshold  = 0;
    MatrixCmp   cmp        = MatrixCmp::GT;   // comparison op for the filter (default keeps value>threshold)
    uint32_t    upper      = 0;               // BETWEEN's inclusive upper bound (ignored for other ops)
    int64_t     lo_i64     = 0;   // int64 filter primary bound / BETWEEN lower (I64 columns)
    int64_t     hi_i64     = 0;   // int64 filter BETWEEN upper (I64 columns)
    double      lo_f64     = 0.0; // double filter primary bound / BETWEEN lower (F64 columns)
    double      hi_f64     = 0.0; // double filter BETWEEN upper (F64 columns)
    bool        grouped    = false;
    uint64_t    key_col    = 0;
    uint32_t    num_groups = 0;
    // limit crosses the wire (matrix_serialize_request round-trips it byte-for-byte) but has NO effect via
    // the network QUERY dispatch today: serve_core calls execute_query()/execute_query_impl(), which never
    // reads q.limit -- top-N (ORDER BY agg DESC LIMIT n) only exists via top_query(sql), a separate
    // SQL-string CLI helper that calls top_groups() directly and isn't reachable from the wire protocol.
    // A network client that sets limit gets a normal (non-top-N) query result, not silently wrong data,
    // but also not top-N -- this is a missing feature, not a data-corruption risk. Wiring top-N into
    // serve_core's QUERY case is the fix; not done here (out of scope for the field-dropping bug this
    // struct's other fields were fixed for).
    uint64_t    limit      = 0;
    uint64_t    filter_col = 0;   // cross-column WHERE: u32 column the filter reads (0 = filter the value column itself)
};

// Result of CPUMockEngine::execute_query — OK or a specific input-validation rejection (the query
// boundary never crashes on caller input; on any ERR the out vector is left empty).
enum class MatrixQueryStatus { OK, ERR_UNKNOWN_COLUMN, ERR_INVALID_GROUP, ERR_TOO_MANY_GROUPS, ERR_UNSUPPORTED_TYPE, ERR_PARSE };

// Grouped reduction core (GROUP BY key). Filtered==true applies WHERE matrix_pred_match(value, pred) (compiled
// out when false via if constexpr, so the unfiltered path is byte-identical to the original). Dense
// groups [0, num_groups); keys >= num_groups ignored; out initialized per op (empty-group sentinels
// match matrix_cpu_reduce: COUNT/SUM/MAX -> 0, MIN -> UINT64_MAX). SUM accumulates in u64. One pass;
// the op branch is inside the loop because grouped reduction is scatter-bound (random out[k] writes),
// not branch-bound.
template <bool Filtered>
inline void matrix_group_reduce_impl(const uint32_t* keys, const uint32_t* values, size_t n,
                                     uint32_t num_groups, MatrixAggOp op, const MatrixPredicate& pred, uint64_t* out,
                                     const uint8_t* nulls = nullptr) {
    const uint64_t init = (op == AGG_MIN) ? UINT64_MAX : 0;
    for (uint32_t g = 0; g < num_groups; ++g) out[g] = init;
    for (size_t i = 0; i < n; ++i) {
        if (nulls && nulls[i]) continue;                     // NULL row: excluded from every group aggregate
        const uint32_t k = keys[i];
        if (k >= num_groups) continue;                       // out-of-range key: ignored
        const uint32_t v = values[i];
        if constexpr (Filtered) { if (!matrix_pred_match(v, pred)) continue; }  // WHERE predicate
        switch (op) {
            case AGG_SUM:   out[k] += v; break;
            case AGG_MIN:   if (v < out[k]) out[k] = v; break;
            case AGG_MAX:   if (v > out[k]) out[k] = v; break;
            case AGG_COUNT:
            default:        out[k] += 1; break;
        }
    }
}
// GROUP BY key (all rows). Unchanged signature from GBY-1 — now a thin wrapper. `nulls` (default none) skips NULL rows.
inline void matrix_cpu_group_reduce(const uint32_t* keys, const uint32_t* values, size_t n,
                                    uint32_t num_groups, MatrixAggOp op, uint64_t* out, const uint8_t* nulls = nullptr) {
    matrix_group_reduce_impl<false>(keys, values, n, num_groups, op, MatrixPredicate{}, out, nulls);
}
// GROUP BY key WHERE value > threshold.
inline void matrix_cpu_group_reduce_where(const uint32_t* keys, const uint32_t* values, size_t n,
                                          uint32_t num_groups, MatrixAggOp op, uint32_t threshold, uint64_t* out) {
    matrix_group_reduce_impl<true>(keys, values, n, num_groups, op, MatrixPredicate{MatrixCmp::GT, threshold, 0}, out);
}
inline void matrix_cpu_group_reduce_pred(const uint32_t* keys, const uint32_t* values, size_t n,
                                         uint32_t num_groups, MatrixAggOp op, const MatrixPredicate& pred, uint64_t* out,
                                         const uint8_t* nulls = nullptr) {
    matrix_group_reduce_impl<true>(keys, values, n, num_groups, op, pred, out, nulls);
}

// Grouped reduction over an int64 value column keyed by a uint32 key column (dense group ids). Same
// semantics as matrix_group_reduce_impl but values + accumulators are signed int64. Empty-group
// sentinels: COUNT/SUM 0, MIN INT64_MAX, MAX INT64_MIN. NOTE: MAX inits to INT64_MIN (not 0) because
// int64 values may be negative. Filtered applies the int64 predicate.
template <bool Filtered>
inline void matrix_group_reduce_i64_impl(const uint32_t* keys, const int64_t* values, size_t n,
                                         uint32_t num_groups, MatrixAggOp op, const MatrixPredicateI64& pred, int64_t* out,
                                         const uint8_t* nulls = nullptr) {
    for (uint32_t g = 0; g < num_groups; ++g)
        out[g] = (op == AGG_MIN) ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0);
    for (size_t i = 0; i < n; ++i) {
        if (nulls && nulls[i]) continue;                     // NULL row: excluded
        const uint32_t k = keys[i];
        if (k >= num_groups) continue;
        const int64_t v = values[i];
        if constexpr (Filtered) { if (!matrix_pred_match_i64(v, pred)) continue; }
        switch (op) {
            case AGG_SUM:   out[k] += v; break;
            case AGG_MIN:   if (v < out[k]) out[k] = v; break;
            case AGG_MAX:   if (v > out[k]) out[k] = v; break;
            case AGG_COUNT:
            default:        out[k] += 1; break;
        }
    }
}
inline void matrix_cpu_group_reduce_i64(const uint32_t* keys, const int64_t* values, size_t n,
                                        uint32_t num_groups, MatrixAggOp op, int64_t* out, const uint8_t* nulls = nullptr) {
    matrix_group_reduce_i64_impl<false>(keys, values, n, num_groups, op, MatrixPredicateI64{}, out, nulls);
}
inline void matrix_cpu_group_reduce_i64_pred(const uint32_t* keys, const int64_t* values, size_t n,
                                             uint32_t num_groups, MatrixAggOp op, const MatrixPredicateI64& pred, int64_t* out,
                                             const uint8_t* nulls = nullptr) {
    matrix_group_reduce_i64_impl<true>(keys, values, n, num_groups, op, pred, out, nulls);
}

// Grouped reduction over a double value column keyed by a uint32 key column (dense group ids). Double
// accumulators; empty-group sentinels COUNT/SUM 0.0, MIN +inf, MAX -inf (MAX inits -inf so a negative
// group yields the right max). Filtered applies the double predicate.
template <bool Filtered>
inline void matrix_group_reduce_f64_impl(const uint32_t* keys, const double* values, size_t n,
                                         uint32_t num_groups, MatrixAggOp op, const MatrixPredicateF64& pred, double* out,
                                         const uint8_t* nulls = nullptr) {
    for (uint32_t g = 0; g < num_groups; ++g)
        out[g] = (op == AGG_MIN) ? std::numeric_limits<double>::infinity()
               : (op == AGG_MAX ? -std::numeric_limits<double>::infinity() : 0.0);
    for (size_t i = 0; i < n; ++i) {
        if (nulls && nulls[i]) continue;                     // NULL row: excluded
        const uint32_t k = keys[i];
        if (k >= num_groups) continue;
        const double v = values[i];
        if constexpr (Filtered) { if (!matrix_pred_match_f64(v, pred)) continue; }
        switch (op) {
            case AGG_SUM:   out[k] += v; break;
            case AGG_MIN:   if (v < out[k]) out[k] = v; break;
            case AGG_MAX:   if (v > out[k]) out[k] = v; break;
            case AGG_COUNT:
            default:        out[k] += 1.0; break;
        }
    }
}
inline void matrix_cpu_group_reduce_f64(const uint32_t* keys, const double* values, size_t n,
                                        uint32_t num_groups, MatrixAggOp op, double* out, const uint8_t* nulls = nullptr) {
    matrix_group_reduce_f64_impl<false>(keys, values, n, num_groups, op, MatrixPredicateF64{}, out, nulls);
}
inline void matrix_cpu_group_reduce_f64_pred(const uint32_t* keys, const double* values, size_t n,
                                             uint32_t num_groups, MatrixAggOp op, const MatrixPredicateF64& pred, double* out,
                                             const uint8_t* nulls = nullptr) {
    matrix_group_reduce_f64_impl<true>(keys, values, n, num_groups, op, pred, out, nulls);
}

// --- Cross-column scalar reduce: aggregate the VALUE column `a` over rows where the u32 filter predicate
// `p` holds on a DIFFERENT, aligned column `f` (the analytical "SELECT agg(value) WHERE other <pred>").
// Filter column is u32 (covers dict-encoded strings + u32 dims); value is u32/i64/f64. Same empty-set
// sentinels as matrix_cpu_reduce_all*. COUNT returns the number of rows matching the filter. ---
inline uint64_t matrix_cpu_reduce_filtered_by(const uint32_t* f, const MatrixPredicate& p, const uint32_t* a, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { uint64_t s = 0; for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p)) s += a[i]; return s; }
        case AGG_MIN: { uint64_t m = UINT64_MAX; for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p) && a[i] < m) m = a[i]; return m; }
        case AGG_MAX: { uint64_t m = 0; for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p) && a[i] > m) m = a[i]; return m; }
        case AGG_COUNT:
        default:      { uint64_t c = 0; for (size_t i = 0; i < n; ++i) c += matrix_pred_match(f[i], p); return c; }
    }
}
inline int64_t matrix_cpu_reduce_filtered_by_i64(const uint32_t* f, const MatrixPredicate& p, const int64_t* a, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { int64_t s = 0; for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p)) s += a[i]; return s; }
        case AGG_MIN: { int64_t m = INT64_MAX; for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p) && a[i] < m) m = a[i]; return m; }
        case AGG_MAX: { int64_t m = INT64_MIN; for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p) && a[i] > m) m = a[i]; return m; }
        case AGG_COUNT:
        default:      { int64_t c = 0; for (size_t i = 0; i < n; ++i) c += matrix_pred_match(f[i], p); return c; }
    }
}
inline double matrix_cpu_reduce_filtered_by_f64(const uint32_t* f, const MatrixPredicate& p, const double* a, size_t n, MatrixAggOp op) {
    switch (op) {
        case AGG_SUM: { double s = 0.0; for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p)) s += a[i]; return s; }
        case AGG_MIN: { double m = std::numeric_limits<double>::infinity();  for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p) && a[i] < m) m = a[i]; return m; }
        case AGG_MAX: { double m = -std::numeric_limits<double>::infinity(); for (size_t i = 0; i < n; ++i) if (matrix_pred_match(f[i], p) && a[i] > m) m = a[i]; return m; }
        case AGG_COUNT:
        default:      { double c = 0.0; for (size_t i = 0; i < n; ++i) c += matrix_pred_match(f[i], p); return c; }
    }
}

// --- Grouped cross-column reduce: GROUP BY `keys`, aggregate `values`, over rows where the u32 filter
// predicate `fp` holds on a separate aligned column `f` ("SELECT agg(value) WHERE dim <pred> GROUP BY key").
// Per-group out[] carries the same encoding/sentinels as matrix_cpu_group_reduce[_i64/_f64]. ---
inline void matrix_cpu_group_reduce_filtered_by(const uint32_t* keys, const uint32_t* f, const MatrixPredicate& fp,
                                                const uint32_t* values, size_t n, uint32_t num_groups, MatrixAggOp op, uint64_t* out) {
    for (uint32_t g = 0; g < num_groups; ++g) out[g] = (op == AGG_MIN) ? UINT64_MAX : 0;
    for (size_t i = 0; i < n; ++i) {
        const uint32_t k = keys[i];
        if (k >= num_groups || !matrix_pred_match(f[i], fp)) continue;
        const uint32_t v = values[i];
        switch (op) {
            case AGG_SUM: out[k] += v; break;
            case AGG_MIN: if (v < out[k]) out[k] = v; break;
            case AGG_MAX: if (v > out[k]) out[k] = v; break;
            case AGG_COUNT: default: out[k] += 1; break;
        }
    }
}
inline void matrix_cpu_group_reduce_filtered_by_i64(const uint32_t* keys, const uint32_t* f, const MatrixPredicate& fp,
                                                    const int64_t* values, size_t n, uint32_t num_groups, MatrixAggOp op, int64_t* out) {
    for (uint32_t g = 0; g < num_groups; ++g) out[g] = (op == AGG_MIN) ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0);
    for (size_t i = 0; i < n; ++i) {
        const uint32_t k = keys[i];
        if (k >= num_groups || !matrix_pred_match(f[i], fp)) continue;
        const int64_t v = values[i];
        switch (op) {
            case AGG_SUM: out[k] += v; break;
            case AGG_MIN: if (v < out[k]) out[k] = v; break;
            case AGG_MAX: if (v > out[k]) out[k] = v; break;
            case AGG_COUNT: default: out[k] += 1; break;
        }
    }
}
inline void matrix_cpu_group_reduce_filtered_by_f64(const uint32_t* keys, const uint32_t* f, const MatrixPredicate& fp,
                                                    const double* values, size_t n, uint32_t num_groups, MatrixAggOp op, double* out) {
    const double inf = std::numeric_limits<double>::infinity();
    for (uint32_t g = 0; g < num_groups; ++g) out[g] = (op == AGG_MIN) ? inf : (op == AGG_MAX ? -inf : 0.0);
    for (size_t i = 0; i < n; ++i) {
        const uint32_t k = keys[i];
        if (k >= num_groups || !matrix_pred_match(f[i], fp)) continue;
        const double v = values[i];
        switch (op) {
            case AGG_SUM: out[k] += v; break;
            case AGG_MIN: if (v < out[k]) out[k] = v; break;
            case AGG_MAX: if (v > out[k]) out[k] = v; break;
            case AGG_COUNT: default: out[k] += 1.0; break;
        }
    }
}

/**
 * @brief Stable counting-sort of a batch by owning page (the "bin in the batcher" step).
 * Fills `binned` with the queries grouped by page — order within a page preserved, so
 * same-slot writes keep batch order and last-writer-wins is deterministic. `offsets`
 * is a CSR index: page p's queries are binned[offsets[p] .. offsets[p+1]).
 * Caller owns the buffers (binned >= count, offsets >= PAGE_COUNT+1) — zero alloc here.
 */
inline void matrix_bin_by_page(const DatabaseQuery* batch, size_t count,
                               DatabaseQuery* binned, uint32_t* offsets) {
    uint32_t counts[MATRIX_PAGE_COUNT] = {0};
    for (size_t i = 0; i < count; ++i) {
        counts[matrix_page_of(batch[i].query_id)]++;
    }
    offsets[0] = 0;
    for (size_t p = 0; p < MATRIX_PAGE_COUNT; ++p) {
        offsets[p + 1] = offsets[p] + counts[p];
    }
    uint32_t cursor[MATRIX_PAGE_COUNT];
    for (size_t p = 0; p < MATRIX_PAGE_COUNT; ++p) cursor[p] = offsets[p];
    for (size_t i = 0; i < count; ++i) {
        const size_t p = matrix_page_of(batch[i].query_id);
        binned[cursor[p]++] = batch[i];
    }
}


In [ ]:
%%writefile memory_model.hpp
#pragma once

// Where a dataset physically lives / which processor addresses it.
enum class MemorySpace {
    HOST,    // CPU RAM
    DEVICE,  // GPU VRAM (discrete)
    COLD,    // SSD — cold columns + durability log (append-only, high latency)
    UNIFIED, // shared CPU+GPU pool (DGX Spark / Grace-Hopper) — placement is zero-copy
};

// Boot-time description of the machine's memory architecture. The CostModel reads this
// so the data-transfer term is included (discrete) or zero (unified).
//   DISCRETE: HOST and DEVICE are distinct; placing data in DEVICE costs a transfer.
//   UNIFIED : one physical pool; "placement" only chooses the executor, never moves data.
struct MemoryModel {
    enum Kind { DISCRETE, UNIFIED } kind = DISCRETE;

    // ponytail: unified-memory hardware isn't available to test on, so detection
    // defaults to DISCRETE. When a unified box (DGX Spark / GH) is in hand, set this
    // from cudaDeviceGetAttribute(cudaDevAttrPageableMemoryAccess / Managed) and
    // implement the UNIFIED cost branch. Seam only for now.
    static MemoryModel detect(bool gpu_available) {
        MemoryModel m;
        m.kind = DISCRETE; // discrete-only until validated on real unified hardware
        (void)gpu_available;
        return m;
    }

    bool is_unified() const { return kind == UNIFIED; }
};


In [ ]:
%%writefile tier_model.hpp
#pragma once

#include "memory_model.hpp"
#include <cstddef>

// The storage tiers, ordered fastest-scan to coldest. DEVICE=VRAM, HOST=RAM, COLD=SSD.
// UNIFIED is the existing seam from memory_model.hpp (CPU+GPU share one pool); when the
// machine is unified, DEVICE and HOST collapse and migration between them is a no-op.
//
// NOTE: MemorySpace is DEFINED in memory_model.hpp as { HOST, DEVICE, COLD, UNIFIED };
// this header layers the per-tier physics on top of that enum.

// Physics of one tier: what it's good at and what it costs. bandwidth in bytes/microsecond
// (so it composes with CostModel's existing *_BPus units); latency in microseconds to first
// byte; capacity_bytes is the usable budget (0 == "treat as unbounded for now").
struct TierPhysics {
    double   scan_bytes_per_us;  // sequential scan bandwidth
    double   access_latency_us;  // latency to first byte (random-access cost proxy)
    size_t   capacity_bytes;     // usable capacity budget; 0 = unbounded (not yet enforced)
    const char* concern;         // the tier's defining risk, for docs/telemetry
};

// First-estimate physics per tier (CALIBRATION TARGETS, like CostModel's constants).
// VRAM/RAM scan numbers are measured; SSD and latencies are estimates to refine on the box.
inline TierPhysics tier_physics(MemorySpace s) {
    switch (s) {
        case MemorySpace::DEVICE: // VRAM
            return TierPhysics{240'000.0, 5.0, 16ull*1024*1024*1024, "scarce capacity; PCIe to reach"};
        case MemorySpace::HOST:   // RAM
            return TierPhysics{10'000.0, 0.1, 256ull*1024*1024*1024, "medium capacity"};
        case MemorySpace::COLD:   // SSD
            return TierPhysics{3'000.0, 100.0, 0 /*unbounded*/, "high latency; write wear; append-only"};
        case MemorySpace::UNIFIED:
            return TierPhysics{240'000.0, 0.1, 0, "unified pool (future hardware)"};
    }
    return TierPhysics{1.0, 1.0, 0, "unknown"};
}


In [ ]:
%%writefile cost_model.hpp
#pragma once

#include "memory_model.hpp"
#include "tier_model.hpp"
#include <cstddef>

// Cost-based placement: predict microseconds for a workload on each processor, choose
// the cheaper home. Constants are MEASURED on the target machine (values below are first
// estimates from Tesla T4 runs — see the spec's calibration note). The structure is the
// deliverable; the constants get one calibration pass before the boundary is trusted.
class CostModel {
public:
    explicit CostModel(MemoryModel mm, bool gpu_available = true)
        : mm_(mm), gpu_available_(gpu_available) {}

    // Point ops: CPU always wins (PCIe slower than CPU cache, ~1 memory op/query).
    MemorySpace place_point() const { return MemorySpace::HOST; }

    // Scan of `bytes`: place where the predicted scan time is lower.
    MemorySpace place_scan(size_t bytes) const {
        if (!gpu_available_) return MemorySpace::HOST;
        return device_scan_us(bytes) < host_scan_us(bytes)
                   ? MemorySpace::DEVICE
                   : MemorySpace::HOST;
    }

    // Predicted microseconds (exposed for tests / future tuning).
    double host_scan_us(size_t bytes) const {
        return static_cast<double>(bytes) / CPU_SCAN_BPus;
    }
    double device_scan_us(size_t bytes) const {
        // Transfer is amortized to 0 today (a column is placed once and scanned many).
        // Both branches are 0.0 ON PURPOSE: this is the seam where the discrete one-time
        // transfer term will go; UNIFIED stays 0. Deliberately inert until calibrated.
        const double transfer = mm_.is_unified() ? 0.0 : 0.0;
        return LAUNCH_US + transfer + static_cast<double>(bytes) / GPU_SCAN_BPus;
    }

    // Predicted scan time (us) for `bytes` resident in tier `s`. Uses each tier's measured
    // bandwidth from tier_model. This generalizes host_scan_us/device_scan_us to all tiers
    // (the existing two stay for the router's HOST/DEVICE fast path and back-compat).
    double scan_us(MemorySpace s, size_t bytes) const {
        const TierPhysics p = tier_physics(s);
        const double launch = (s == MemorySpace::DEVICE) ? LAUNCH_US : 0.0;
        return launch + static_cast<double>(bytes) / p.scan_bytes_per_us;
    }

    // Predicted one-time cost (us) to move `bytes` from tier `from` to tier `to`. Zero if
    // same tier (no move) or if memory is unified (zero-copy). Otherwise bounded by the
    // slower of read-from-source and write-to-dest bandwidth.
    double migration_us(MemorySpace from, MemorySpace to, size_t bytes) const {
        if (from == to) return 0.0;
        if (mm_.is_unified()) return 0.0; // unified pool: placement is zero-copy
        const double read_bw  = tier_physics(from).scan_bytes_per_us;
        const double write_bw = tier_physics(to).scan_bytes_per_us;
        const double slower = (read_bw < write_bw) ? read_bw : write_bw;
        return static_cast<double>(bytes) / slower;
    }

private:
    // --- measured constants (bytes per microsecond) — CALIBRATION TARGETS ---
    static constexpr double CPU_SCAN_BPus = 10'000.0;   // ~10 GB/s  (measured)
    static constexpr double GPU_SCAN_BPus = 240'000.0;  // ~240 GB/s (measured at 64 MB)
    static constexpr double LAUNCH_US     = 30.0;       // per-scan GPU launch floor (measured)

    MemoryModel mm_;
    bool gpu_available_;
};


In [ ]:
%%writefile router.hpp
#pragma once

#include "compute.hpp"
#include "cost_model.hpp"
#include "memory_model.hpp"
#include <cassert>
#include <cstdint>
#include <vector>

// Routing policy over two live engines. Holds no data and runs no kernels: it decides
// (via CostModel) where each dataset lives, records it in a placement map, and dispatches
// each query to the engine that owns its data. gpu may be null (no-GPU build) — then the
// CostModel places everything HOST and all routing goes to the CPU engine.
class Router {
public:
    Router(ComputeInterface* cpu, ComputeInterface* gpu, CostModel cm)
        : cpu_(cpu), gpu_(gpu), cm_(cm) {}

    // Point ops run where the cost model places them — always HOST today (page-ownership
    // KV store lives in CPU RAM; GPU loses point ops). Routed via the model so the
    // invariant lives in one place.
    void route_batch(DatabaseQuery* batch, size_t count) {
        ComputeInterface* eng =
            (cm_.place_point() == MemorySpace::DEVICE && gpu_) ? gpu_ : cpu_;
        eng->execute_batch(batch, count);
    }

    // Register a scan column of `bytes`, deciding its home now. Returns a dataset id used
    // by route_scan. One home per dataset — recorded once, never duplicated.
    uint64_t place_scan_column(size_t bytes) {
        const MemorySpace home =
            (gpu_ != nullptr) ? cm_.place_scan(bytes) : MemorySpace::HOST;
        placement_.push_back(home);
        return static_cast<uint64_t>(placement_.size() - 1);
    }

    // Dispatch a scan to the engine that owns the column's data.
    void route_scan(DatabaseQuery& q, uint64_t dataset_id) {
        assert(dataset_id < placement_.size() && "route_scan: unknown dataset_id");
        const MemorySpace home = placement_[dataset_id];
        ComputeInterface* eng = (home == MemorySpace::DEVICE && gpu_) ? gpu_ : cpu_;
        eng->execute_scan(q);
    }

    MemorySpace home_of(uint64_t dataset_id) const {
        assert(dataset_id < placement_.size() && "home_of: unknown dataset_id");
        return placement_[dataset_id];
    }

private:
    ComputeInterface* cpu_;
    ComputeInterface* gpu_;             // may be null
    CostModel cm_;
    std::vector<MemorySpace> placement_; // dataset id -> home (the entire coherence story)
};


In [ ]:
%%writefile kv_store.hpp
#pragma once

#include <cstdint>
#include <cstddef>
#include <vector>

// Open-addressing hash table (linear probing), key -> value, fixed capacity.
// This is the point-op store (gap DM-1). The prototype used store[key & MASK], which
// SILENTLY OVERWROTE colliding keys — a data-loss bug. Here, distinct keys probe to
// distinct slots and never overwrite each other; a full table is an explicit error
// (put returns false), never silent corruption.
//
// Single-owner: the engine's point-op path is single-threaded per the page-ownership
// model, so no internal locking. capacity is rounded up to a power of two so the slot
// index is a mask, not a modulo. SSD-spill (gap DM-9 / Inc 3) is a future seam: when
// full we return false today; later that path demotes cold entries to the cold tier.
class KVStore {
public:
    explicit KVStore(size_t capacity)
        : mask_(round_up_pow2(capacity) - 1),
          slots_(round_up_pow2(capacity)) {}

    // Insert or update. Returns false ONLY if the table is full and the key is new
    // (explicit failure, never overwrite of a different key).
    bool put(uint64_t key, uint64_t value) {
        size_t i = key & mask_;
        for (size_t probe = 0; probe <= mask_; ++probe) {
            Slot& s = slots_[i];
            if (!s.occupied) {            // empty slot: new insert
                s.key = key; s.value = value; s.occupied = true;
                ++size_;
                return true;
            }
            if (s.key == key) {           // same key: update in place
                s.value = value;
                return true;
            }
            i = (i + 1) & mask_;          // collision with a DIFFERENT key: probe on
        }
        return false;                     // table full, key not present: explicit error
    }

    // Look up. Returns true and sets out if present; false if absent.
    bool get(uint64_t key, uint64_t& out) const {
        size_t i = key & mask_;
        for (size_t probe = 0; probe <= mask_; ++probe) {
            const Slot& s = slots_[i];
            if (!s.occupied) return false;     // empty slot ends the probe chain: miss
            if (s.key == key) { out = s.value; return true; }
            i = (i + 1) & mask_;
        }
        return false;
    }

    size_t size() const { return size_; }
    size_t capacity() const { return mask_ + 1; }

    // Order-independent fingerprint: sum of stored values (matches the engine's old
    // store_checksum semantics so cross-checks stay meaningful).
    uint64_t checksum() const {
        uint64_t sum = 0;
        for (const Slot& s : slots_) if (s.occupied) sum += s.value;
        return sum;
    }

    // Visit every live entry (snapshot / iteration). Allocation-free; same walk as checksum().
    template <typename F>
    void for_each(F&& f) const { for (const Slot& s : slots_) if (s.occupied) f(s.key, s.value); }

private:
    struct Slot { uint64_t key = 0; uint64_t value = 0; bool occupied = false; };

    static size_t round_up_pow2(size_t n) {
        size_t p = 1;
        while (p < n) p <<= 1;
        return p < 2 ? 2 : p; // minimum 2 slots
    }

    size_t mask_;
    size_t size_ = 0;
    std::vector<Slot> slots_;
};


In [ ]:
%%writefile cold_store.hpp
#pragma once

#include "types.hpp"   // OP_WRITE
#include <cstdio>
#include <cstdint>
#include <cstddef>
#include <cstring>
#include <cstdlib>
#include <string>
#include <vector>
#include <utility>
#include <unistd.h>    // fsync, fileno

// SSD-backed write-ahead log (gap DU-1/2/3 / three-tier cold tier). Append-only: a record
// is [u32 length][u32 crc32(payload)][payload]. Synchronous (fsync per policy) so a
// returned append_put is durable. replay() rebuilds state front-to-back and stops at the
// first torn or corrupt record — never replays corruption. "SSD" is a file here; the
// interface is identical on real flash.

enum class SyncPolicy {
    SYNC_EACH,  // fsync after every append — committed write survives a crash (default)
    SYNC_OFF,   // no fsync — tests / throughput; crash loses unflushed tail
};

// On-disk payload layout (serialized explicitly, NOT a C struct — avoids padding/ABI
// dependence): key (8 bytes) + value (8 bytes) + opcode (4 bytes) = 20 bytes.
constexpr size_t MATRIX_WAL_PAYLOAD_BYTES = 20;
constexpr uint32_t MATRIX_WAL_MAX_RECORD = 4096;    // sane upper bound for the length field
constexpr size_t   MATRIX_WAL_COMMIT_BYTES = 4;          // commit-marker record length
constexpr uint32_t MATRIX_WAL_COMMIT       = 0x434F4D4Du; // 'COMM' — commit-marker payload

// Standard CRC32 (reflected, poly 0xEDB88320). Inline, no dependency.
inline uint32_t matrix_crc32(const unsigned char* data, size_t n) {
    uint32_t crc = 0xFFFFFFFFu;
    for (size_t i = 0; i < n; ++i) {
        crc ^= data[i];
        for (int k = 0; k < 8; ++k)
            crc = (crc & 1u) ? (crc >> 1) ^ 0xEDB88320u : (crc >> 1);
    }
    return crc ^ 0xFFFFFFFFu;
}

class ColdStore {
public:
    explicit ColdStore(std::string path, SyncPolicy policy = SyncPolicy::SYNC_EACH)
        : path_(std::move(path)), policy_(policy) {
        // "ab" creates if missing and positions writes at end (append).
        fp_ = std::fopen(path_.c_str(), "ab");
        if (!fp_) {
            // A WAL that can't open its file cannot guarantee durability — fail loudly
            // rather than null-deref on the first append.
            std::fprintf(stderr, "ColdStore: cannot open WAL '%s'\n", path_.c_str());
            std::abort();
        }
    }

    ~ColdStore() {
        if (fp_) std::fclose(fp_);
    }

    ColdStore(const ColdStore&) = delete;
    ColdStore& operator=(const ColdStore&) = delete;

    // Append one auto-commit put durably (applied immediately on replay). Behavior unchanged.
    void append_put(uint64_t key, uint64_t value) { append_record(OP_WRITE, key, value); }

    // Append a transactional put — buffered on replay until a commit marker; discarded if a crash
    // leaves it without one. Written as part of the engine's commit() group.
    void append_txn_put(uint64_t key, uint64_t value) { append_record(OP_TXN_WRITE, key, value); }

    // Append a commit marker durably — replay applies all txn-puts buffered since the last commit.
    void append_commit() {
        const uint32_t magic = MATRIX_WAL_COMMIT;
        const uint32_t length = static_cast<uint32_t>(MATRIX_WAL_COMMIT_BYTES);
        const uint32_t crc = matrix_crc32(reinterpret_cast<const unsigned char*>(&magic), MATRIX_WAL_COMMIT_BYTES);
        std::fwrite(&length, sizeof(length), 1, fp_);
        std::fwrite(&crc,    sizeof(crc),    1, fp_);
        std::fwrite(&magic,  1, MATRIX_WAL_COMMIT_BYTES, fp_);
        std::fflush(fp_);
        if (policy_ == SyncPolicy::SYNC_EACH) ::fsync(::fileno(fp_));
        ++records_written_;
    }

    // Empty the log after its contents are captured in a checkpoint. Durable per SyncPolicy.
    void truncate() {
        std::fclose(fp_);
        fp_ = std::fopen(path_.c_str(), "wb");   // "wb" truncates to zero length
        if (!fp_) { std::fprintf(stderr, "ColdStore::truncate: reopen failed %s\n", path_.c_str()); std::abort(); }
        if (policy_ == SyncPolicy::SYNC_EACH) ::fsync(::fileno(fp_));
        std::fclose(fp_);
        fp_ = std::fopen(path_.c_str(), "ab");   // back to append mode
        if (!fp_) { std::fprintf(stderr, "ColdStore::truncate: reopen-append failed %s\n", path_.c_str()); std::abort(); }
        records_written_ = 0;
    }

    // Replay every intact record in append order, calling apply(key, value). Stops at the
    // first short read or CRC mismatch (torn/corrupt tail). Missing/empty file → nothing.
    template <typename Apply>
    void replay(Apply&& apply) const {
        FILE* r = std::fopen(path_.c_str(), "rb");
        if (!r) return;
        std::vector<std::pair<uint64_t, uint64_t>> txn; // txn-puts buffered since the last commit
        for (;;) {
            uint32_t length = 0;
            if (std::fread(&length, sizeof(length), 1, r) != 1) break;     // clean EOF
            if (length == 0 || length > MATRIX_WAL_MAX_RECORD) break;      // torn tail
            uint32_t crc = 0;
            if (std::fread(&crc, sizeof(crc), 1, r) != 1) break;           // torn tail
            unsigned char buf[MATRIX_WAL_MAX_RECORD];
            if (std::fread(buf, 1, length, r) != length) break;            // torn tail
            if (matrix_crc32(buf, length) != crc) break;                   // corruption
            if (length == MATRIX_WAL_PAYLOAD_BYTES) {                      // 20-byte put record
                uint32_t opcode = 0; std::memcpy(&opcode, buf + 16, 4);
                uint64_t key = 0, value = 0;
                std::memcpy(&key,   buf + 0, 8);
                std::memcpy(&value, buf + 8, 8);
                if (opcode == OP_WRITE)          apply(key, value);        // auto-commit (unchanged)
                else if (opcode == OP_TXN_WRITE) txn.emplace_back(key, value); // buffer until commit
                // other opcode at length 20: skip (forward-compat)
            } else if (length == MATRIX_WAL_COMMIT_BYTES) {               // 4-byte marker
                uint32_t magic = 0; std::memcpy(&magic, buf, 4);
                if (magic == MATRIX_WAL_COMMIT) { for (auto& kv : txn) apply(kv.first, kv.second); txn.clear(); }
                // other 4-byte record: skip (forward-compat)
            }
            // other lengths: skip (forward-compat)
        }
        std::fclose(r);   // EOF: any still-buffered txn was uncommitted -> discarded
    }

    uint64_t records_written() const { return records_written_; }
    SyncPolicy policy() const { return policy_; }   // the durability level in force (DU-5)

private:
    // Write one length-20 [key8][value8][opcode4] record durably (the put/txn-put wire form).
    void append_record(uint32_t opcode, uint64_t key, uint64_t value) {
        unsigned char payload[MATRIX_WAL_PAYLOAD_BYTES];
        std::memcpy(payload + 0,  &key,    8);
        std::memcpy(payload + 8,  &value,  8);
        std::memcpy(payload + 16, &opcode, 4);
        const uint32_t length = MATRIX_WAL_PAYLOAD_BYTES;
        const uint32_t crc = matrix_crc32(payload, MATRIX_WAL_PAYLOAD_BYTES);
        std::fwrite(&length, sizeof(length), 1, fp_);
        std::fwrite(&crc,    sizeof(crc),    1, fp_);
        std::fwrite(payload, 1, MATRIX_WAL_PAYLOAD_BYTES, fp_);
        std::fflush(fp_);
        if (policy_ == SyncPolicy::SYNC_EACH) ::fsync(::fileno(fp_));
        ++records_written_;
    }

    std::string path_;
    SyncPolicy policy_;
    FILE* fp_ = nullptr;
    uint64_t records_written_ = 0;
};


In [ ]:
%%writefile tier_manager.hpp
#pragma once

#include "cost_model.hpp"
#include "memory_model.hpp"
#include "tier_model.hpp"
#include <cstdint>
#include <cstddef>
#include <vector>
#include <unordered_map>
#include <algorithm>

// A migration the brain decided on (the future executor will move the bytes).
struct MigrationDecision {
    uint64_t column_id;
    MemorySpace from;
    MemorySpace to;
};

// The auto-tiering brain. Tracks per-column access heat and, on rebalance(), returns the
// cost-benefit migrations that lower total scan cost. Decides only — moves no bytes, does
// no I/O. Pure logic over the Increment-1 CostModel + tier_model.
class TierManager {
public:
    TierManager(CostModel cm, size_t device_capacity_bytes, size_t host_capacity_bytes)
        : cm_(cm), device_cap_(device_capacity_bytes), host_cap_(host_capacity_bytes) {}

    void register_column(uint64_t id, size_t bytes, MemorySpace initial_tier) {
        cols_[id] = Column{id, bytes, initial_tier, /*heat*/0.0, /*recent*/0,
                           /*arrived_tick*/tick_};
    }

    // TierManager (public): update a column's byte size in place after an append. Preserves tier/heat/
    // recent_bytes (unlike register_column, which resets them). No-op if id is unknown.
    void update_bytes(uint64_t id, size_t bytes) {
        auto it = cols_.find(id);
        if (it != cols_.end()) it->second.bytes = bytes;
    }

    // --- tunables (calibration targets) ---
    static constexpr double HEAT_ALPHA = 0.5;          // EWMA weight on recent accesses
    static constexpr double HYSTERESIS = 1.5;          // promote only if benefit > 1.5x cost
    static constexpr int    SCAN_HORIZON = 8;          // cap on est. future scans
    static constexpr uint64_t MIN_RESIDENCY_TICKS = 2; // anti-thrash: min ticks before evict
    static constexpr double SWAP_MARGIN = 1.5;         // swap-on-promote: candidate must be > 1.5x
                                                       // the victim's keep-value (anti-thrash band)

    // Record that `bytes` of column `id` were scanned. O(1); accumulates until rebalance.
    void record_access(uint64_t id, size_t bytes) {
        auto it = cols_.find(id);
        if (it != cols_.end()) it->second.recent_bytes += bytes;
    }

    // Global pass. (This task: age heat only. Later tasks add promotion + eviction.)
    std::vector<MigrationDecision> rebalance() {
        ++tick_;
        for (auto& kv : cols_) {
            Column& c = kv.second;
            c.heat = HEAT_ALPHA * static_cast<double>(c.recent_bytes)
                     + (1.0 - HEAT_ALPHA) * c.heat;
            c.recent_bytes = 0;
        }

        std::vector<MigrationDecision> decisions;

        // Promotion: move qualifying columns one tier toward DEVICE — but capacity-gated,
        // so the emitted plan never over-subscribes a bounded tier (the executor in Inc 4
        // could not honor an over-capacity plan). Approve best-first (highest net benefit)
        // and only if the column fits its target tier given current + already-approved
        // same-tick promotions (resident_bytes reflects approved moves as we apply them).
        std::vector<uint64_t> candidates;
        for (auto& kv : cols_) if (should_promote(kv.second)) candidates.push_back(kv.first);
        std::sort(candidates.begin(), candidates.end(),
                  [this](uint64_t a, uint64_t b) {
                      const double na = promote_net_benefit(cols_.at(a));
                      const double nb = promote_net_benefit(cols_.at(b));
                      if (na != nb) return na > nb;     // best benefit first
                      return a < b;                      // tie-break by id (determinism)
                  });
        for (uint64_t id : candidates) {
            Column& c = cols_.at(id);
            const MemorySpace to = faster_tier(c.tier);
            const size_t cap = capacity_of(to);
            const bool fits = (cap == 0) || (resident_bytes(to) + c.bytes <= cap);
            if (!fits) {
                // `to` is full. Swap-on-promote: displace a colder resident if cand is worth it.
                // (Free-space promotion is the common path above; this is the contended path.)
                try_swap_promote(c, to, cap, decisions); // does nothing if no worthwhile victim
                continue;
            }
            const MemorySpace from = c.tier;
            decisions.push_back(MigrationDecision{c.id, from, to});
            c.tier = to;
            c.arrived_tick = tick_;
        }

        // Capacity eviction: for each bounded tier over capacity, demote the lowest
        // keep_score residents (cost-benefit, not pure LRU) until it fits. Respect
        // MIN_RESIDENCY_TICKS so a freshly-arrived column isn't immediately thrashed out.
        for (MemorySpace tier : {MemorySpace::DEVICE, MemorySpace::HOST}) {
            const size_t cap = capacity_of(tier);
            if (cap == 0) continue;
            for (;;) {
                if (resident_bytes(tier) <= cap) break;
                Column* victim = nullptr;
                double worst = 1e301;
                for (auto& kv : cols_) {
                    Column& c = kv.second;
                    if (c.tier != tier) continue;
                    if (tick_ - c.arrived_tick < MIN_RESIDENCY_TICKS) continue; // anti-thrash
                    const double s = keep_score(c);
                    if (s < worst) { worst = s; victim = &c; }
                }
                if (!victim) break; // nothing evictable (all within min residency) this pass
                const MemorySpace from = victim->tier;
                const MemorySpace to = slower_tier(from);
                decisions.push_back(MigrationDecision{victim->id, from, to});
                victim->tier = to;
                victim->arrived_tick = tick_;
            }
        }

        return decisions;
    }

    MemorySpace tier_of(uint64_t id) const { return cols_.at(id).tier; }
    double heat_of(uint64_t id) const { return cols_.at(id).heat; }

    size_t resident_bytes(MemorySpace tier) const {
        size_t sum = 0;
        for (const auto& kv : cols_) if (kv.second.tier == tier) sum += kv.second.bytes;
        return sum;
    }

private:
    struct Column {
        uint64_t id;
        size_t   bytes;
        MemorySpace tier;
        double   heat;
        size_t   recent_bytes;   // accesses since last rebalance
        uint64_t arrived_tick;   // when it last landed on its current tier
    };

    static MemorySpace faster_tier(MemorySpace t) {
        if (t == MemorySpace::COLD) return MemorySpace::HOST;
        if (t == MemorySpace::HOST) return MemorySpace::DEVICE;
        return t; // DEVICE already fastest
    }

    // Heat-derived estimate of upcoming scans for a column, clamped to the horizon.
    int est_future_scans(const Column& c) const {
        if (c.bytes == 0) return 0;
        const double scans = c.heat / static_cast<double>(c.bytes); // ~scans-per-tick
        int n = static_cast<int>(scans + 0.5);
        if (n < 0) n = 0;
        if (n > SCAN_HORIZON) n = SCAN_HORIZON;
        return n;
    }

    // Benefit (scan-time saved over the horizon) and cost (one-time migration) of promoting
    // a column one tier toward DEVICE. Single source of the promotion arithmetic so
    // should_promote and promote_net_benefit can never drift. benefit==cost==0 if already
    // on the fastest tier.
    struct PromoteEval { double benefit; double cost; };
    PromoteEval promote_eval(const Column& c) const {
        const MemorySpace faster = faster_tier(c.tier);
        if (faster == c.tier) return PromoteEval{0.0, 0.0};
        const double benefit = (cm_.scan_us(c.tier, c.bytes) - cm_.scan_us(faster, c.bytes))
                               * static_cast<double>(est_future_scans(c));
        const double cost = cm_.migration_us(c.tier, faster, c.bytes);
        return PromoteEval{benefit, cost};
    }

    // Should this column be promoted one tier now? (cost-benefit + hysteresis)
    bool should_promote(const Column& c) const {
        if (faster_tier(c.tier) == c.tier) return false; // already fastest
        const PromoteEval e = promote_eval(c);
        return e.benefit > HYSTERESIS * e.cost;
    }

    // Net benefit (benefit - cost) of promoting one tier — used to rank candidates so the
    // best columns win scarce capacity first. Only meaningful when should_promote is true.
    double promote_net_benefit(const Column& c) const {
        const PromoteEval e = promote_eval(c);
        return e.benefit - e.cost;
    }

    static MemorySpace slower_tier(MemorySpace t) {
        if (t == MemorySpace::DEVICE) return MemorySpace::HOST;
        if (t == MemorySpace::HOST) return MemorySpace::COLD;
        return t; // COLD already slowest
    }

    // Usable capacity of a tier; 0 means unbounded (COLD/SSD).
    size_t capacity_of(MemorySpace t) const {
        if (t == MemorySpace::DEVICE) return device_cap_;
        if (t == MemorySpace::HOST)   return host_cap_;
        return 0; // COLD unbounded
    }

    // Cost-benefit score of keeping a column on its tier (higher = more worth keeping).
    // Lower-scored residents are evicted first when a tier is over capacity.
    double keep_score(const Column& c) const {
        const MemorySpace slower = slower_tier(c.tier);
        if (slower == c.tier) return 1e300; // COLD: never "evict" further (infinite keep)
        const double penalty = (cm_.scan_us(slower, c.bytes) - cm_.scan_us(c.tier, c.bytes))
                               * static_cast<double>(est_future_scans(c));
        return penalty; // time/heat that would be lost by demoting
    }

    // Swap-on-promote: `cand` wants to move up into tier `to` but `to` is full. If the lowest-
    // keep_score resident of `to` (a) is past MIN_RESIDENCY_TICKS and (b) is decisively colder
    // than cand (promote_eval(cand).benefit > SWAP_MARGIN * keep_score(victim)), and evicting that
    // one victim makes room, demote the victim one tier down and promote cand. Emits both
    // decisions, updates tiers/arrival ticks, returns true. Single victim by design (see spec §2).
    //
    // The comparison is gross-benefit vs gross-keep (both are horizon scan-µs deltas, so it's
    // dimensionally sound). cand's one-time migration cost is omitted here, but cand only reaches
    // this path after should_promote (benefit > HYSTERESIS*cost), so net benefit is already
    // positive; SWAP_MARGIN then demands cand be 1.5x the incumbent's keep value to displace it.
    // ponytail: single-pass + best-first. On a true 3-tier system a HOST column could be both a
    // DEVICE-promotion candidate and a swap victim this tick — cols_.at(id) re-reads the live tier
    // so there is no corruption, only a transient sub-optimal order that self-corrects next tick.
    // (Moot on the CPU build: DEVICE is inert via device_cap=1.)
    bool try_swap_promote(Column& cand, MemorySpace to, size_t cap,
                          std::vector<MigrationDecision>& decisions) {
        if (cap == 0) return false; // unbounded tier never needs to evict to fit
        Column* victim = nullptr;
        double worst = 1e301;
        for (auto& kv : cols_) {
            Column& v = kv.second;
            if (v.tier != to) continue;
            if (tick_ - v.arrived_tick < MIN_RESIDENCY_TICKS) continue; // don't evict a fresh arrival
            const double s = keep_score(v);
            if (s < worst) { worst = s; victim = &v; }
        }
        if (!victim) return false;                                   // nothing evictable
        // Margin gate: cand must be decisively more valuable than the incumbent to displace it.
        // Note by design this does NOT protect a stone-cold victim: when keep_score(victim)==0
        // (idle, heat decayed out), SWAP_MARGIN*0==0 and any positive-benefit candidate wins —
        // a worthless resident SHOULD yield. The margin only damps swaps between close-heat columns.
        if (promote_eval(cand).benefit <= SWAP_MARGIN * keep_score(*victim)) return false; // not worth it
        if (resident_bytes(to) - victim->bytes + cand.bytes > cap) return false; // one eviction won't fit cand
        const MemorySpace v_to = slower_tier(to);
        decisions.push_back(MigrationDecision{victim->id, victim->tier, v_to});
        victim->tier = v_to;
        victim->arrived_tick = tick_;
        decisions.push_back(MigrationDecision{cand.id, cand.tier, to});
        cand.tier = to;
        cand.arrived_tick = tick_;
        return true;
    }

    CostModel cm_;
    size_t device_cap_;
    size_t host_cap_;
    uint64_t tick_ = 0;
    std::unordered_map<uint64_t, Column> cols_;
};


In [ ]:
%%writefile tiered_column.hpp
#pragma once

#include "memory_model.hpp"
#include <cstdint>
#include <cstddef>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>
#include <atomic>
#include <unistd.h>   // getpid — per-process COLD-file namespace
#if defined(MATRIX_USE_CUDA)
#include <cuda_runtime.h>
#endif

// A column's bytes resident in exactly ONE tier (HOST/DEVICE/COLD). migrate_to() moves
// them by always staging through HOST: read the bytes to a host buffer, free the source
// tier, push to the destination. This collapses the 3x3 transition matrix into two halves
// and routes DEVICE<->COLD via HOST for free. The integrity invariant: checksum() is the
// same regardless of which tier holds the bytes. DEVICE requires a CUDA build.
class TieredColumn {
public:
    TieredColumn(uint64_t id, const unsigned char* bytes, size_t n)
        : id_(id), size_(n), tier_(MemorySpace::HOST), host_(bytes, bytes + n) {}

    ~TieredColumn() { free_current(); }

    TieredColumn(const TieredColumn&) = delete;
    TieredColumn& operator=(const TieredColumn&) = delete;

    MemorySpace tier() const { return tier_; }
    size_t size_bytes() const { return size_; }
    uint64_t id() const { return id_; }

    // Pointer to the resident HOST bytes for in-place reads (e.g. a scan). Valid only while
    // tier()==HOST; nullptr otherwise (the bytes live on SSD/VRAM — migrate to HOST first).
    // Zero-copy: returns the live buffer, no allocation.
    const unsigned char* host_ptr() const {
        return tier_ == MemorySpace::HOST ? host_.data() : nullptr;
    }

    // Move the column's bytes to `dest` (no-op if already there). Always stages through HOST.
    void migrate_to(MemorySpace dest) {
        if (dest == tier_) return;
        std::vector<unsigned char> staged = read_to_host(); // pull bytes from wherever we are
        free_current();                                     // release the source tier's resource
        tier_ = MemorySpace::HOST;                          // logically in HOST now (staged holds bytes)
        host_ = std::move(staged);
        if (dest == MemorySpace::HOST) return;
        if (dest == MemorySpace::COLD) {
            write_cold(host_);
            host_.clear(); host_.shrink_to_fit();
            tier_ = MemorySpace::COLD;
            return;
        }
        if (dest == MemorySpace::DEVICE) {
#if defined(MATRIX_USE_CUDA)
            push_device(host_);
            host_.clear(); host_.shrink_to_fit();
            tier_ = MemorySpace::DEVICE;
            return;
#else
            std::fprintf(stderr, "TieredColumn: DEVICE tier requires a CUDA build\n");
            std::abort();
#endif
        }
        std::fprintf(stderr, "TieredColumn: unsupported destination tier\n");
        std::abort();
    }

    // TieredColumn (public): append bytes to a HOST-resident column (grow in place). Caller ensures the
    // column is HOST (the engine borrows COLD->HOST first). size_ grows so checksum()/host_ptr() span it.
    void append_bytes(const unsigned char* b, size_t n) {
        if (tier_ != MemorySpace::HOST) {
            std::fprintf(stderr, "TieredColumn::append_bytes requires HOST tier\n"); std::abort();
        }
        host_.insert(host_.end(), b, b + n);
        size_ += n;
    }

    // Byte checksum wherever the column lives (DEVICE copies back to host first).
    uint64_t checksum() const {
        const std::vector<unsigned char> b = read_to_host();
        uint64_t sum = 0;
        for (unsigned char c : b) sum += c;
        return sum;
    }

#if defined(MATRIX_USE_CUDA)
    const void* device_ptr() const { return device_; } // valid only while tier()==DEVICE
#endif

private:
    // Process-unique serial so two columns (even with the same logical id, even in two engine
    // instances) never share a COLD file. Assigned once at construction, stable for the object's
    // life (so HOST<->COLD round-trips hit the same path).
    static uint64_t next_serial() {
        static std::atomic<uint64_t> counter{0};
        return counter.fetch_add(1, std::memory_order_relaxed);
    }

    std::string cold_path() const {
        // pid + per-object serial: unique within and across engine instances/processes.
        return std::string("/tmp/matrixdb_tcol_")
             + std::to_string(static_cast<long long>(getpid())) + "_"
             + std::to_string(serial_) + ".bin";
    }

    std::vector<unsigned char> read_to_host() const {
        if (tier_ == MemorySpace::HOST) return host_;
        if (tier_ == MemorySpace::COLD) {
            std::vector<unsigned char> b(size_);
            FILE* f = std::fopen(cold_path().c_str(), "rb");
            if (!f) { std::fprintf(stderr, "TieredColumn: cold read failed %s\n", cold_path().c_str()); std::abort(); }
            const size_t got = std::fread(b.data(), 1, size_, f);
            std::fclose(f);
            if (got != size_) { std::fprintf(stderr, "TieredColumn: short cold read\n"); std::abort(); }
            return b;
        }
#if defined(MATRIX_USE_CUDA)
        if (tier_ == MemorySpace::DEVICE) {
            std::vector<unsigned char> b(size_);
            cudaMemcpy(b.data(), device_, size_, cudaMemcpyDeviceToHost);
            return b;
        }
#endif
        std::fprintf(stderr, "TieredColumn: read from unsupported tier\n");
        std::abort();
    }

    void write_cold(const std::vector<unsigned char>& b) {
        FILE* f = std::fopen(cold_path().c_str(), "wb");
        if (!f) { std::fprintf(stderr, "TieredColumn: cold write failed %s\n", cold_path().c_str()); std::abort(); }
        const size_t wrote = std::fwrite(b.data(), 1, b.size(), f);
        std::fclose(f);
        // Fail loud on a short write (e.g. disk full) rather than leave a truncated cold
        // copy that only surfaces as corruption on read-back — symmetric with read_to_host.
        if (wrote != b.size()) {
            std::fprintf(stderr, "TieredColumn: short cold write (%zu/%zu)\n", wrote, b.size());
            std::abort();
        }
    }

    void free_current() {
        if (tier_ == MemorySpace::COLD) std::remove(cold_path().c_str());
#if defined(MATRIX_USE_CUDA)
        if (tier_ == MemorySpace::DEVICE && device_) { cudaFree(device_); device_ = nullptr; }
#endif
        // HOST: host_ vector frees itself / is overwritten by the caller.
    }

#if defined(MATRIX_USE_CUDA)
    void push_device(const std::vector<unsigned char>& b) {
        cudaMalloc(&device_, size_);
        cudaMemcpy(device_, b.data(), size_, cudaMemcpyHostToDevice);
    }
    void* device_ = nullptr;
#endif

    uint64_t id_;
    size_t size_;
    MemorySpace tier_;
    const uint64_t serial_ = next_serial();
    std::vector<unsigned char> host_;
};


In [ ]:
%%writefile migration_executor.hpp
#pragma once

#include "tier_manager.hpp"   // MigrationDecision
#include "tiered_column.hpp"
#include <cstdint>
#include <cstddef>
#include <vector>
#include <unordered_map>
#include <cstdio>

// Turns a TierManager migration plan into physical byte movement. The brain decides
// (which column goes where); the executor actuates (calls migrate_to on the real bytes).
class MigrationExecutor {
public:
    // Apply each decision by migrating the named column to its target tier. A decision whose
    // column_id is not in the registry is skipped (logged). Returns the number applied.
    size_t apply(const std::vector<MigrationDecision>& plan,
                 std::unordered_map<uint64_t, TieredColumn*>& columns) {
        size_t applied = 0;
        for (const MigrationDecision& d : plan) {
            auto it = columns.find(d.column_id);
            if (it == columns.end()) {
                std::fprintf(stderr, "MigrationExecutor: no column %llu — skipped\n",
                             static_cast<unsigned long long>(d.column_id));
                continue;
            }
            it->second->migrate_to(d.to);
            ++applied;
        }
        return applied;
    }
};


In [ ]:
%%writefile column_io.hpp
#pragma once
#include <cstdint>
#include <cstddef>
#include <cstdio>
#include <cstdlib>
#include <string>
#include <vector>

// Binary column file format v0: [u32 magic][u64 count][count × u32 data].
// ponytail: host-endian, raw little-endian-native writes — a same-machine persistence/cache, NOT
// portable across architectures (a big-endian reader would see a magic mismatch and abort, which is
// the safe failure). A byte-swapped portable encoding is the v1 upgrade if cross-machine files matter.
inline constexpr uint32_t MATRIX_COLUMN_MAGIC = 0x4D43'4F4Cu; // 'MCOL' — MatrixDB column file v0

// Write a uint32 column to `path` as [magic][u64 count][count×u32]. Fail-loud (abort) on
// open/short-write — never leave a partially/wrongly written file mistaken for valid.
inline void matrix_write_column(const std::string& path, const uint32_t* data, size_t n) {
    FILE* f = std::fopen(path.c_str(), "wb");
    if (!f) { std::fprintf(stderr, "matrix_write_column: open failed %s\n", path.c_str()); std::abort(); }
    const uint32_t magic = MATRIX_COLUMN_MAGIC;
    const uint64_t count = n;
    const bool ok = std::fwrite(&magic, sizeof magic, 1, f) == 1
                 && std::fwrite(&count, sizeof count, 1, f) == 1
                 && (n == 0 || std::fwrite(data, sizeof(uint32_t), n, f) == n);
    std::fclose(f);
    if (!ok) { std::fprintf(stderr, "matrix_write_column: short write %s\n", path.c_str()); std::abort(); }
}

// Read a column written by matrix_write_column. Fail-loud on open / bad magic / short read.
inline void matrix_read_column(const std::string& path, std::vector<uint32_t>& out) {
    FILE* f = std::fopen(path.c_str(), "rb");
    if (!f) { std::fprintf(stderr, "matrix_read_column: open failed %s\n", path.c_str()); std::abort(); }
    uint32_t magic = 0; uint64_t count = 0;
    bool ok = std::fread(&magic, sizeof magic, 1, f) == 1 && magic == MATRIX_COLUMN_MAGIC
           && std::fread(&count, sizeof count, 1, f) == 1;
    if (ok) { out.resize(static_cast<size_t>(count));
              ok = (count == 0 || std::fread(out.data(), sizeof(uint32_t), out.size(), f) == out.size()); }
    std::fclose(f);
    if (!ok) { std::fprintf(stderr, "matrix_read_column: bad/short file %s\n", path.c_str()); std::abort(); }
}

// Typed single-column file v1: [u32 magic][u32 type][u64 byte_count][byte_count raw bytes]. Carries the
// element type (0=U32, 1=I64, 2=F64) so int64/double columns persist to a single file (the v0 functions
// above stay for the u32-only path). Fail-loud, host-endian — same contract as matrix_write/read_column.
inline constexpr uint32_t MATRIX_COLUMN_MAGIC_TYPED = 0x314F'434Du; // 'MCO1' — typed single-column file v1
inline void matrix_write_column_typed(const std::string& path, const void* bytes, size_t byte_count, uint32_t type) {
    FILE* f = std::fopen(path.c_str(), "wb");
    if (!f) { std::fprintf(stderr, "matrix_write_column_typed: open failed %s\n", path.c_str()); std::abort(); }
    const uint32_t magic = MATRIX_COLUMN_MAGIC_TYPED;
    const uint64_t bc = byte_count;
    const bool ok = std::fwrite(&magic, sizeof magic, 1, f) == 1
                 && std::fwrite(&type,  sizeof type,  1, f) == 1
                 && std::fwrite(&bc,    sizeof bc,    1, f) == 1
                 && (byte_count == 0 || std::fwrite(bytes, 1, byte_count, f) == byte_count);
    std::fclose(f);
    if (!ok) { std::fprintf(stderr, "matrix_write_column_typed: short write %s\n", path.c_str()); std::abort(); }
}
// Read a typed column file; fills `out` with the raw bytes and `out_type` with the element type.
// Fail-loud on open / bad magic / short read (same as matrix_read_column).
inline void matrix_read_column_typed(const std::string& path, std::vector<unsigned char>& out, uint32_t& out_type) {
    FILE* f = std::fopen(path.c_str(), "rb");
    if (!f) { std::fprintf(stderr, "matrix_read_column_typed: open failed %s\n", path.c_str()); std::abort(); }
    uint32_t magic = 0, type = 0; uint64_t bc = 0;
    bool ok = std::fread(&magic, sizeof magic, 1, f) == 1 && magic == MATRIX_COLUMN_MAGIC_TYPED
           && std::fread(&type, sizeof type, 1, f) == 1 && std::fread(&bc, sizeof bc, 1, f) == 1;
    if (ok) { out.resize(static_cast<size_t>(bc));
              ok = (bc == 0 || std::fread(out.data(), 1, out.size(), f) == out.size()); }
    std::fclose(f);
    if (!ok) { std::fprintf(stderr, "matrix_read_column_typed: bad/short file %s\n", path.c_str()); std::abort(); }
    out_type = type;
}


In [ ]:
%%writefile csv_ingest.hpp
#pragma once
#include <cstdint>
#include <charconv>      // std::from_chars — locale-free, no-throw integer parse
#include <cstdlib>       // std::strtod  (DM-3g double CSV parse — portable across libc++ versions)
#include <cerrno>        // errno, ERANGE (DM-3g double overflow detection)
#include <fstream>
#include <string>
#include <vector>

// Parse one uint32 column (0-based col_index) out of a simple CSV file. has_header skips line 1.
// Returns true + fills `out` on success; false + empty `out` on any error (open fail / short row /
// non-integer / overflow). CSV is untrusted input, so malformed data is a graceful false, NOT abort
// (cf. column_io.hpp, which aborts on corruption of our OWN binary format). See VAL-1.
// ponytail: no quoted-field / escape handling — simple unquoted integer CSV only; quote-aware split is
// the upgrade path if real files need it.
inline bool matrix_read_csv_column(const std::string& path, size_t col_index, bool has_header,
                                   char delim, std::vector<uint32_t>& out) {
    out.clear();
    std::ifstream in(path);
    if (!in) return false;
    std::string line;
    bool first = true;
    while (std::getline(in, line)) {
        if (!line.empty() && line.back() == '\r') line.pop_back();   // tolerate CRLF
        if (has_header && first) { first = false; continue; }
        first = false;
        // Walk to the col_index-th field without allocating a vector of all fields.
        size_t start = 0, field = 0;
        while (field < col_index) {
            size_t comma = line.find(delim, start);
            if (comma == std::string::npos) { out.clear(); return false; }   // short row
            start = comma + 1;
            ++field;
        }
        size_t end = line.find(delim, start);
        if (end == std::string::npos) end = line.size();
        const char* b = line.data() + start;
        const char* e = line.data() + end;
        uint32_t value = 0;
        auto [ptr, ec] = std::from_chars(b, e, value);
        if (ec != std::errc{} || ptr != e) { out.clear(); return false; }    // non-integer, overflow, or trailing junk
        out.push_back(value);
    }
    return true;
}

// String sibling of matrix_read_csv_column: keeps the col_index-th field verbatim (no numeric parse), for
// dictionary-encoded string columns. Graceful false on open failure / short row; tolerates CRLF + a header.
inline bool matrix_read_csv_column_str(const std::string& path, size_t col_index, bool has_header,
                                       char delim, std::vector<std::string>& out) {
    out.clear();
    std::ifstream in(path);
    if (!in) return false;
    std::string line;
    bool first = true;
    while (std::getline(in, line)) {
        if (!line.empty() && line.back() == '\r') line.pop_back();   // tolerate CRLF
        if (has_header && first) { first = false; continue; }
        first = false;
        size_t start = 0, field = 0;
        while (field < col_index) {
            size_t comma = line.find(delim, start);
            if (comma == std::string::npos) { out.clear(); return false; }   // short row
            start = comma + 1;
            ++field;
        }
        size_t end = line.find(delim, start);
        if (end == std::string::npos) end = line.size();
        out.emplace_back(line.substr(start, end - start));
    }
    return true;
}

// int64 sibling of matrix_read_csv_column (DM-3g). Parses the col_index-th field as a signed 64-bit
// integer via std::from_chars (rejects trailing junk / overflow). Graceful false on any error.
inline bool matrix_read_csv_column_i64(const std::string& path, size_t col_index, bool has_header,
                                       char delim, std::vector<int64_t>& out) {
    out.clear();
    std::ifstream in(path);
    if (!in) return false;
    std::string line; bool first = true;
    while (std::getline(in, line)) {
        if (!line.empty() && line.back() == '\r') line.pop_back();
        if (has_header && first) { first = false; continue; }
        first = false;
        size_t start = 0, field = 0;
        while (field < col_index) {
            size_t comma = line.find(delim, start);
            if (comma == std::string::npos) { out.clear(); return false; }
            start = comma + 1; ++field;
        }
        size_t end = line.find(delim, start);
        if (end == std::string::npos) end = line.size();
        int64_t value = 0;
        auto [ptr, ec] = std::from_chars(line.data() + start, line.data() + end, value);
        if (ec != std::errc{} || ptr != line.data() + end) { out.clear(); return false; }
        out.push_back(value);
    }
    return true;
}

// double sibling (DM-3g). Parses via std::strtod (portable across libc++ versions), requiring the parse
// to consume exactly the field (no trailing junk) and not overflow. Graceful false on any error.
// ponytail: strtod skips leading whitespace and is C-locale '.'-decimal — fine for simple numeric CSV.
inline bool matrix_read_csv_column_f64(const std::string& path, size_t col_index, bool has_header,
                                       char delim, std::vector<double>& out) {
    out.clear();
    std::ifstream in(path);
    if (!in) return false;
    std::string line; bool first = true;
    while (std::getline(in, line)) {
        if (!line.empty() && line.back() == '\r') line.pop_back();
        if (has_header && first) { first = false; continue; }
        first = false;
        size_t start = 0, field = 0;
        while (field < col_index) {
            size_t comma = line.find(delim, start);
            if (comma == std::string::npos) { out.clear(); return false; }
            start = comma + 1; ++field;
        }
        size_t end = line.find(delim, start);
        if (end == std::string::npos) end = line.size();
        if (start == end) { out.clear(); return false; }     // empty field
        errno = 0;
        char* endptr = nullptr;
        const double value = std::strtod(line.c_str() + start, &endptr);
        if (errno == ERANGE || endptr != line.c_str() + end) { out.clear(); return false; }  // overflow / trailing junk / not a number
        out.push_back(value);
    }
    return true;
}


In [ ]:
%%writefile server.hpp
#pragma once
// Server core: a serializable request/response protocol + a stateless dispatcher over the engine.
// Transport-agnostic — a TCP/Unix-socket accept-loop is a thin adapter that shuttles these byte
// buffers (deferred; not buildable in this sandbox). Wire form is host-endian (same-machine /
// trusted-transport assumption, like column_io); a cross-arch encoding is a future upgrade.
#include "compute_mock.cpp"   // CPUMockEngine + compute.hpp (MatrixQuery/MatrixQueryStatus/MatrixAggOp)
#include <cstdint>
#include <cstring>
#include <string>
#include <unordered_map>
#include <unordered_set>
#include <vector>

enum class ReqKind : uint32_t { GET = 1, PUT = 2, QUERY = 3, HEALTH = 4, STATS = 5 };
enum class ServerStatus : uint32_t { OK = 0, ERR_BADREQUEST = 1000, ERR_FORBIDDEN = 1001, ERR_UNAUTHENTICATED = 1002 };

struct MatrixRequest {
    ReqKind  kind  = ReqKind::GET;
    uint64_t key   = 0;
    uint64_t value = 0;
    MatrixQuery query{};
};
struct MatrixResponse {
    uint32_t status = 0;
    std::vector<uint64_t> results;
};

// One audited request: principal, kind (ReqKind value, or 0 for a malformed request), the response
// status, and the target (GET/PUT key; QUERY value_col). The irrelevant target field is 0.
struct AuditEntry { uint64_t principal; uint32_t kind; uint32_t status; uint64_t key; uint64_t column; };

// Append-only audit trail of served requests (allowed, denied, and malformed alike).
class AuditLog {
public:
    void record(const AuditEntry& e) { entries_.push_back(e); }
    const std::vector<AuditEntry>& entries() const { return entries_; }
    size_t size() const { return entries_.size(); }
private:
    std::vector<AuditEntry> entries_;
};

// Per-principal authorization. Grants are additive; a principal with no grants can do nothing.
// permissive() allows everything (the no-auth / backward-compat default).
class AccessPolicy {
public:
    static AccessPolicy permissive() { AccessPolicy p; p.allow_all_ = true; return p; }
    void allow_column(uint64_t principal, uint64_t col) { cols_[principal].insert(col); } // QUERY access
    void allow_read(uint64_t principal)  { read_.insert(principal); }                      // GET
    void allow_write(uint64_t principal) { write_.insert(principal); }                     // PUT
    bool can_query(uint64_t principal, uint64_t col) const {
        if (allow_all_) return true;
        auto it = cols_.find(principal); return it != cols_.end() && it->second.count(col) != 0;
    }
    bool can_read(uint64_t principal)  const { return allow_all_ || read_.count(principal)  != 0; }
    bool can_write(uint64_t principal) const { return allow_all_ || write_.count(principal) != 0; }
private:
    bool allow_all_ = false;
    std::unordered_map<uint64_t, std::unordered_set<uint64_t>> cols_;
    std::unordered_set<uint64_t> read_, write_;
};

// SE-1 authentication: validate a bearer credential (token) -> principal id. Distinct from AccessPolicy
// (authz): authn establishes WHO a caller is; authz decides what that principal may do. The transport
// extracts the token from the connection and passes it here; an unknown/empty token authenticates to
// nobody. (Tokens are opaque strings — a real deployment stores salted hashes, not plaintext; this is the
// mechanism, not a credential store.)
class Authenticator {
public:
    void add_credential(const std::string& token, uint64_t principal) { tokens_.push_back({token, principal}); }
    // True + principal if the token is known; false (principal untouched) otherwise. Empty token -> false.
    //
    // Deliberately NOT a hash-map lookup: unordered_map::find hashes the input then compares it against
    // one bucket with std::string::operator==, which returns as soon as it finds a differing byte. That
    // leaks two things via response timing — whether a guess landed in the same bucket as any real token,
    // and (for a same-length guess) how many leading bytes matched before diverging — the textbook
    // timing side-channel on a bearer-token comparison (CWE-208). Instead this compares the presented
    // token against EVERY registered credential, every time, with no early exit on a match: total work is
    // the same whether the token matches nothing, matches the first entry, or matches the last, so an
    // attacker's response-time samples carry no signal about which (if any) token is closest to a guess.
    // O(n) in the number of registered tokens, but n is a handful of provisioned credentials checked once
    // per new connection (not a per-query hot path) — the cost is negligible next to what it closes.
    bool authenticate(const std::string& token, uint64_t& principal) const {
        if (token.empty()) return false;
        bool found = false; uint64_t matched = 0;
        for (const auto& [t, p] : tokens_) {
            if (timing_safe_equal(token, t)) { found = true; matched = p; }
        }
        if (found) principal = matched;
        return found;
    }
private:
    // Constant-time-in-content byte comparison: no branch or early return depends on *which* bytes
    // matched, only the running XOR-OR of every byte. The length check short-circuits (standard practice
    // for this primitive — matches Go's crypto/subtle.ConstantTimeCompare and Python's
    // hmac.compare_digest — token length isn't the secret being protected, its content is).
    static bool timing_safe_equal(const std::string& a, const std::string& b) {
        if (a.size() != b.size()) return false;
        uint8_t diff = 0;
        for (size_t i = 0; i < a.size(); ++i) diff |= static_cast<uint8_t>(a[i]) ^ static_cast<uint8_t>(b[i]);
        return diff == 0;
    }
    std::vector<std::pair<std::string, uint64_t>> tokens_;
};

namespace matrixsrv_detail {
    inline void put_u32(std::vector<uint8_t>& b, uint32_t v) { const uint8_t* p = reinterpret_cast<const uint8_t*>(&v); b.insert(b.end(), p, p + 4); }
    inline void put_u64(std::vector<uint8_t>& b, uint64_t v) { const uint8_t* p = reinterpret_cast<const uint8_t*>(&v); b.insert(b.end(), p, p + 8); }
    struct Reader {
        const uint8_t* p; const uint8_t* end; bool ok = true;
        bool u8(uint8_t& o)   { if (end - p < 1) { ok = false; return false; } o = *p++; return true; }
        bool u32(uint32_t& o) { if (end - p < 4) { ok = false; return false; } std::memcpy(&o, p, 4); p += 4; return true; }
        bool u64(uint64_t& o) { if (end - p < 8) { ok = false; return false; } std::memcpy(&o, p, 8); p += 8; return true; }
        bool done() const { return p == end; }
    };
}

inline std::vector<uint8_t> matrix_serialize_request(const MatrixRequest& r) {
    using namespace matrixsrv_detail;
    std::vector<uint8_t> b;
    put_u32(b, static_cast<uint32_t>(r.kind));
    put_u64(b, r.key); put_u64(b, r.value);
    put_u64(b, r.query.value_col);
    put_u32(b, static_cast<uint32_t>(r.query.agg));
    b.push_back(r.query.has_filter ? 1 : 0);
    put_u32(b, r.query.threshold);
    put_u32(b, static_cast<uint32_t>(r.query.cmp));
    put_u32(b, r.query.upper);
    put_u64(b, matrix_bit_cast<uint64_t>(r.query.lo_i64));
    put_u64(b, matrix_bit_cast<uint64_t>(r.query.hi_i64));
    put_u64(b, matrix_bit_cast<uint64_t>(r.query.lo_f64));
    put_u64(b, matrix_bit_cast<uint64_t>(r.query.hi_f64));
    b.push_back(r.query.grouped ? 1 : 0);
    put_u64(b, r.query.key_col);
    put_u32(b, r.query.num_groups);
    put_u64(b, r.query.limit);
    put_u64(b, r.query.filter_col);
    return b;
}
inline bool matrix_deserialize_request(const std::vector<uint8_t>& b, MatrixRequest& out) {
    using namespace matrixsrv_detail;
    Reader r{ b.data(), b.data() + b.size() };
    uint32_t kind = 0, agg = 0, cmp = 0; uint8_t hf = 0, gr = 0;
    uint64_t lo_i64_bits = 0, hi_i64_bits = 0, lo_f64_bits = 0, hi_f64_bits = 0;
    r.u32(kind); r.u64(out.key); r.u64(out.value);
    r.u64(out.query.value_col); r.u32(agg); r.u8(hf); r.u32(out.query.threshold);
    r.u32(cmp); r.u32(out.query.upper);
    r.u64(lo_i64_bits); r.u64(hi_i64_bits); r.u64(lo_f64_bits); r.u64(hi_f64_bits);
    r.u8(gr); r.u64(out.query.key_col); r.u32(out.query.num_groups);
    r.u64(out.query.limit); r.u64(out.query.filter_col);
    if (!r.ok || !r.done()) return false;
    if (kind < 1 || kind > 5) return false;
    if (cmp > static_cast<uint32_t>(MatrixCmp::BETWEEN)) return false;   // reject a garbage cmp, don't guess
    out.kind = static_cast<ReqKind>(kind);
    out.query.agg = static_cast<MatrixAggOp>(agg);
    out.query.has_filter = (hf != 0);
    out.query.cmp = static_cast<MatrixCmp>(cmp);
    out.query.lo_i64 = matrix_bit_cast<int64_t>(lo_i64_bits);
    out.query.hi_i64 = matrix_bit_cast<int64_t>(hi_i64_bits);
    out.query.lo_f64 = matrix_bit_cast<double>(lo_f64_bits);
    out.query.hi_f64 = matrix_bit_cast<double>(hi_f64_bits);
    out.query.grouped = (gr != 0);
    return true;
}
inline std::vector<uint8_t> matrix_serialize_response(const MatrixResponse& r) {
    using namespace matrixsrv_detail;
    std::vector<uint8_t> b;
    put_u32(b, r.status);
    put_u64(b, static_cast<uint64_t>(r.results.size()));
    for (uint64_t x : r.results) put_u64(b, x);
    return b;
}
inline bool matrix_deserialize_response(const std::vector<uint8_t>& b, MatrixResponse& out) {
    using namespace matrixsrv_detail;
    Reader r{ b.data(), b.data() + b.size() };
    uint64_t count = 0;
    if (!r.u32(out.status) || !r.u64(count)) return false;
    if (count > (1ull << 28)) return false;
    if (static_cast<uint64_t>(r.end - r.p) != count * 8) return false;
    out.results.resize(static_cast<size_t>(count));
    for (uint64_t i = 0; i < count; ++i) r.u64(out.results[i]);
    return r.ok && r.done();
}

namespace matrixsrv_detail {
// The full serve pipeline (deserialize -> authorize -> dispatch), also filling `out` with the
// audit record for this request. Returns the serialized response. Shared by all matrix_serve overloads.
inline std::vector<uint8_t> serve_core(CPUMockEngine& eng, const AccessPolicy& policy,
                                       uint64_t principal, const std::vector<uint8_t>& req_bytes,
                                       AuditEntry& out) {
    MatrixRequest req; MatrixResponse resp;
    out.principal = principal; out.kind = 0; out.key = 0; out.column = 0;
    if (!matrix_deserialize_request(req_bytes, req)) {
        resp.status = static_cast<uint32_t>(ServerStatus::ERR_BADREQUEST);
        out.status = resp.status;
        return matrix_serialize_response(resp);
    }
    out.kind = static_cast<uint32_t>(req.kind);
    out.key = req.key;
    out.column = req.query.value_col;
    bool authorized = false;
    switch (req.kind) {
        case ReqKind::GET:   authorized = policy.can_read(principal);  break;
        case ReqKind::PUT:   authorized = policy.can_write(principal); break;
        case ReqKind::QUERY: authorized = policy.can_query(principal, req.query.value_col)
                                 && (!req.query.grouped || policy.can_query(principal, req.query.key_col)); break;
        case ReqKind::HEALTH: authorized = true; break;   // operational status (no data) — always probeable
        case ReqKind::STATS:  authorized = true; break;   // operational metrics (no row data) — always readable
    }
    if (!authorized) {
        resp.status = static_cast<uint32_t>(ServerStatus::ERR_FORBIDDEN);
        out.status = resp.status;
        return matrix_serialize_response(resp);
    }
    switch (req.kind) {
        case ReqKind::GET: {
            uint64_t v = 0; if (eng.kv_get(req.key, v)) resp.results.push_back(v);
            resp.status = static_cast<uint32_t>(ServerStatus::OK);
            break;
        }
        case ReqKind::PUT: {
            eng.begin(); eng.txn_put(req.key, req.value); eng.commit();
            resp.status = static_cast<uint32_t>(ServerStatus::OK);
            break;
        }
        case ReqKind::QUERY: {
            std::vector<uint64_t> o;
            const MatrixQueryStatus st = eng.execute_query(req.query, o);
            resp.status = static_cast<uint32_t>(st);
            resp.results = std::move(o);
            break;
        }
        case ReqKind::HEALTH: {
            // Pack the health snapshot into the result vector (fixed order — the client reads by index):
            // [ready, durable, catalog_columns, host_resident_bytes, wal_records_pending, dropped_writes].
            const HealthStatus h = eng.health();
            resp.results = { h.ready ? 1u : 0u, h.durable ? 1u : 0u,
                             static_cast<uint64_t>(h.catalog_columns), static_cast<uint64_t>(h.host_resident_bytes),
                             h.wal_records_pending, h.dropped_writes };
            resp.status = static_cast<uint32_t>(ServerStatus::OK);
            break;
        }
        case ReqKind::STATS: {
            // Operational metrics over the wire (OB-5/OB-2). Fixed order — the client reads by index:
            // [cold_borrows, rebalances, migrations, catalog_columns, host_resident_bytes,
            //  cold_resident_bytes, query_count, total_query_ns, max_query_ns, p50_ns, p99_ns, version_u64].
            const EngineStats s = eng.stats();
            resp.results = { s.cold_borrows, s.rebalances, s.migrations,
                             static_cast<uint64_t>(s.catalog_columns), static_cast<uint64_t>(s.host_resident_bytes),
                             static_cast<uint64_t>(s.cold_resident_bytes), s.query_count, s.total_query_ns,
                             s.max_query_ns, eng.query_latency_percentile_ns(0.50), eng.query_latency_percentile_ns(0.99),
                             eng.version_u64() };
            resp.status = static_cast<uint32_t>(ServerStatus::OK);
            break;
        }
    }
    out.status = resp.status;
    return matrix_serialize_response(resp);
}
} // namespace matrixsrv_detail

// Authorizing dispatch (principal supplied by the authenticated caller; denied -> ERR_FORBIDDEN
// with no engine side effects; bad request -> ERR_BADREQUEST). See SE-2.
inline std::vector<uint8_t> matrix_serve(CPUMockEngine& eng, const AccessPolicy& policy,
                                         uint64_t principal, const std::vector<uint8_t>& req_bytes) {
    AuditEntry ignored;
    return matrixsrv_detail::serve_core(eng, policy, principal, req_bytes, ignored);
}
// Same, but records the request (allowed / denied / malformed) to `audit`.
inline std::vector<uint8_t> matrix_serve(CPUMockEngine& eng, const AccessPolicy& policy,
                                         uint64_t principal, const std::vector<uint8_t>& req_bytes,
                                         AuditLog& audit) {
    AuditEntry entry;
    auto resp = matrixsrv_detail::serve_core(eng, policy, principal, req_bytes, entry);
    audit.record(entry);
    return resp;
}
// No-auth convenience (backward-compat): permissive policy, anonymous principal.
inline std::vector<uint8_t> matrix_serve(CPUMockEngine& eng, const std::vector<uint8_t>& req_bytes) {
    static const AccessPolicy permissive = AccessPolicy::permissive();
    return matrix_serve(eng, permissive, /*principal=*/0, req_bytes);
}

// SE-1 authenticating dispatch: validate the bearer `token` -> principal FIRST; an unknown/empty token is
// rejected with ERR_UNAUTHENTICATED and NO engine call (authn precedes authz precedes any work). On a valid
// token, serve as that principal (so AccessPolicy still gates what it may do). The transport supplies the
// token from the connection; it is never read from the request payload (no spoofing).
inline std::vector<uint8_t> matrix_serve(CPUMockEngine& eng, const AccessPolicy& policy,
                                         const Authenticator& auth, const std::string& token,
                                         const std::vector<uint8_t>& req_bytes) {
    uint64_t principal = 0;
    if (!auth.authenticate(token, principal)) {
        MatrixResponse resp; resp.status = static_cast<uint32_t>(ServerStatus::ERR_UNAUTHENTICATED);
        return matrix_serialize_response(resp);
    }
    return matrix_serve(eng, policy, principal, req_bytes);
}


In [ ]:
%%writefile server_tcp.hpp
#pragma once
// TCP transport adapter for the serve LOGIC (NW-1). Length-prefixed wire framing:
//   request  on the wire: [u32 len][len request bytes]   (the matrix_serialize_request blob)
//   response on the wire: [u32 len][len response bytes]   (the matrix_serialize_response blob)
// matrix_serve_conn() (serve one framed request on a connected fd) is the wire protocol — runtime-tested
// over a socketpair (no bind needed). matrix_serve_tcp() is the bind/listen/accept loop — HOST-ONLY:
// loopback bind() is blocked in the build sandbox (proven), so its accept loop is compile-verified here
// and runtime-verified only on a non-sandboxed host. Multiple connections are served concurrently: each
// accepted connection gets its own thread, all sharing one ConcurrentServer (NW-2) instance so writes
// still serialize across the whole daemon, not just within one connection.
#include "server.hpp"          // matrix_serve, AccessPolicy, CPUMockEngine
#include "concurrent_server.hpp"
#include <sys/socket.h>
#include <sys/time.h>          // struct timeval (SO_RCVTIMEO)
#include <netinet/in.h>
#include <netinet/tcp.h>       // TCP_NODELAY
#include <unistd.h>
#include <cstdint>
#include <thread>
#include <vector>

// NW-5 connection management: bound how long a recv may block, so a client that connects but never sends
// (slowloris-style) can't hang the single-owner serve loop forever. After the timeout, recv fails →
// recv_all returns false → matrix_serve_conn returns false → the loop drops the stuck connection and moves
// on. ms == 0 clears the timeout (block forever, the default). Returns true on success.
inline bool matrix_set_recv_timeout(int fd, unsigned ms) {
    struct timeval tv;
    tv.tv_sec  = static_cast<long>(ms / 1000);
    tv.tv_usec = static_cast<long>((ms % 1000) * 1000);
    return ::setsockopt(fd, SOL_SOCKET, SO_RCVTIMEO, &tv, sizeof tv) == 0;
}
// Symmetric send timeout (SO_SNDTIMEO): bound how long a send may block, so a client that connects but
// never READS the response (slow-reader, the other half of the DoS) can't wedge the loop's send_all once
// the socket buffers fill. Same kernel mechanism as the recv timeout (verified there) — send returns
// EAGAIN past the deadline → send_all returns false → serve_conn returns false → the loop drops it.
inline bool matrix_set_send_timeout(int fd, unsigned ms) {
    struct timeval tv;
    tv.tv_sec  = static_cast<long>(ms / 1000);
    tv.tv_usec = static_cast<long>((ms % 1000) * 1000);
    return ::setsockopt(fd, SOL_SOCKET, SO_SNDTIMEO, &tv, sizeof tv) == 0;
}

namespace matrixsrv_detail {
// Read/write EXACTLY n bytes over a stream socket, looping over partial transfers; false on EOF/error.
inline bool recv_all(int fd, void* buf, size_t n) {
    auto* p = static_cast<unsigned char*>(buf); size_t got = 0;
    while (got < n) { const ssize_t r = ::recv(fd, p + got, n - got, 0); if (r <= 0) return false; got += static_cast<size_t>(r); }
    return true;
}
inline bool send_all(int fd, const void* buf, size_t n) {
    // MSG_NOSIGNAL (Linux): writing to a peer that closed returns EPIPE instead of raising SIGPIPE (which
    // would kill the process). macOS/BSD lacks the flag — there the caller ignores SIGPIPE / sets
    // SO_NOSIGPIPE (matrix_serve_tcp/clients should `signal(SIGPIPE, SIG_IGN)`); EPIPE then surfaces as w<0.
#ifdef MSG_NOSIGNAL
    const int flags = MSG_NOSIGNAL;
#else
    const int flags = 0;
#endif
    const auto* p = static_cast<const unsigned char*>(buf); size_t sent = 0;
    while (sent < n) { const ssize_t w = ::send(fd, p + sent, n - sent, flags); if (w <= 0) return false; sent += static_cast<size_t>(w); }
    return true;
}
} // namespace matrixsrv_detail

// Serve ONE framed request on a connected socket `fd`: read [u32 len][req], dispatch via matrix_serve
// (authorizing), write [u32 len][resp]. Returns false if the peer closed or the framing was short.
inline bool matrix_serve_conn(CPUMockEngine& eng, const AccessPolicy& policy, uint64_t principal, int fd) {
    uint32_t len = 0;
    if (!matrixsrv_detail::recv_all(fd, &len, sizeof len)) return false;
    if (len > (1u << 28)) return false;                       // sane cap — never alloc on a bogus length
    std::vector<uint8_t> req(len);
    if (len != 0 && !matrixsrv_detail::recv_all(fd, req.data(), len)) return false;
    const std::vector<uint8_t> resp = matrix_serve(eng, policy, principal, req);
    const uint32_t rlen = static_cast<uint32_t>(resp.size());
    return matrixsrv_detail::send_all(fd, &rlen, sizeof rlen)
        && (rlen == 0 || matrixsrv_detail::send_all(fd, resp.data(), rlen));
}

// Same framing as matrix_serve_conn, but dispatches through a shared ConcurrentServer (NW-2) instead of
// the bare matrix_serve — what a thread-per-connection accept loop uses so reads across connections run
// in parallel and writes serialize against the SAME mutex as every other connection, not a private one.
inline bool matrix_serve_conn_concurrent(ConcurrentServer& srv, uint64_t principal, int fd) {
    uint32_t len = 0;
    if (!matrixsrv_detail::recv_all(fd, &len, sizeof len)) return false;
    if (len > (1u << 28)) return false;                       // sane cap — never alloc on a bogus length
    std::vector<uint8_t> req(len);
    if (len != 0 && !matrixsrv_detail::recv_all(fd, req.data(), len)) return false;
    const std::vector<uint8_t> resp = srv.serve(req, principal);
    const uint32_t rlen = static_cast<uint32_t>(resp.size());
    return matrixsrv_detail::send_all(fd, &rlen, sizeof rlen)
        && (rlen == 0 || matrixsrv_detail::send_all(fd, resp.data(), rlen));
}

namespace matrixsrv_detail {
// SE-1 handshake shared by matrix_serve_conn_auth and the concurrent accept loop: read the leading
// token frame ([u32 len][token]) and authenticate it. False on a transport error or an invalid/empty
// token (principal untouched) — the caller closes the connection without serving anything.
inline bool authenticate_conn(const Authenticator& auth, int fd, uint64_t& principal) {
    uint32_t tlen = 0;
    if (!recv_all(fd, &tlen, sizeof tlen)) return false;
    if (tlen > 4096) return false;                            // sane token cap — never alloc on a bogus length
    std::string token(tlen, '\0');
    if (tlen != 0 && !recv_all(fd, token.data(), tlen)) return false;
    return auth.authenticate(token, principal);
}
} // namespace matrixsrv_detail

// SE-1 over the transport: a connection authenticates ONCE with a leading token frame, then serves that
// principal's requests for the life of the connection. Reads [u32 len][token]; authenticates → principal
// (on failure: serve nothing, return false so the caller closes the connection); then runs the normal
// matrix_serve_conn loop as that principal (AccessPolicy still gates each request). This is the realistic
// model — authenticate per connection, not per request — and the inverse of MatrixClient::authenticate.
inline bool matrix_serve_conn_auth(CPUMockEngine& eng, const AccessPolicy& policy,
                                   const Authenticator& auth, int fd) {
    uint64_t principal = 0;
    if (!matrixsrv_detail::authenticate_conn(auth, fd, principal)) return false;   // unauthenticated: serve nothing, close
    while (matrix_serve_conn(eng, policy, principal, fd)) { /* serve until the peer closes */ }
    return true;
}

// TCP accept-loop on `port`: each accepted connection is served on its own thread (detached — the daemon
// runs forever and connections are transient, so there is no orderly "join everyone" shutdown point
// today; this matches matrixdbd's existing lack of a graceful-shutdown story). All threads share ONE
// ConcurrentServer over `eng`, so its mutex actually serializes writes/tier-borrows across every
// connection, not just within one — this is what makes "many clients at once" true rather than nominal.
// HOST-ONLY (see file header — bind is blocked in the build sandbox). Returns -1 on a setup error.
// principal=0 here (no auth); a real deployment derives the principal from the connection (see
// matrix_serve_tcp_auth). Each accepted connection gets a recv/send timeout (NW-5) so one stuck client
// can't hang its own thread forever (it still can't hang any OTHER connection's thread either way).
// ponytail: no cap on concurrent connections/threads (NW-5's own remainder, unchanged by this — a
// connection limit or thread pool is future work, not silently solved here).
inline int matrix_serve_tcp(CPUMockEngine& eng, const AccessPolicy& policy, uint16_t port,
                            unsigned recv_timeout_ms = 30000) {
    const int srv_fd = ::socket(AF_INET, SOCK_STREAM, 0);
    if (srv_fd < 0) return -1;
    int yes = 1; ::setsockopt(srv_fd, SOL_SOCKET, SO_REUSEADDR, &yes, sizeof yes);
    sockaddr_in addr{}; addr.sin_family = AF_INET; addr.sin_addr.s_addr = INADDR_ANY; addr.sin_port = htons(port);
    if (::bind(srv_fd, reinterpret_cast<sockaddr*>(&addr), sizeof addr) != 0) { ::close(srv_fd); return -1; }
    if (::listen(srv_fd, 16) != 0) { ::close(srv_fd); return -1; }
    ConcurrentServer srv(eng, policy);
    for (;;) {
        const int c = ::accept(srv_fd, nullptr, nullptr);
        if (c < 0) continue;
        // Every response is 2 send() calls (len prefix, then payload); without this, Nagle holds
        // the 2nd send for the peer's delayed-ACK timer (~40ms) — see FINDINGS.md 3.7.
        int nodelay = 1; ::setsockopt(c, IPPROTO_TCP, TCP_NODELAY, &nodelay, sizeof nodelay);
        if (recv_timeout_ms) matrix_set_recv_timeout(c, recv_timeout_ms);   // NW-5: drop a stuck reader/writer
        if (recv_timeout_ms) matrix_set_send_timeout(c, recv_timeout_ms);   // both directions, same deadline
        std::thread([&srv, c] {
            while (matrix_serve_conn_concurrent(srv, /*principal=*/0, c)) { /* serve until the peer closes */ }
            ::close(c);
        }).detach();
    }
}

// Auth-aware accept loop (the realistic production entrypoint): each connection authenticates with a leading
// token frame (matrixsrv_detail::authenticate_conn) and then serves as its principal, gated by `policy`, on
// its own thread — all threads sharing one ConcurrentServer over `eng` (see matrix_serve_tcp's comment for
// why that's the part that makes concurrent connections actually work). HOST-ONLY — bind is blocked in the
// build sandbox, so this loop is compile-verified here; the per-connection auth+serve path it relies on
// is runtime-verified over a socketpair in test_server_tcp.cpp.
inline int matrix_serve_tcp_auth(CPUMockEngine& eng, const AccessPolicy& policy, const Authenticator& auth,
                                 uint16_t port, unsigned recv_timeout_ms = 30000) {
    const int srv_fd = ::socket(AF_INET, SOCK_STREAM, 0);
    if (srv_fd < 0) return -1;
    int yes = 1; ::setsockopt(srv_fd, SOL_SOCKET, SO_REUSEADDR, &yes, sizeof yes);
    sockaddr_in addr{}; addr.sin_family = AF_INET; addr.sin_addr.s_addr = INADDR_ANY; addr.sin_port = htons(port);
    if (::bind(srv_fd, reinterpret_cast<sockaddr*>(&addr), sizeof addr) != 0) { ::close(srv_fd); return -1; }
    if (::listen(srv_fd, 16) != 0) { ::close(srv_fd); return -1; }
    ConcurrentServer srv(eng, policy);
    for (;;) {
        const int c = ::accept(srv_fd, nullptr, nullptr);
        if (c < 0) continue;
        int nodelay = 1; ::setsockopt(c, IPPROTO_TCP, TCP_NODELAY, &nodelay, sizeof nodelay);  // see matrix_serve_tcp
        if (recv_timeout_ms) { matrix_set_recv_timeout(c, recv_timeout_ms); matrix_set_send_timeout(c, recv_timeout_ms); }  // NW-5
        std::thread([&srv, &auth, c] {
            uint64_t principal = 0;
            if (matrixsrv_detail::authenticate_conn(auth, c, principal)) {
                while (matrix_serve_conn_concurrent(srv, principal, c)) { /* serve until the peer closes */ }
            }
            ::close(c);
        }).detach();
    }
}


In [ ]:
%%writefile client.hpp
#pragma once
// NW-4 client driver — the app-side of the wire protocol. Wraps a connected stream socket `fd` with
// typed calls that frame a request ([u32 len][bytes]), send it, read the framed response, and
// deserialize. The inverse of matrix_serve_conn; reuses the same length-prefixed framing + recv_all/
// send_all, so it interoperates with matrix_serve_tcp on a real host. One request in flight at a time
// (matches the single-owner serving model). A transport failure (peer closed / short frame) -> false.
#include "server.hpp"
#include "server_tcp.hpp"   // matrixsrv_detail::recv_all / send_all + the framing contract
#include <cstdint>
#include <vector>

class MatrixClient {
public:
    explicit MatrixClient(int fd) : fd_(fd) {}

    // SE-1 handshake: send the leading token frame ([u32 len][token]) that authenticates this connection.
    // Call ONCE before any op when the server uses matrix_serve_conn_auth. Returns false on a transport
    // error. (The server validates the token; an invalid one makes it close the connection, so subsequent
    // ops here return false.)
    bool authenticate(const std::string& token) {
        const uint32_t len = static_cast<uint32_t>(token.size());
        if (!matrixsrv_detail::send_all(fd_, &len, sizeof len)) return false;
        return len == 0 || matrixsrv_detail::send_all(fd_, token.data(), len);
    }

    // GET key -> value. Returns false on a transport error OR a miss (no such key); true + out on a hit.
    bool get(uint64_t key, uint64_t& out) {
        MatrixRequest r; r.kind = ReqKind::GET; r.key = key;
        MatrixResponse resp;
        if (!round_trip(r, resp) || resp.status != 0 || resp.results.empty()) return false;
        out = resp.results[0];
        return true;
    }

    // PUT key=value (durably committed server-side). Returns true on OK.
    bool put(uint64_t key, uint64_t value) {
        MatrixRequest r; r.kind = ReqKind::PUT; r.key = key; r.value = value;
        MatrixResponse resp;
        return round_trip(r, resp) && resp.status == 0;
    }

    // Run an analytical query. Returns false on a transport error; otherwise true with the query's
    // MatrixQueryStatus in `st` and the result vector in `out` (so a wire failure is distinct from a
    // query-level error like ERR_UNKNOWN_COLUMN).
    bool query(const MatrixQuery& q, MatrixQueryStatus& st, std::vector<uint64_t>& out) {
        MatrixRequest r; r.kind = ReqKind::QUERY; r.query = q;
        MatrixResponse resp;
        if (!round_trip(r, resp)) return false;
        st = static_cast<MatrixQueryStatus>(resp.status);
        out = resp.results;
        return true;
    }

    // HEALTH probe -> the server's readiness snapshot. False on transport error / unexpected shape.
    bool health(HealthStatus& out) {
        MatrixRequest r; r.kind = ReqKind::HEALTH;
        MatrixResponse resp;
        if (!round_trip(r, resp) || resp.status != 0 || resp.results.size() != 6) return false;
        const auto& f = resp.results;
        out = HealthStatus{ f[0] != 0, f[1] != 0, static_cast<size_t>(f[2]),
                            static_cast<size_t>(f[3]), f[4], f[5] };
        return true;
    }

    // STATS -> the raw 12-field metrics vector (the STATS wire layout; see serve_core). False on error.
    bool stats(std::vector<uint64_t>& out) {
        MatrixRequest r; r.kind = ReqKind::STATS;
        MatrixResponse resp;
        if (!round_trip(r, resp) || resp.status != 0 || resp.results.size() != 12) return false;
        out = resp.results;
        return true;
    }

    // The server's build version (packed major<<32|minor<<16|patch), read from the STATS snapshot. 0 on error.
    uint64_t server_version() {
        std::vector<uint64_t> s;
        return stats(s) ? s[11] : 0;
    }

private:
    int fd_;
    // Frame + send the request, read + deframe the response. False on any short transfer / EOF / bad frame.
    bool round_trip(const MatrixRequest& req, MatrixResponse& resp) {
        const std::vector<uint8_t> rb = matrix_serialize_request(req);
        const uint32_t len = static_cast<uint32_t>(rb.size());
        if (!matrixsrv_detail::send_all(fd_, &len, sizeof len)) return false;
        if (len != 0 && !matrixsrv_detail::send_all(fd_, rb.data(), len)) return false;
        uint32_t rlen = 0;
        if (!matrixsrv_detail::recv_all(fd_, &rlen, sizeof rlen)) return false;
        if (rlen > (1u << 28)) return false;                       // sane cap — never alloc on a bogus length
        std::vector<uint8_t> respb(rlen);
        if (rlen != 0 && !matrixsrv_detail::recv_all(fd_, respb.data(), rlen)) return false;
        return matrix_deserialize_response(respb, resp);
    }
};


In [ ]:
%%writefile concurrent_server.hpp
#pragma once
// NW-2 concurrent serving (single-writer / many-readers). A thin dispatch layer over the existing
// matrix_serve wire protocol: it owns a std::shared_mutex and takes it shared for reads / exclusive for
// writes, so many analytical reads run in parallel while writes (and reads that need a tier borrow)
// serialize. QUERY first tries the lock-free fast path (execute_query_shared) under the shared lock; if the
// query touches a non-HOST column it returns NEEDS_EXCLUSIVE and we re-run the full matrix_serve under the
// exclusive lock. ConcurrentServer::serve() is safe to call from many threads; that is the concurrency
// unit (verified via threads under ThreadSanitizer). Wired into matrixdbd's real TCP accept loop
// (server_tcp.hpp's matrix_serve_tcp/matrix_serve_tcp_auth): one thread per accepted connection, all
// sharing one ConcurrentServer instance so the mutex actually serializes across connections, not just
// within one.
#include "server.hpp"
#include <mutex>
#include <shared_mutex>

class ConcurrentServer {
public:
    // `policy` is stored BY VALUE (AccessPolicy is a small map+two-sets value type, cheap to copy, and is
    // routinely handed to callers as a temporary via AccessPolicy::permissive()/factory-style construction
    // -- storing it as `const AccessPolicy&` here made `ConcurrentServer srv(eng, AccessPolicy::permissive())`
    // a dangling reference the instant the constructor call's full expression ended (the temporary is
    // destroyed; every later serve() call reads through a reference to freed memory). By-value ownership
    // makes that call shape simply correct instead of silently undefined -- caught by this file's own new
    // test (test_server_tcp.cpp's concurrent accept-loop test), which failed every request with
    // ERR_FORBIDDEN because the dangling policy's allow_all_ read back as garbage-false.
    ConcurrentServer(CPUMockEngine& eng, AccessPolicy policy)
        : eng_(eng), policy_(std::move(policy)) {}

    // Serve one framed request as `principal`; safe to call concurrently (each call may come from a
    // different connection/thread with its own authenticated principal). Returns the serialized response.
    std::vector<uint8_t> serve(const std::vector<uint8_t>& req_bytes, uint64_t principal = 0) {
        MatrixRequest req;
        const bool ok = matrix_deserialize_request(req_bytes, req);
        if (ok && req.kind == ReqKind::QUERY) {
            {
                std::shared_lock<std::shared_mutex> r(mu_);
                const bool authorized = policy_.can_query(principal, req.query.value_col)
                                      && (!req.query.grouped || policy_.can_query(principal, req.query.key_col));
                if (!authorized)                                   // FORBIDDEN: matrix_serve answers, no engine touch
                    return matrix_serve(eng_, policy_, principal, req_bytes);
                std::vector<uint64_t> out;
                if (eng_.execute_query_shared(req.query, out) == CPUMockEngine::ReadOutcome::SERVED) {
                    MatrixResponse resp;
                    resp.status = static_cast<uint32_t>(ServerStatus::OK);
                    resp.results = std::move(out);
                    return matrix_serialize_response(resp);
                }
            }   // authorized but not all-HOST -> escalate to the exclusive (borrowing) path
            std::unique_lock<std::shared_mutex> w(mu_);
            return matrix_serve(eng_, policy_, principal, req_bytes);
        }
        if (ok && req.kind == ReqKind::PUT) {                       // writes serialize
            std::unique_lock<std::shared_mutex> w(mu_);
            return matrix_serve(eng_, policy_, principal, req_bytes);
        }
        std::shared_lock<std::shared_mutex> r(mu_);                 // GET / HEALTH / STATS / malformed
        return matrix_serve(eng_, policy_, principal, req_bytes);
    }

private:
    CPUMockEngine& eng_;
    AccessPolicy policy_;
    std::shared_mutex mu_;
};


In [ ]:
%%writefile version.hpp
#pragma once
// BP-3 versioning: the build's version, so a running instance can report which build it is (ops correlate
// behavior ↔ build). Semantic versioning; bump on release and pair with a git tag (the release process).
// Pre-1.0 (0.x): the analytical CPU engine + tiering + durability + server protocol are feature-complete
// and verified locally; 1.0 is gated on the GPU backend (Colab) and a real network deployment.
#include <cstdint>

#define MATRIXDB_VERSION_MAJOR 0
#define MATRIXDB_VERSION_MINOR 1
#define MATRIXDB_VERSION_PATCH 0

inline constexpr const char* MATRIXDB_VERSION = "0.1.0";
inline const char* matrixdb_version() { return MATRIXDB_VERSION; }

// Packed numeric form for the wire / a version comparison: major<<32 | minor<<16 | patch. Lets a client
// read and compare the server's version without parsing a string (fits a uint64 result field).
inline constexpr uint64_t matrixdb_version_u64() {
    return (static_cast<uint64_t>(MATRIXDB_VERSION_MAJOR) << 32)
         | (static_cast<uint64_t>(MATRIXDB_VERSION_MINOR) << 16)
         |  static_cast<uint64_t>(MATRIXDB_VERSION_PATCH);
}


In [ ]:
%%writefile logging.hpp
#pragma once
// OB-1 structured logging: a tiny leveled logger with a runtime-settable threshold, so operators can
// filter/route diagnostics instead of grepping unconditional `cout`. Levels DEBUG<INFO<WARN<ERROR;
// default WARN (quiet — only problems surface). Writes "[LEVEL] msg\n" to stderr (logs are not data — keeps
// stdout clean). One global threshold (the engine is single-owner; a real deployment swaps the sink for a
// structured/JSON appender + per-subsystem levels).
#include <iostream>
#include <string>

enum class LogLevel : int { DEBUG = 0, INFO = 1, WARN = 2, ERROR = 3 };

class Log {
public:
    static void set_level(LogLevel l) { level() = l; }
    static LogLevel get_level() { return level(); }
    // True if a message at `l` passes the current threshold (would be emitted). Lets callers skip building
    // an expensive message — and lets tests assert the filter without capturing stderr.
    static bool enabled(LogLevel l) { return static_cast<int>(l) >= static_cast<int>(level()); }
    static void emit(LogLevel l, const std::string& msg) {
        if (!enabled(l)) return;
        std::cerr << prefix(l) << msg << '\n';
    }
private:
    static LogLevel& level() { static LogLevel lv = LogLevel::WARN; return lv; }
    static const char* prefix(LogLevel l) {
        switch (l) {
            case LogLevel::DEBUG: return "[DEBUG] ";
            case LogLevel::INFO:  return "[INFO] ";
            case LogLevel::WARN:  return "[WARN] ";
            default:              return "[ERROR] ";
        }
    }
};


In [ ]:
%%writefile compute_mock.cpp
#include "compute.hpp"
#include "kv_store.hpp"
#include "cold_store.hpp"
#include "migration_executor.hpp"  // MigrationExecutor + TierManager + TieredColumn + CostModel
#include "memory_model.hpp"        // MemorySpace, MemoryModel
#include "column_io.hpp"           // matrix_write_column / matrix_read_column (binary column persistence)
#include "csv_ingest.hpp"          // matrix_read_csv_column (CSV column ingest, graceful on bad input)
#include "version.hpp"             // MATRIXDB_VERSION (BP-3)
#include "logging.hpp"             // Log / LogLevel (OB-1 structured logging)
#include <unordered_map>
#include <unordered_set>
#include <map>
#include <algorithm>
#include <limits>
#include <memory>
#include <string>
#include <vector>
#include <array>
#include <atomic>
#include <bit>
#include <cassert>
#include <cctype>
#include <chrono>
#include <iostream>

/**
 * @brief CPU Mock Engine — the no-GPU fallback (Component 5: Local Sandbox).
 * Page-ownership model: bins the batch by owning page, then processes each page's
 * queries against its own slice of the store. One owner per page ⇒ no atomics, no
 * delta log, deterministic last-writer-wins. The point-op store is a real KVStore
 * (DM-1 fix: distinct colliding keys never overwrite). The CUDA engine's point-op
 * store is still the flat key&MASK array (replaced in a later increment), so CPU/GPU
 * store parity holds only while keys don't collide; CPU is the DM-1-correct path.
 */
// Engine observability snapshot: tiering activity counters + current resident-bytes gauges.
struct EngineStats {
    uint64_t cold_borrows;        // COLD->HOST borrows performed for a scan/aggregate
    uint64_t rebalances;          // rebalance() passes run (scan-driven)
    uint64_t migrations;          // migration decisions actually executed
    size_t   catalog_columns;     // # columns in the tiered catalog
    size_t   host_resident_bytes; // catalog bytes currently in RAM (TierManager's view)
    size_t   cold_resident_bytes; // catalog bytes currently on SSD
    uint64_t query_count;         // execute_query calls served (OK and ERR)
    uint64_t total_query_ns;      // summed execute_query wall-time (ns)
    uint64_t max_query_ns;        // slowest single execute_query (ns)
};

// One catalog column's metadata (DM-2 introspection): id, optional name (""), element type, row count, tier.
struct ColumnInfo { uint64_t id; std::string name; MatrixType type; size_t rows; MemorySpace tier; };

// OB-3 health/readiness snapshot for an orchestrator probe: a ready verdict + the gauges that justify it.
// `ready` is false when the engine is degraded (point-op writes dropped — the KVStore filled), so a
// liveness/readiness probe can pull the instance out of rotation and page someone.
struct HealthStatus {
    bool     ready;                 // false if degraded (dropped_writes > 0)
    bool     durable;               // a WAL is attached (committed point-ops survive restart)
    size_t   catalog_columns;       // analytical columns registered
    size_t   host_resident_bytes;   // catalog bytes currently in RAM
    uint64_t wal_records_pending;   // un-checkpointed WAL records (a restart-recovery-cost proxy)
    uint64_t dropped_writes;        // point-op writes lost to a full KVStore (the degradation signal)
};

// Tokenize a query string: identifiers/keywords/numbers, the comparison operators (> >= < <= = !=),
// and parens. Whitespace-separated; operators and parens need no surrounding spaces. (free helper)
inline std::vector<std::string> matrixparse_tokenize(const std::string& s) {
    std::vector<std::string> t;
    const size_t n = s.size();
    for (size_t i = 0; i < n; ) {
        const char c = s[i];
        if (std::isspace(static_cast<unsigned char>(c))) { ++i; continue; }
        if (c == '(' || c == ')') { t.emplace_back(1, c); ++i; continue; }
        if (c == '>' || c == '<' || c == '=' || c == '!') {
            if (i + 1 < n && s[i + 1] == '=') { t.push_back(s.substr(i, 2)); i += 2; }
            else { t.emplace_back(1, c); ++i; }
            continue;
        }
        size_t j = i;   // a run up to the next space / paren / operator char (covers names and signed numbers)
        while (j < n && !std::isspace(static_cast<unsigned char>(s[j])) && s[j] != '(' && s[j] != ')'
               && s[j] != '>' && s[j] != '<' && s[j] != '=' && s[j] != '!') ++j;
        t.push_back(s.substr(i, j - i));
        i = j;
    }
    return t;
}

#if defined(MATRIX_USE_CUDA)
// GPU-3: defined in compute_cuda.cuh (included after this file in the nvcc TU). Reduce a DEVICE/VRAM-
// resident column's bytes with the cross-checked GPU kernels; the return matches the CPU reducers'
// encoding so execute_query's DEVICE path is bit-identical to its CPU path.
inline uint64_t matrix_gpu_reduce_dev_u32(const void* d, size_t n, MatrixPredicate pred, MatrixAggOp op, bool has_filter);
inline int64_t  matrix_gpu_reduce_dev_i64(const void* d, size_t n, MatrixPredicateI64 pred, MatrixAggOp op, bool has_filter);
inline double   matrix_gpu_reduce_dev_f64(const void* d, size_t n, MatrixPredicateF64 pred, MatrixAggOp op, bool has_filter);
inline void     matrix_gpu_group_reduce_dev(const void* keys, const void* vals, size_t n, uint32_t num_groups, MatrixAggOp op, MatrixPredicate pred, bool has_filter, uint64_t* out_host);
inline void     matrix_gpu_group_reduce_dev_i64(const void* keys, const void* vals, size_t n, uint32_t num_groups, MatrixAggOp op, MatrixPredicateI64 pred, bool has_filter, uint64_t* out_host);
inline void     matrix_gpu_group_reduce_dev_f64(const void* keys, const void* vals, size_t n, uint32_t num_groups, MatrixAggOp op, MatrixPredicateF64 pred, bool has_filter, uint64_t* out_host);
#endif

class CPUMockEngine : public ComputeInterface {
public:
    // host_cap is the RAM budget (bytes) for the tiered catalog; default unbounded so the
    // existing pipeline (empty catalog) is unaffected. device_cap=1 makes the DEVICE tier
    // inert on the CPU build: scan_us() ignores gpu_available, so the brain would otherwise
    // emit HOST->DEVICE promotions the CPU executor's migrate_to(DEVICE) aborts on; a 1-byte
    // cap means no real column ever fits, so no DEVICE decision is emitted (cap==0 == unbounded).
    explicit CPUMockEngine(size_t /*worker_count*/ = 0, std::string wal_path = "",
                           size_t host_cap = SIZE_MAX, SyncPolicy sync = SyncPolicy::SYNC_EACH)
#if defined(MATRIX_USE_CUDA)
        // GPU-3: a real device budget + gpu=true cost model makes the DEVICE tier live, so hot analytical
        // columns promote to VRAM and scan_tiered_column* reduces them in place on the GPU (1 GiB << T4's 16 GB).
        : tier_mgr_(CostModel(MemoryModel::detect(true), true), /*device_cap=*/(size_t)1 << 30, host_cap)
#else
        : tier_mgr_(CostModel(MemoryModel::detect(false), false), /*device_cap=*/1, host_cap)
#endif
        , binned_(MATRIX_BATCH_MAX)
        , scan_column_(MATRIX_SCAN_COLUMN_SIZE) {
        for (size_t i = 0; i < MATRIX_SCAN_COLUMN_SIZE; ++i)
            scan_column_[i] = static_cast<uint32_t>(i); // resident analytical column
        // Durability is opt-in: with a WAL path, recover the point-op store by replaying
        // the log into kv_ (a write committed before a crash is restored here). DU-5: `sync` picks the
        // durability level — SYNC_EACH (default) fsyncs each append so a committed write survives power
        // loss; SYNC_OFF trades that for throughput (a crash may lose the unflushed tail).
        if (!wal_path.empty()) {
            checkpoint_path_ = wal_path + ".ckpt";
            load_checkpoint(checkpoint_path_);                                    // restore the last compaction (no-op if none)
            cold_store_ = std::make_unique<ColdStore>(wal_path, sync);
            cold_store_->replay([this](uint64_t k, uint64_t v){ kv_.put(k, v); }); // post-checkpoint records on top
            kv_.for_each([this](uint64_t k, uint64_t v){ key_index_[k] = v; });     // rebuild the ordered index from recovered state
        }
        std::cerr << "CPUMockEngine initialized (page-ownership, "
                  << MATRIX_PAGE_COUNT << " pages, "
                  << MATRIX_SCAN_COLUMN_SIZE << "-value scan column"
                  << (cold_store_ ? ", WAL durability ON" : "") << ")." << std::endl;
    }

    // RM-2 admission control: cap the groups a single grouped query may allocate (the out vector is
    // num_groups * 8 bytes, so this bounds one query's result memory). Default is MAX_QUERY_GROUPS (2^28,
    // ~2 GB); an operator can tighten it so one runaway GROUP BY can't OOM the box. A query above the cap
    // returns ERR_TOO_MANY_GROUPS (no allocation). Runtime-settable (a step toward OB-4 runtime config).
    void set_max_query_groups(uint32_t n) { max_query_groups_ = n; }
    uint32_t max_query_groups() const { return max_query_groups_; }

    // OB-4 runtime config: tune the heat-rebalance cadence (run the brain + executor every N tiered scans)
    // without recompiling. Default REBALANCE_EVERY (4). A smaller N re-tiers more aggressively (more
    // responsive, more migration work); a large N relaxes it. Clamped to ≥ 1 (0 → 1, rebalance every scan).
    void set_rebalance_interval(uint64_t n) { rebalance_every_ = n ? n : 1; }
    uint64_t rebalance_interval() const { return rebalance_every_; }

    // BP-3: the build version this instance is running (semver string + packed numeric form for the wire).
    const char* version() const { return matrixdb_version(); }
    uint64_t version_u64() const { return matrixdb_version_u64(); }

    // OB-1/OB-4: set the diagnostic log threshold at runtime (DEBUG<INFO<WARN<ERROR; default WARN). Global
    // (the logger is process-wide); exposed here for API discoverability alongside the other tuning knobs.
    void set_log_level(LogLevel l) { Log::set_level(l); }
    LogLevel log_level() const { return Log::get_level(); }

    // Register a uint32 analytical column into the tiered catalog (born resident in HOST).
    // id must be > 0 (0 is reserved for the legacy fixed scan column).
    void load_scan_column(uint64_t id, const uint32_t* data, size_t n) {
        assert(id != 0 && "column id 0 is reserved for the legacy fixed scan column");
        // Register once: re-registering an id would desync the catalog from the brain (and
        // could orphan a demoted column's COLD file). Callers assign distinct ids.
        assert(catalog_.find(id) == catalog_.end() && "column id already registered");
        const size_t bytes = n * sizeof(uint32_t);
        catalog_[id] = std::make_unique<TieredColumn>(
            id, reinterpret_cast<const unsigned char*>(data), bytes);
        tier_mgr_.register_column(id, bytes, MemorySpace::HOST);
    }

    // Register a signed int64 analytical column (born HOST-resident, like load_scan_column). DM-3a.
    void load_scan_column_i64(uint64_t id, const int64_t* data, size_t n) {
        assert(id != 0 && "column id 0 is reserved for the legacy fixed scan column");
        assert(catalog_.find(id) == catalog_.end() && "column id already registered");
        const size_t bytes = n * sizeof(int64_t);
        catalog_[id] = std::make_unique<TieredColumn>(id, reinterpret_cast<const unsigned char*>(data), bytes);
        tier_mgr_.register_column(id, bytes, MemorySpace::HOST);
        col_types_[id] = MatrixType::I64;
    }

    // Register a double (float64) analytical column (born HOST-resident, like load_scan_column). DM-3e.
    void load_scan_column_f64(uint64_t id, const double* data, size_t n) {
        assert(id != 0 && "column id 0 is reserved for the legacy fixed scan column");
        assert(catalog_.find(id) == catalog_.end() && "column id already registered");
        const size_t bytes = n * sizeof(double);
        catalog_[id] = std::make_unique<TieredColumn>(id, reinterpret_cast<const unsigned char*>(data), bytes);
        tier_mgr_.register_column(id, bytes, MemorySpace::HOST);
        col_types_[id] = MatrixType::F64;
    }

    // --- Minimal variable-length string columns (DM-3i) ---
    // A SELF-CONTAINED store, separate from the byte catalog_ (whose columns are fixed-width TieredColumns).
    // Supports load + row count + equality-filter count + element access — the meaningful string ops
    // (SUM/MIN/MAX don't apply numerically). ponytail: a plain id->vector<string> map — not tiered, and
    // NOT in catalog_columns()/execute_query (those stay fixed-width-typed); full integration needs the
    // catalog generalized beyond TieredColumn (XL — the upgrade path).
    void load_string_column(uint64_t id, const std::vector<std::string>& data) {
        assert(id != 0 && "column id 0 is reserved");
        string_columns_[id] = data;
    }
    size_t string_column_rows(uint64_t id) const {
        auto it = string_columns_.find(id);
        return it == string_columns_.end() ? 0 : it->second.size();
    }
    // COUNT of rows whose string equals `value` (a string WHERE col = 'literal' count).
    uint64_t string_count_where_eq(uint64_t id, const std::string& value) const {
        auto it = string_columns_.find(id);
        if (it == string_columns_.end()) return 0;
        uint64_t c = 0;
        for (const std::string& s : it->second) if (s == value) ++c;
        return c;
    }
    std::string string_column_at(uint64_t id, size_t row) const {
        auto it = string_columns_.find(id);
        assert(it != string_columns_.end() && row < it->second.size() && "string_column_at: bad id/row");
        return it->second[row];
    }

    // --- Dictionary-encoded string columns: strings as a first-class queryable type ---
    // load_string_column_dict encodes the strings to u32 codes (distinct value -> code in SORTED /
    // lexicographic order, so a code's order == its string's order) and registers the code vector as an
    // ordinary u32 catalog column (via load_scan_column) — so a string column instantly inherits the whole
    // analytical engine: tiering, snapshot, scalar+grouped aggregation, and the GPU path. The id->string
    // dictionary (code -> string, sorted) stays here for decode + range translation. "Top categories":
    // GROUP BY the code column; "WHERE s == 'x'": EQ string_encode(id,"x"); "WHERE s > 'x'": the parser maps
    // the literal to a code rank via the sorted dict; COUNT(DISTINCT) on the codes == string_dict_size.
    void load_string_column_dict(uint64_t id, const std::vector<std::string>& data) {
        assert(id != 0 && "column id 0 is reserved");
        std::vector<std::string> dict(data.begin(), data.end());   // distinct strings in sorted order ...
        std::sort(dict.begin(), dict.end());
        dict.erase(std::unique(dict.begin(), dict.end()), dict.end());
        std::unordered_map<std::string, uint32_t> enc;             // ... so code == lexicographic rank
        enc.reserve(dict.size());
        for (uint32_t c = 0; c < dict.size(); ++c) enc.emplace(dict[c], c);
        std::vector<uint32_t> codes; codes.reserve(data.size());
        for (const std::string& s : data) codes.push_back(enc[s]);
        load_scan_column(id, codes.data(), codes.size());   // codes become a first-class u32 catalog column
        string_dicts_[id] = std::move(dict);
        string_encoders_[id] = std::move(enc);
    }
    // Distinct-string count of a dict-encoded column (== its GROUP BY group count, == COUNT(DISTINCT)).
    uint32_t string_dict_size(uint64_t id) const {
        auto it = string_dicts_.find(id);
        return it == string_dicts_.end() ? 0u : static_cast<uint32_t>(it->second.size());
    }
    // Decode a code back to its string (empty if id isn't dict-encoded or code is out of range).
    std::string string_decode(uint64_t id, uint32_t code) const {
        auto it = string_dicts_.find(id);
        if (it == string_dicts_.end() || code >= it->second.size()) return std::string{};
        return it->second[code];
    }
    // Encode a literal to its code for building a filter (WHERE s == value). Returns string_dict_size(id) —
    // a code no row holds — when the value is absent, so an EQ filter on it correctly matches nothing.
    uint32_t string_encode(uint64_t id, const std::string& value) const {
        auto it = string_encoders_.find(id);
        if (it == string_encoders_.end()) return 0u;
        auto e = it->second.find(value);
        return e == it->second.end() ? static_cast<uint32_t>(it->second.size()) : e->second;
    }

    // --- Nullable columns (DM-3j) ---
    // Mark which rows of a U32 catalog column are NULL (1=null), so scalar aggregates skip them (SQL NULL
    // semantics). is_null must have one byte per row. ponytail: U32 unfiltered scalar only for this slice —
    // int64/double/filtered/grouped null-awareness is the follow-up (the maskless path is byte-identical).
    void set_column_nulls(uint64_t id, const std::vector<uint8_t>& is_null) {
        assert(catalog_has(id) && "set_column_nulls: unknown catalog column");   // any byte-catalog column (U32/I64/F64)
        assert(is_null.size() == column_rows(id) && "null mask must have one entry per row");
        null_masks_[id] = is_null;
    }

    // Append `n` more rows to an existing catalog column, growing it (DM-9). The store is no longer
    // load-once. Asserts the column exists and the element type matches; works across the COLD tier
    // (append_raw borrows COLD->HOST, grows, returns the borrow). Appended rows are immediately queryable.
    void append_to_column(uint64_t id, const uint32_t* data, size_t n) {
        assert(catalog_has(id) && column_type(id) == MatrixType::U32 && "append type must match column (u32)");
        append_raw(id, reinterpret_cast<const unsigned char*>(data), n * sizeof(uint32_t));
    }
    void append_to_column_i64(uint64_t id, const int64_t* data, size_t n) {
        assert(catalog_has(id) && column_type(id) == MatrixType::I64 && "append type must match column (int64)");
        append_raw(id, reinterpret_cast<const unsigned char*>(data), n * sizeof(int64_t));
    }
    void append_to_column_f64(uint64_t id, const double* data, size_t n) {
        assert(catalog_has(id) && column_type(id) == MatrixType::F64 && "append type must match column (double)");
        append_raw(id, reinterpret_cast<const unsigned char*>(data), n * sizeof(double));
    }

    // Inspection (tests): where the bytes actually live vs where the brain believes, the
    // HOST bytes the brain is accounting for, and a column's integrity checksum.
    MemorySpace column_tier(uint64_t id) const { return catalog_.at(id)->tier(); }
    MemorySpace manager_tier(uint64_t id) const { return tier_mgr_.tier_of(id); }
    size_t host_resident_bytes() const { return tier_mgr_.resident_bytes(MemorySpace::HOST); }
    uint64_t column_checksum(uint64_t id) const { return catalog_.at(id)->checksum(); }
    // A column's element type (DM-3a). Absent from col_types_ ⇒ U32 (every legacy/untagged column).
    MatrixType column_type(uint64_t id) const { auto it = col_types_.find(id); return it == col_types_.end() ? MatrixType::U32 : it->second; }

    // Type-aware row count of a catalog column (U32 = 4 bytes/row, I64 and F64 = 8).
    size_t column_rows(uint64_t id) const {
        const size_t w = (column_type(id) == MatrixType::I64 || column_type(id) == MatrixType::F64) ? 8 : sizeof(uint32_t);
        return catalog_.at(id)->size_bytes() / w;
    }

    // Attach (or overwrite) a name for an existing catalog column. Duplicate names: last wins for column_id.
#if defined(MATRIX_USE_CUDA)
    // GPU-3: pin a catalog column to DEVICE/VRAM now (deterministic promotion for tests/ops; the heat
    // brain also promotes hot columns under the device budget). Returns false if the id is unknown.
    bool pin_device(uint64_t col_id) {
        auto it = catalog_.find(col_id);
        if (it == catalog_.end()) return false;
        it->second->migrate_to(MemorySpace::DEVICE);
        return true;
    }
#endif

    void name_column(uint64_t id, const std::string& name) {
        assert(catalog_has(id) && "name_column: unknown column id");
        column_names_[id] = name;
        name_to_id_[name] = id;
    }
    // Resolve a column name to its id; 0 (the reserved legacy id) if no such name.
    uint64_t column_id(const std::string& name) const {
        auto it = name_to_id_.find(name);
        return it == name_to_id_.end() ? 0 : it->second;
    }
    // A column's name, or "" if unnamed.
    std::string column_name(uint64_t id) const {
        auto it = column_names_.find(id);
        return it == column_names_.end() ? std::string{} : it->second;
    }
    // List every catalog column with its metadata (id, name, type, row count, tier). Order unspecified.
    std::vector<ColumnInfo> catalog_columns() const {
        std::vector<ColumnInfo> out;
        out.reserve(catalog_.size());
        for (const auto& kv : catalog_)
            out.push_back(ColumnInfo{ kv.first, column_name(kv.first), column_type(kv.first),
                                      column_rows(kv.first), kv.second->tier() });
        return out;
    }

    // Group existing, equal-length columns into a named table (a row-aligned unit) — DM-2c. Returns false
    // (no table created) if a column id is unknown or the columns differ in row count (the table invariant).
    // Organizational schema over the named columns; queries still run per-column (a table-scoped query
    // planner is the upgrade). ponytail: stores column ids; a dropped column would dangle (no drop exists).
    bool create_table(const std::string& name, const std::vector<uint64_t>& col_ids) {
        if (col_ids.empty()) return false;
        size_t rows = 0; bool first = true;
        for (uint64_t id : col_ids) {
            if (!catalog_has(id)) return false;
            const size_t r = column_rows(id);
            if (first) { rows = r; first = false; } else if (r != rows) return false;   // row-aligned invariant
        }
        tables_[name] = col_ids;
        return true;
    }
    // The columns of a named table (in declared order), as ColumnInfo; empty if no such table.
    std::vector<ColumnInfo> table_columns(const std::string& name) const {
        std::vector<ColumnInfo> out;
        auto it = tables_.find(name);
        if (it == tables_.end()) return out;
        for (uint64_t id : it->second)
            if (catalog_has(id))
                out.push_back(ColumnInfo{ id, column_name(id), column_type(id), column_rows(id), catalog_.at(id)->tier() });
        return out;
    }
    // Names of all registered tables (order unspecified).
    std::vector<std::string> tables() const {
        std::vector<std::string> out; out.reserve(tables_.size());
        for (const auto& kv : tables_) out.push_back(kv.first);
        return out;
    }

    // Point-op read accessor (tests): true + sets out if present. Mirrors execute_batch's OP_READ.
    bool kv_get(uint64_t key, uint64_t& out) const { return kv_.get(key, out); }

    // Range scan over the point-op store: every (key, value) with lo <= key <= hi (inclusive). Order
    // unspecified (hash order — sort if needed). ponytail: O(capacity) full scan via KVStore::for_each;
    // a sorted secondary index for log-time selective ranges is the deferred upgrade (the "index" half
    // of DM-7).
    std::vector<std::pair<uint64_t, uint64_t>> kv_range(uint64_t lo, uint64_t hi) const {
        std::vector<std::pair<uint64_t, uint64_t>> out;
        kv_.for_each([&](uint64_t k, uint64_t v) { if (k >= lo && k <= hi) out.emplace_back(k, v); });
        return out;
    }

    // Sorted secondary index range scan (DM-7): every (key, value) with lo <= key <= hi, in ASCENDING
    // key order, via the ordered index — O(log n) to locate `lo` + O(result) to walk the range (not the
    // O(n) full scan of kv_range). The result is sorted by key (kv_range's is unordered).
    std::vector<std::pair<uint64_t, uint64_t>> kv_range_sorted(uint64_t lo, uint64_t hi) const {
        std::vector<std::pair<uint64_t, uint64_t>> out;
        for (auto it = key_index_.lower_bound(lo); it != key_index_.end() && it->first <= hi; ++it)
            out.emplace_back(it->first, it->second);
        return out;
    }

    // --- Atomic transactions (WAL group commit) ---
    void begin() { assert(!in_txn_ && "transaction already open"); txn_buf_.clear(); in_txn_ = true; }
    void txn_put(uint64_t key, uint64_t value) { assert(in_txn_ && "txn_put outside a transaction"); txn_buf_.emplace_back(key, value); }
    // Durably commit the buffered writes as one all-or-nothing group, then apply them.
    void commit() {
        assert(in_txn_ && "commit without begin");
        if (cold_store_) {
            for (auto& kv : txn_buf_) cold_store_->append_txn_put(kv.first, kv.second);
            cold_store_->append_commit();   // the durable, atomic commit point
        }
        for (auto& kv : txn_buf_) apply_committed_write(kv.first, kv.second);
        in_txn_ = false; ++transactions_committed_; txn_buf_.clear();
    }
    void rollback() { assert(in_txn_ && "rollback without begin"); in_txn_ = false; ++transactions_rolled_back_; txn_buf_.clear(); }
    uint64_t transactions_committed() const { return transactions_committed_; }
    uint64_t transactions_rolled_back() const { return transactions_rolled_back_; }

    // Compact the WAL: snapshot the point-op store, then truncate the log (DU-4). No-op if durability off.
    void checkpoint() {
        assert(!in_txn_ && "checkpoint inside a transaction");
        if (!cold_store_) return;
        save_checkpoint(checkpoint_path_);
        cold_store_->truncate();
        ++checkpoints_;
    }

    uint64_t checkpoints() const { return checkpoints_; }
    uint64_t wal_records() const { return cold_store_ ? cold_store_->records_written() : 0; }
    // DU-5: the durability level in force. SYNC_EACH = a committed write is fsync'd (survives power loss);
    // SYNC_OFF = buffered (faster, a crash may lose the tail). SYNC_OFF when no WAL is attached (nothing to sync).
    SyncPolicy durability_level() const { return cold_store_ ? cold_store_->policy() : SyncPolicy::SYNC_OFF; }

    // RM-4 graceful shutdown: stop cleanly and bound restart-recovery time. Rolls back any open
    // (uncommitted) transaction — its writes are correctly discarded on a clean stop — then, if a WAL is
    // attached, checkpoints (snapshot kv_ + truncate the log) so a restart replays an ~empty WAL.
    // Idempotent and safe to call before destruction; a no-op without a WAL.
    void shutdown() {
        if (in_txn_) rollback();
        if (cold_store_) checkpoint();
    }

    // Observability snapshot (counters since construction + current resident-bytes gauges).
    EngineStats stats() const {
        return EngineStats{ cold_borrows_, rebalances_, migrations_, catalog_.size(),
                            tier_mgr_.resident_bytes(MemorySpace::HOST),
                            tier_mgr_.resident_bytes(MemorySpace::COLD),
                            query_count_.load(), total_query_ns_.load(), max_query_ns_.load() };
    }

    // OB-3 health/readiness probe: a ready verdict + the gauges behind it. `ready` is false when any
    // point-op write has been dropped (KVStore full = data loss in progress) — a real degradation signal
    // an orchestrator can act on. Cheap + const, so it's safe to poll on a liveness interval.
    HealthStatus health() const {
        return HealthStatus{ store_overflows_ == 0, cold_store_ != nullptr, catalog_.size(),
                             tier_mgr_.resident_bytes(MemorySpace::HOST), wal_records(), store_overflows_ };
    }

    // OB-2b: the per-query latency histogram — log2 buckets, bucket b counts queries with latency in
    // ~[2^(b-1), 2^b) ns. Sums to stats().query_count.
    std::array<uint64_t, 40> query_latency_histogram() const {   // 40 == LAT_BUCKETS (not visible in return-type position)
        std::array<uint64_t, LAT_BUCKETS> h{};
        for (int b = 0; b < LAT_BUCKETS; ++b) h[static_cast<size_t>(b)] = latency_hist_[static_cast<size_t>(b)].load(std::memory_order_relaxed);
        return h;
    }
    // Estimate the p-th percentile (0..1) query latency in ns from the histogram (bucket-granular —
    // returns ~the bucket's upper bound). p50/p99 are the real ops latency metrics (vs mean/max).
    uint64_t query_latency_percentile_ns(double p) const {
        const uint64_t qc = query_count_.load(std::memory_order_relaxed);
        if (qc == 0) return 0;
        const uint64_t target = static_cast<uint64_t>(p * static_cast<double>(qc));
        uint64_t cum = 0;
        for (int b = 0; b < LAT_BUCKETS; ++b) {
            cum += latency_hist_[static_cast<size_t>(b)].load(std::memory_order_relaxed);
            if (cum >= target) return (b == 0) ? 0ull : (1ull << b);
        }
        return max_query_ns_.load(std::memory_order_relaxed);
    }

    // Persist a catalog column (any type) to `path` via the typed file format (borrows to HOST, returns it).
    void save_column(uint64_t id, const std::string& path) {
        TieredColumn& col = *catalog_.at(id);
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        matrix_write_column_typed(path, col.host_ptr(), col.size_bytes(), static_cast<uint32_t>(column_type(id)));
        if (home != MemorySpace::HOST) col.migrate_to(home);
    }
    // Load a typed column file into the catalog under `id` (born HOST-resident; dispatched by element type).
    void load_column_from_file(uint64_t id, const std::string& path) {
        std::vector<unsigned char> bytes; uint32_t type = 0;
        matrix_read_column_typed(path, bytes, type);
        if (type == static_cast<uint32_t>(MatrixType::I64))
            load_scan_column_i64(id, reinterpret_cast<const int64_t*>(bytes.data()), bytes.size() / sizeof(int64_t));
        else if (type == static_cast<uint32_t>(MatrixType::F64))
            load_scan_column_f64(id, reinterpret_cast<const double*>(bytes.data()), bytes.size() / sizeof(double));
        else
            load_scan_column(id, reinterpret_cast<const uint32_t*>(bytes.data()), bytes.size() / sizeof(uint32_t));
    }

    // Ingest one uint32 column from a CSV file into the catalog under `id` (born HOST-resident, like
    // load_column_from_file). Returns false (no catalog entry created) if the CSV is malformed — CSV is
    // untrusted input, so a bad file is reported, never a crash. See DM-5b / VAL-1.
    bool load_column_from_csv(uint64_t id, const std::string& path, size_t col_index,
                              bool has_header = false, char delim = ',') {
        std::vector<uint32_t> data;
        if (!matrix_read_csv_column(path, col_index, has_header, delim, data)) return false;
        load_scan_column(id, data.data(), data.size());
        return true;
    }

    // int64 / double siblings of load_column_from_csv (DM-3g). Ingest a signed-64-bit or floating-point
    // column straight from CSV. Same graceful contract: malformed CSV → false, no catalog entry, no crash.
    bool load_column_from_csv_i64(uint64_t id, const std::string& path, size_t col_index,
                                  bool has_header = false, char delim = ',') {
        std::vector<int64_t> data;
        if (!matrix_read_csv_column_i64(path, col_index, has_header, delim, data)) return false;
        load_scan_column_i64(id, data.data(), data.size());
        return true;
    }
    bool load_column_from_csv_f64(uint64_t id, const std::string& path, size_t col_index,
                                  bool has_header = false, char delim = ',') {
        std::vector<double> data;
        if (!matrix_read_csv_column_f64(path, col_index, has_header, delim, data)) return false;
        load_scan_column_f64(id, data.data(), data.size());
        return true;
    }

    // String sibling: ingest a string column straight from CSV, dictionary-encoded (see
    // load_string_column_dict). Same graceful contract: malformed CSV / open failure → false, no catalog
    // entry, no crash. Completes "strings are first-class, ingestable from CSV".
    bool load_string_column_from_csv(uint64_t id, const std::string& path, size_t col_index,
                                     bool has_header = false, char delim = ',') {
        std::vector<std::string> data;
        if (!matrix_read_csv_column_str(path, col_index, has_header, delim, data)) return false;
        load_string_column_dict(id, data);
        return true;
    }

    // Snapshot every catalog column to `path`, ATOMICALLY: write to path.tmp then rename onto path
    // (POSIX-atomic on one filesystem), so a crash mid-write leaves the prior snapshot intact — never a
    // half-written file at `path`. Returns false (no abort) on any I/O error so the caller/CLI can report it.
    // Borrows COLD columns to HOST to read, returns them. (magic 'MCA2' doubles as the format version.)
    bool save_catalog(const std::string& path) {
        const std::string tmp = path + ".tmp";
        FILE* f = std::fopen(tmp.c_str(), "wb");
        if (!f) { std::fprintf(stderr, "save_catalog: open failed %s\n", tmp.c_str()); return false; }
        const uint32_t magic = MATRIX_CATALOG_MAGIC;
        const uint64_t ncols = catalog_.size();
        bool ok = std::fwrite(&magic, sizeof magic, 1, f) == 1
               && std::fwrite(&ncols, sizeof ncols, 1, f) == 1;
        for (auto& kv : catalog_) {
            if (!ok) break;
            TieredColumn& col = *kv.second;
            const MemorySpace home = col.tier();
            if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
            const uint64_t id = kv.first;
            const uint32_t type = static_cast<uint32_t>(column_type(id));   // 0=U32, 1=I64, 2=F64
            const uint64_t count = column_rows(id);                          // type-aware row count
            const std::string nm = column_name(id);                          // optional name ("" if unnamed)
            const uint32_t namelen = static_cast<uint32_t>(nm.size());
            ok = std::fwrite(&id,      sizeof id,      1, f) == 1
              && std::fwrite(&type,    sizeof type,    1, f) == 1
              && std::fwrite(&count,   sizeof count,   1, f) == 1
              && std::fwrite(&namelen, sizeof namelen, 1, f) == 1
              && (namelen == 0 || std::fwrite(nm.data(), 1, namelen, f) == namelen)   // the name
              && (col.size_bytes() == 0
                  || std::fwrite(col.host_ptr(), 1, col.size_bytes(), f) == col.size_bytes());  // raw bytes
            if (home != MemorySpace::HOST) col.migrate_to(home);
        }
        // Trailing section: the string dictionaries (code -> string) for dict-encoded columns, so a restored
        // string column decodes (its codes ride the normal column path above). Backward-compatible — old
        // snapshots simply lack this section and load_catalog's read is EOF-tolerant.
        if (ok) {
            const uint64_t ndicts = string_dicts_.size();
            ok = std::fwrite(&ndicts, sizeof ndicts, 1, f) == 1;
            for (auto& kv : string_dicts_) {
                if (!ok) break;
                const uint64_t did = kv.first, nstr = kv.second.size();
                ok = std::fwrite(&did, sizeof did, 1, f) == 1 && std::fwrite(&nstr, sizeof nstr, 1, f) == 1;
                for (const std::string& s : kv.second) {
                    if (!ok) break;
                    const uint32_t len = static_cast<uint32_t>(s.size());
                    ok = std::fwrite(&len, sizeof len, 1, f) == 1 && (len == 0 || std::fwrite(s.data(), 1, len, f) == len);
                }
            }
        }
        // Trailing section: NULL masks (DM-3j). Additive to the dicts section above the same way dicts were
        // additive to the original per-column format — old snapshots simply lack this section and
        // load_catalog's read is EOF-tolerant. Without this, every backup()/restore() round-trip and every
        // engine restart from a saved catalog silently turned NULL rows into ordinary 0/0.0 values (a
        // different, wrong answer for the same query, with no error) — see PRODUCTION_READINESS.md DM-3j.
        if (ok) {
            const uint64_t nmasks = null_masks_.size();
            ok = std::fwrite(&nmasks, sizeof nmasks, 1, f) == 1;
            for (auto& kv : null_masks_) {
                if (!ok) break;
                const uint64_t mid = kv.first, mlen = kv.second.size();
                ok = std::fwrite(&mid, sizeof mid, 1, f) == 1 && std::fwrite(&mlen, sizeof mlen, 1, f) == 1
                  && (mlen == 0 || std::fwrite(kv.second.data(), 1, mlen, f) == mlen);
            }
        }
        std::fclose(f);
        if (!ok) { std::remove(tmp.c_str()); std::fprintf(stderr, "save_catalog: short write %s\n", tmp.c_str()); return false; }
        if (std::rename(tmp.c_str(), path.c_str()) != 0) { std::remove(tmp.c_str()); std::fprintf(stderr, "save_catalog: rename -> %s failed\n", path.c_str()); return false; }
        return true;
    }
    // Restore a snapshot into the catalog (columns land in HOST; the TierManager re-tiers under load).
    // Returns false (no abort) on a missing / corrupt / wrong-magic / short snapshot, so a caller/CLI can
    // report it instead of the process dying. (Intended for a fresh engine; a mid-load failure may leave
    // partial columns — callers open into an empty catalog.)
    bool load_catalog(const std::string& path) {
        FILE* f = std::fopen(path.c_str(), "rb");
        if (!f) { std::fprintf(stderr, "load_catalog: open failed %s\n", path.c_str()); return false; }
        uint32_t magic = 0; uint64_t ncols = 0;
        bool ok = std::fread(&magic, sizeof magic, 1, f) == 1 && magic == MATRIX_CATALOG_MAGIC
               && std::fread(&ncols, sizeof ncols, 1, f) == 1;
        for (uint64_t c = 0; ok && c < ncols; ++c) {
            uint64_t id = 0, count = 0; uint32_t type = 0, namelen = 0;
            ok = std::fread(&id, sizeof id, 1, f) == 1 && std::fread(&type, sizeof type, 1, f) == 1
              && std::fread(&count, sizeof count, 1, f) == 1 && std::fread(&namelen, sizeof namelen, 1, f) == 1;
            if (!ok) break;
            if (namelen > 4096) { ok = false; break; }   // sane bound — corrupt snapshot guard
            std::string nm(static_cast<size_t>(namelen), '\0');
            if (namelen > 0) { ok = std::fread(nm.data(), 1, namelen, f) == namelen; if (!ok) break; }
            if (type == static_cast<uint32_t>(MatrixType::I64)) {
                std::vector<int64_t> d(static_cast<size_t>(count));
                ok = (count == 0 || std::fread(d.data(), sizeof(int64_t), d.size(), f) == d.size());
                if (ok) load_scan_column_i64(id, d.data(), d.size());
            } else if (type == static_cast<uint32_t>(MatrixType::F64)) {
                std::vector<double> d(static_cast<size_t>(count));
                ok = (count == 0 || std::fread(d.data(), sizeof(double), d.size(), f) == d.size());
                if (ok) load_scan_column_f64(id, d.data(), d.size());
            } else if (type == static_cast<uint32_t>(MatrixType::U32)) {
                std::vector<uint32_t> d(static_cast<size_t>(count));
                ok = (count == 0 || std::fread(d.data(), sizeof(uint32_t), d.size(), f) == d.size());
                if (ok) load_scan_column(id, d.data(), d.size());
            } else {
                ok = false;   // unknown element type -> bad/corrupt snapshot, fail loud below
            }
            if (ok && namelen > 0) name_column(id, nm);   // restore the column's name (DM-2b)
        }
        // Trailing string-dictionaries section (EOF-tolerant: a missing section = an old snapshot, fine).
        if (ok) {
            uint64_t ndicts = 0;
            if (std::fread(&ndicts, sizeof ndicts, 1, f) == 1) {   // present -> new format
                for (uint64_t di = 0; ok && di < ndicts; ++di) {
                    uint64_t did = 0, nstr = 0;
                    ok = std::fread(&did, sizeof did, 1, f) == 1 && std::fread(&nstr, sizeof nstr, 1, f) == 1;
                    if (!ok || nstr > (1ull << 32)) { ok = false; break; }   // sane bound — corrupt guard
                    std::vector<std::string> dict; dict.reserve(static_cast<size_t>(nstr));
                    std::unordered_map<std::string, uint32_t> enc;
                    for (uint64_t si = 0; ok && si < nstr; ++si) {
                        uint32_t len = 0;
                        ok = std::fread(&len, sizeof len, 1, f) == 1;
                        if (!ok || len > (1u << 20)) { ok = false; break; }
                        std::string s(static_cast<size_t>(len), '\0');
                        if (len > 0) { ok = std::fread(s.data(), 1, len, f) == len; if (!ok) break; }
                        enc.emplace(s, static_cast<uint32_t>(dict.size()));
                        dict.push_back(std::move(s));
                    }
                    if (ok) { string_dicts_[did] = std::move(dict); string_encoders_[did] = std::move(enc); }
                }
            }
        }
        // Trailing NULL-masks section (DM-3j) — same EOF-tolerant convention as the dicts section above:
        // a missing section (an older snapshot) means no column had a mask, which is the correct default.
        if (ok) {
            uint64_t nmasks = 0;
            if (std::fread(&nmasks, sizeof nmasks, 1, f) == 1) {   // present -> new format
                for (uint64_t mi = 0; ok && mi < nmasks; ++mi) {
                    uint64_t mid = 0, mlen = 0;
                    ok = std::fread(&mid, sizeof mid, 1, f) == 1 && std::fread(&mlen, sizeof mlen, 1, f) == 1;
                    if (!ok || mlen > (1ull << 32)) { ok = false; break; }   // sane bound — corrupt guard
                    std::vector<uint8_t> mask(static_cast<size_t>(mlen));
                    if (mlen > 0) { ok = std::fread(mask.data(), 1, mlen, f) == mlen; if (!ok) break; }
                    null_masks_[mid] = std::move(mask);
                }
            }
        }
        std::fclose(f);
        if (!ok) { std::fprintf(stderr, "load_catalog: bad/short snapshot %s\n", path.c_str()); return false; }
        return true;
    }

    // Back up the whole durable state under one path prefix: <prefix>.catalog (tiered analytical columns,
    // all types) + <prefix>.kv (the point-op store). A point-in-time snapshot; reuses the existing fail-loud
    // writers. Works with or without an attached WAL (save_checkpoint snapshots the in-memory kv_).
    void backup(const std::string& prefix) {
        save_catalog(prefix + ".catalog");
        save_checkpoint(prefix + ".kv");
    }

    // Restore a backup written by backup() into THIS engine (intended for a fresh engine — loading an
    // already-registered column id asserts). Catalog-only backups restore the analytical store; a missing
    // <prefix>.kv leaves the point-op store empty (load_checkpoint returns false).
    void restore(const std::string& prefix) {
        load_catalog(prefix + ".catalog");
        load_checkpoint(prefix + ".kv");
    }

    // Snapshot kv_ atomically to `path`: write temp -> fsync -> rename (POSIX-atomic replace). Fail-loud.
    // ponytail: file data is fsync'd; the rename's own power-loss durability would need a directory fsync
    // (a pre-existing gap shared with the WAL) — the upgrade path if rename-durability matters.
    void save_checkpoint(const std::string& path) {
        const std::string tmp = path + ".tmp";
        FILE* f = std::fopen(tmp.c_str(), "wb");
        if (!f) { std::fprintf(stderr, "save_checkpoint: open failed %s\n", tmp.c_str()); std::abort(); }
        const uint32_t magic = MATRIX_CKPT_MAGIC;
        const uint64_t count = kv_.size();
        bool ok = std::fwrite(&magic, sizeof magic, 1, f) == 1
               && std::fwrite(&count, sizeof count, 1, f) == 1;
        kv_.for_each([&](uint64_t k, uint64_t v) {
            ok = ok && std::fwrite(&k, sizeof k, 1, f) == 1 && std::fwrite(&v, sizeof v, 1, f) == 1;
        });
        std::fflush(f);
        ::fsync(::fileno(f));                       // checkpoint durable BEFORE it replaces the old one
        std::fclose(f);
        if (!ok) { std::fprintf(stderr, "save_checkpoint: short write %s\n", tmp.c_str()); std::abort(); }
        if (std::rename(tmp.c_str(), path.c_str()) != 0) { std::fprintf(stderr, "save_checkpoint: rename failed\n"); std::abort(); }
    }

    // Load a checkpoint into kv_. Missing file -> false (none taken yet). Bad/short -> abort (our format).
    bool load_checkpoint(const std::string& path) {
        FILE* f = std::fopen(path.c_str(), "rb");
        if (!f) return false;
        uint32_t magic = 0; uint64_t count = 0;
        bool ok = std::fread(&magic, sizeof magic, 1, f) == 1 && magic == MATRIX_CKPT_MAGIC
               && std::fread(&count, sizeof count, 1, f) == 1;
        for (uint64_t i = 0; ok && i < count; ++i) {
            uint64_t k = 0, v = 0;
            ok = std::fread(&k, sizeof k, 1, f) == 1 && std::fread(&v, sizeof v, 1, f) == 1;
            if (ok) kv_.put(k, v);
        }
        std::fclose(f);
        if (!ok) { std::fprintf(stderr, "load_checkpoint: bad/short %s\n", path.c_str()); std::abort(); }
        return true;
    }

    // Null-mask pointer for a value column, but only if it has one AND it covers all n rows; else nullptr
    // (no masking). ponytail: a mask whose length != n is treated as absent (fail-safe to "no nulls")
    // rather than risk an out-of-bounds read in the reducer.
    const uint8_t* value_nulls(uint64_t value_id, size_t n) const {
        auto it = null_masks_.find(value_id);
        return (it != null_masks_.end() && it->second.size() == n) ? it->second.data() : nullptr;
    }

    // Distinct non-NULL value count over a typed array (helper for count_distinct). Exact hash set.
    template <typename T>
    static uint64_t distinct_count(const T* v, size_t n, const uint8_t* nulls) {
        std::unordered_set<T> seen;
        for (size_t i = 0; i < n; ++i) if (!nulls || !nulls[i]) seen.insert(v[i]);
        return seen.size();
    }

    // uint64 comparison for HAVING (a grouped aggregate can exceed u32, so this is the wide sibling of
    // matrix_pred_match). BETWEEN is inclusive [a, b].
    static bool cmp_u64(MatrixCmp c, uint64_t v, uint64_t a, uint64_t b) {
        switch (c) {
            case MatrixCmp::GE:      return v >= a;
            case MatrixCmp::LT:      return v <  a;
            case MatrixCmp::LE:      return v <= a;
            case MatrixCmp::EQ:      return v == a;
            case MatrixCmp::NE:      return v != a;
            case MatrixCmp::BETWEEN: return v >= a && v <= b;
            case MatrixCmp::GT:
            default:                 return v >  a;
        }
    }

    // GROUP BY: out[g] = aggregate (by op) of value-column rows whose key-column value == g, for
    // g in [0, num_groups). key_id and value_id are distinct catalog columns of equal length
    // (uint32). Borrows both to HOST for the reduction, then returns each to its home tier (so
    // residency stays in lockstep with the TierManager). Records access heat on both columns;
    // migration stays scan-driven (GROUP BY does not itself rebalance — see spec).
    void grouped_aggregate(uint64_t key_id, uint64_t value_id, uint32_t num_groups,
                           MatrixAggOp op, std::vector<uint64_t>& out) {
        assert(key_id != value_id && "group-by key and value must be distinct columns");
        TieredColumn& kc = *catalog_.at(key_id);
        TieredColumn& vc = *catalog_.at(value_id);
        assert(kc.size_bytes() == vc.size_bytes() && "key and value columns must be the same length");
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());

#if defined(MATRIX_USE_CUDA)
        {   // GPU-3g: both columns VRAM-resident + no null mask -> GROUP BY in VRAM, no borrow-down
            const size_t gn = kc.size_bytes() / sizeof(uint32_t);
            if (kc.tier() == MemorySpace::DEVICE && vc.tier() == MemorySpace::DEVICE && value_nulls(value_id, gn) == nullptr) {
                out.resize(num_groups);
                matrix_gpu_group_reduce_dev(kc.device_ptr(), vc.device_ptr(), gn, num_groups, op, MatrixPredicate{}, false, out.data());
                return;
            }
        }
#endif
        const MemorySpace kh = kc.tier(); if (kh != MemorySpace::HOST) { ++cold_borrows_; kc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(kc.host_ptr());
        const uint32_t* vals = reinterpret_cast<const uint32_t*>(vc.host_ptr());
        const size_t n = kc.size_bytes() / sizeof(uint32_t);
        out.resize(num_groups);   // matrix_cpu_group_reduce initializes every slot per op (MIN sentinel ≠ 0)
        matrix_cpu_group_reduce(keys, vals, n, num_groups, op, out.data(), value_nulls(value_id, n));
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);       // return borrows
        if (kh != MemorySpace::HOST) kc.migrate_to(kh);
    }

    // Inner hash equi-join on two uint32 key columns: every (left_row, right_row) with left_key[left_row]
    // == right_key[right_row]. Build a value->left-rows hash, probe with the right. Borrow-and-return both
    // columns (like grouped_aggregate). Result order unspecified; cardinality = result.size(). DM-8.
    // ponytail: builds on the LEFT side unconditionally + materializes all pairs in RAM — a planner would
    // build on the smaller side, and a huge result would need spilling; both are deferred.
    std::vector<std::pair<uint64_t, uint64_t>> hash_join(uint64_t left_key_id, uint64_t right_key_id) {
        assert(catalog_has(left_key_id) && catalog_has(right_key_id) && "hash_join: unknown column id");
        assert(column_type(left_key_id) == MatrixType::U32 && column_type(right_key_id) == MatrixType::U32
               && "hash_join: keys must be uint32 (typed-key join is deferred)");
        TieredColumn& lc = *catalog_.at(left_key_id);
        TieredColumn& rc = *catalog_.at(right_key_id);
        tier_mgr_.record_access(left_key_id, lc.size_bytes());
        tier_mgr_.record_access(right_key_id, rc.size_bytes());
        const MemorySpace lh = lc.tier(); if (lh != MemorySpace::HOST) { ++cold_borrows_; lc.migrate_to(MemorySpace::HOST); }
        const MemorySpace rh = rc.tier(); if (rh != MemorySpace::HOST) { ++cold_borrows_; rc.migrate_to(MemorySpace::HOST); }
        const uint32_t* lk = reinterpret_cast<const uint32_t*>(lc.host_ptr());
        const uint32_t* rk = reinterpret_cast<const uint32_t*>(rc.host_ptr());
        const size_t ln = lc.size_bytes() / sizeof(uint32_t);
        const size_t rn = rc.size_bytes() / sizeof(uint32_t);
        std::unordered_map<uint32_t, std::vector<uint64_t>> build;     // left value -> left rows
        for (size_t i = 0; i < ln; ++i) build[lk[i]].push_back(static_cast<uint64_t>(i));
        std::vector<std::pair<uint64_t, uint64_t>> out;
        for (size_t j = 0; j < rn; ++j) {
            auto it = build.find(rk[j]);
            if (it != build.end()) for (uint64_t i : it->second) out.emplace_back(i, static_cast<uint64_t>(j));
        }
        if (rh != MemorySpace::HOST) rc.migrate_to(rh);                // return borrows
        if (lh != MemorySpace::HOST) lc.migrate_to(lh);
        return out;
    }

    // Count an equi-join's matching pairs WITHOUT materializing them — resource-safe for huge joins
    // (addresses hash_join's materialize-all ceiling). Builds a value->count map (O(distinct) memory,
    // not O(left rows)) and sums match counts on probe. Equals hash_join(left,right).size(). DM-8/§6.
    uint64_t hash_join_count(uint64_t left_key_id, uint64_t right_key_id) {
        assert(catalog_has(left_key_id) && catalog_has(right_key_id) && "hash_join_count: unknown column id");
        assert(column_type(left_key_id) == MatrixType::U32 && column_type(right_key_id) == MatrixType::U32
               && "hash_join_count: keys must be uint32");
        TieredColumn& lc = *catalog_.at(left_key_id);
        TieredColumn& rc = *catalog_.at(right_key_id);
        tier_mgr_.record_access(left_key_id, lc.size_bytes());
        tier_mgr_.record_access(right_key_id, rc.size_bytes());
        const MemorySpace lh = lc.tier(); if (lh != MemorySpace::HOST) { ++cold_borrows_; lc.migrate_to(MemorySpace::HOST); }
        const MemorySpace rh = rc.tier(); if (rh != MemorySpace::HOST) { ++cold_borrows_; rc.migrate_to(MemorySpace::HOST); }
        const uint32_t* lk = reinterpret_cast<const uint32_t*>(lc.host_ptr());
        const uint32_t* rk = reinterpret_cast<const uint32_t*>(rc.host_ptr());
        const size_t ln = lc.size_bytes() / sizeof(uint32_t);
        const size_t rn = rc.size_bytes() / sizeof(uint32_t);
        std::unordered_map<uint32_t, uint64_t> build;                 // left value -> # of left rows
        for (size_t i = 0; i < ln; ++i) ++build[lk[i]];
        uint64_t pairs = 0;
        for (size_t j = 0; j < rn; ++j) { auto it = build.find(rk[j]); if (it != build.end()) pairs += it->second; }
        if (rh != MemorySpace::HOST) rc.migrate_to(rh);
        if (lh != MemorySpace::HOST) lc.migrate_to(lh);
        return pairs;
    }

    // Inner hash equi-join on two int64 key columns (the typed sibling of hash_join — DM-8c). Same
    // build-left / probe-right shape over int64 values; returns matching (left_row, right_row) pairs.
    // (double-key joins are intentionally unsupported — exact float equality is semantically fraught.)
    std::vector<std::pair<uint64_t, uint64_t>> hash_join_i64(uint64_t left_key_id, uint64_t right_key_id) {
        assert(catalog_has(left_key_id) && catalog_has(right_key_id) && "hash_join_i64: unknown column id");
        assert(column_type(left_key_id) == MatrixType::I64 && column_type(right_key_id) == MatrixType::I64
               && "hash_join_i64: keys must be int64");
        TieredColumn& lc = *catalog_.at(left_key_id);
        TieredColumn& rc = *catalog_.at(right_key_id);
        tier_mgr_.record_access(left_key_id, lc.size_bytes());
        tier_mgr_.record_access(right_key_id, rc.size_bytes());
        const MemorySpace lh = lc.tier(); if (lh != MemorySpace::HOST) { ++cold_borrows_; lc.migrate_to(MemorySpace::HOST); }
        const MemorySpace rh = rc.tier(); if (rh != MemorySpace::HOST) { ++cold_borrows_; rc.migrate_to(MemorySpace::HOST); }
        const int64_t* lk = reinterpret_cast<const int64_t*>(lc.host_ptr());
        const int64_t* rk = reinterpret_cast<const int64_t*>(rc.host_ptr());
        const size_t ln = lc.size_bytes() / sizeof(int64_t);
        const size_t rn = rc.size_bytes() / sizeof(int64_t);
        std::unordered_map<int64_t, std::vector<uint64_t>> build;     // left value -> left rows
        for (size_t i = 0; i < ln; ++i) build[lk[i]].push_back(static_cast<uint64_t>(i));
        std::vector<std::pair<uint64_t, uint64_t>> out;
        for (size_t j = 0; j < rn; ++j) {
            auto it = build.find(rk[j]);
            if (it != build.end()) for (uint64_t i : it->second) out.emplace_back(i, static_cast<uint64_t>(j));
        }
        if (rh != MemorySpace::HOST) rc.migrate_to(rh);
        if (lh != MemorySpace::HOST) lc.migrate_to(lh);
        return out;
    }

    // Gather a column's values at the given row indices (the "project" step — e.g. over a join's matched
    // rows). Returns one value per index, in index order, as a uint64: a U32 value zero-extended; an
    // I64/F64 value as its bit pattern (caller reads via static_cast<int64_t> / std::bit_cast<double>,
    // per column_type(col_id) — same convention as execute_query). Borrows COLD->HOST and returns it.
    std::vector<uint64_t> gather(uint64_t col_id, const std::vector<uint64_t>& rows) {
        assert(catalog_has(col_id) && "gather: unknown column id");
        TieredColumn& col = *catalog_.at(col_id);
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const bool wide = (column_type(col_id) != MatrixType::U32);   // I64/F64 are 8 bytes, U32 is 4
        const size_t width = wide ? 8 : 4;
        const size_t n = col.size_bytes() / width;
        const unsigned char* base = col.host_ptr();
        std::vector<uint64_t> out; out.reserve(rows.size());
        for (uint64_t r : rows) {
            assert(r < n && "gather: row index out of range");
            uint64_t v = 0;
            if (wide) std::memcpy(&v, base + r * 8, 8);
            else { uint32_t u = 0; std::memcpy(&u, base + r * 4, 4); v = u; }
            out.push_back(v);
        }
        if (home != MemorySpace::HOST) col.migrate_to(home);
        return out;
    }

    // Row indices where the u32 filter column satisfies the predicate (capped at `limit`, 0 = no cap) —
    // the filter primitive behind projection. Borrow-and-return like the scans.
    std::vector<uint64_t> matching_rows(uint64_t filter_col, const MatrixPredicate& pred, uint64_t limit) {
        TieredColumn& fc = *catalog_.at(filter_col);
        tier_mgr_.record_access(filter_col, fc.size_bytes());
        const MemorySpace home = fc.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; fc.migrate_to(MemorySpace::HOST); }
        const uint32_t* f = reinterpret_cast<const uint32_t*>(fc.host_ptr());
        const size_t n = fc.size_bytes() / sizeof(uint32_t);
        std::vector<uint64_t> rows;
        for (size_t i = 0; i < n; ++i)
            if (matrix_pred_match(f[i], pred)) { rows.push_back(i); if (limit && rows.size() >= limit) break; }
        if (home != MemorySpace::HOST) fc.migrate_to(home);
        return rows;
    }
    // Projection: the values of `col_id` for the rows matching an optional u32 filter (else all rows), capped
    // at `limit` (0 = no cap) — the data behind SELECT col [WHERE fcol <pred>] [LIMIT n]. Values are encoded
    // like gather (u32 zero-extended; i64/f64 as their 8-byte bit pattern). Composes matching_rows + gather.
    std::vector<uint64_t> project(uint64_t col_id, bool has_filter, uint64_t filter_col, const MatrixPredicate& pred, uint64_t limit) {
        std::vector<uint64_t> rows;
        if (has_filter) {
            rows = matching_rows(filter_col, pred, limit);
        } else {
            const size_t n = column_rows(col_id);
            const size_t cap = (limit && static_cast<size_t>(limit) < n) ? static_cast<size_t>(limit) : n;
            rows.reserve(cap);
            for (size_t i = 0; i < cap; ++i) rows.push_back(i);
        }
        return gather(col_id, rows);
    }
    // GROUP BY key WHERE <predicate> — same double borrow-and-return as grouped_aggregate_where.
    void grouped_aggregate_pred(uint64_t key_id, uint64_t value_id, uint32_t num_groups,
                                MatrixAggOp op, const MatrixPredicate& pred, std::vector<uint64_t>& out) {
        assert(key_id != value_id && "group-by key and value must be distinct columns");
        TieredColumn& kc = *catalog_.at(key_id);
        TieredColumn& vc = *catalog_.at(value_id);
        assert(kc.size_bytes() == vc.size_bytes() && "key and value columns must be the same length");
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());
#if defined(MATRIX_USE_CUDA)
        {   // GPU-3g: both columns VRAM-resident + no null mask -> filtered GROUP BY in VRAM, no borrow-down
            const size_t gn = kc.size_bytes() / sizeof(uint32_t);
            if (kc.tier() == MemorySpace::DEVICE && vc.tier() == MemorySpace::DEVICE && value_nulls(value_id, gn) == nullptr) {
                out.resize(num_groups);
                matrix_gpu_group_reduce_dev(kc.device_ptr(), vc.device_ptr(), gn, num_groups, op, pred, true, out.data());
                return;
            }
        }
#endif
        const MemorySpace kh = kc.tier(); if (kh != MemorySpace::HOST) { ++cold_borrows_; kc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(kc.host_ptr());
        const uint32_t* vals = reinterpret_cast<const uint32_t*>(vc.host_ptr());
        const size_t n = kc.size_bytes() / sizeof(uint32_t);
        out.resize(num_groups);
        matrix_cpu_group_reduce_pred(keys, vals, n, num_groups, op, pred, out.data(), value_nulls(value_id, n));
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);
        if (kh != MemorySpace::HOST) kc.migrate_to(kh);
    }

    // GROUP BY key WHERE value > threshold (filtered grouped aggregate). Same contract and double
    // borrow-and-return as grouped_aggregate; only rows with value > threshold contribute.
    void grouped_aggregate_where(uint64_t key_id, uint64_t value_id, uint32_t num_groups,
                                 MatrixAggOp op, uint32_t threshold, std::vector<uint64_t>& out) {
        grouped_aggregate_pred(key_id, value_id, num_groups, op, MatrixPredicate{MatrixCmp::GT, threshold, 0}, out);
    }

    // GROUP BY a uint32 key over an int64 value column (DM-3c). Double borrow-and-return like
    // grouped_aggregate; out holds int64 group aggregates as uint64 bit-patterns. No rebalance (GROUP BY
    // is not scan-driven, matching grouped_aggregate).
    void grouped_aggregate_i64(uint64_t key_id, uint64_t value_id, uint32_t num_groups, MatrixAggOp op,
                               const MatrixPredicateI64& pred, bool has_filter, std::vector<uint64_t>& out) {
        TieredColumn& kc = *catalog_.at(key_id);
        TieredColumn& vc = *catalog_.at(value_id);
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());
#if defined(MATRIX_USE_CUDA)
        {   // GPU-3g: u32 key + i64 value both VRAM-resident + no null mask -> GROUP BY in VRAM
            const size_t gn = vc.size_bytes() / sizeof(int64_t);
            if (kc.tier() == MemorySpace::DEVICE && vc.tier() == MemorySpace::DEVICE && value_nulls(value_id, gn) == nullptr) {
                out.resize(num_groups);
                matrix_gpu_group_reduce_dev_i64(kc.device_ptr(), vc.device_ptr(), gn, num_groups, op, pred, has_filter, out.data());
                return;
            }
        }
#endif
        const MemorySpace kh = kc.tier(); if (kh != MemorySpace::HOST) { ++cold_borrows_; kc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(kc.host_ptr());
        const int64_t*  vals = reinterpret_cast<const int64_t*>(vc.host_ptr());
        const size_t n = vc.size_bytes() / sizeof(int64_t);
        std::vector<int64_t> tmp(num_groups);
        const uint8_t* nulls = value_nulls(value_id, n);
        if (has_filter) matrix_cpu_group_reduce_i64_pred(keys, vals, n, num_groups, op, pred, tmp.data(), nulls);
        else            matrix_cpu_group_reduce_i64(keys, vals, n, num_groups, op, tmp.data(), nulls);
        out.resize(num_groups);
        for (uint32_t g = 0; g < num_groups; ++g) out[g] = static_cast<uint64_t>(tmp[g]);
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);
        if (kh != MemorySpace::HOST) kc.migrate_to(kh);
    }

    // GROUP BY a uint32 key over a double value column (DM-3f). Double borrow-and-return like
    // grouped_aggregate; out holds double group aggregates as uint64 bit-patterns. No rebalance (GROUP BY
    // is not scan-driven, matching grouped_aggregate).
    void grouped_aggregate_f64(uint64_t key_id, uint64_t value_id, uint32_t num_groups, MatrixAggOp op,
                               const MatrixPredicateF64& pred, bool has_filter, std::vector<uint64_t>& out) {
        TieredColumn& kc = *catalog_.at(key_id);
        TieredColumn& vc = *catalog_.at(value_id);
        tier_mgr_.record_access(key_id, kc.size_bytes());
        tier_mgr_.record_access(value_id, vc.size_bytes());
#if defined(MATRIX_USE_CUDA)
        {   // GPU-3g: u32 key + f64 value both VRAM-resident + no null mask -> GROUP BY in VRAM
            const size_t gn = vc.size_bytes() / sizeof(double);
            if (kc.tier() == MemorySpace::DEVICE && vc.tier() == MemorySpace::DEVICE && value_nulls(value_id, gn) == nullptr) {
                out.resize(num_groups);
                matrix_gpu_group_reduce_dev_f64(kc.device_ptr(), vc.device_ptr(), gn, num_groups, op, pred, has_filter, out.data());
                return;
            }
        }
#endif
        const MemorySpace kh = kc.tier(); if (kh != MemorySpace::HOST) { ++cold_borrows_; kc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(kc.host_ptr());
        const double*   vals = reinterpret_cast<const double*>(vc.host_ptr());
        const size_t n = vc.size_bytes() / sizeof(double);
        std::vector<double> tmp(num_groups);
        const uint8_t* nulls = value_nulls(value_id, n);
        if (has_filter) matrix_cpu_group_reduce_f64_pred(keys, vals, n, num_groups, op, pred, tmp.data(), nulls);
        else            matrix_cpu_group_reduce_f64(keys, vals, n, num_groups, op, tmp.data(), nulls);
        out.resize(num_groups);
        for (uint32_t g = 0; g < num_groups; ++g) out[g] = matrix_bit_cast<uint64_t>(tmp[g]);
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);
        if (kh != MemorySpace::HOST) kc.migrate_to(kh);
    }

    // Unified analytical query over catalog columns. Validates input at the boundary and returns a
    // status (never crashes on caller input); on any ERR, out is cleared. Scalar -> out[0];
    // grouped -> out[0..num_groups). Catalog columns only (the legacy id-0 fixed column is the
    // benchmark fixture, not a query target). Public API: times every call (OB-2) — see the
    // execute_query_impl below for the body; this thin wrapper records latency on all return paths.
    MatrixQueryStatus execute_query(const MatrixQuery& q, std::vector<uint64_t>& out) {
        const auto t0 = std::chrono::steady_clock::now();
        const MatrixQueryStatus st = execute_query_impl(q, out);
        const uint64_t ns = static_cast<uint64_t>(
            std::chrono::duration_cast<std::chrono::nanoseconds>(std::chrono::steady_clock::now() - t0).count());
        record_query_latency(ns);   // atomic read-path stats; shared with execute_query_shared
        return st;
    }

    // NW-2 concurrency: outcome of the lock-free read fast path (below).
    enum class ReadOutcome { SERVED, NEEDS_EXCLUSIVE };

    // Concurrent read fast-path: serve a QUERY with NO tier side effects (no borrow / heat / rebalance), so
    // it is safe to run under a shared lock alongside other readers. Serves only fully-valid, all-HOST-
    // resident, non-null-masked queries it can reduce directly over host_ptr() via the same matrix_cpu_*
    // reducers as the exclusive path; anything else returns NEEDS_EXCLUSIVE and the caller re-runs it under
    // the exclusive lock via execute_query (which borrows, validates, and reports proper error statuses).
    // The SERVED==execute_query equivalence is asserted in test_concurrent_serving.cpp.
    ReadOutcome execute_query_shared(const MatrixQuery& q, std::vector<uint64_t>& out) {
        const auto t0 = std::chrono::steady_clock::now();
        const ReadOutcome r = execute_query_shared_impl(q, out);
        if (r == ReadOutcome::SERVED)
            record_query_latency(static_cast<uint64_t>(std::chrono::duration_cast<std::chrono::nanoseconds>(
                std::chrono::steady_clock::now() - t0).count()));   // NEEDS_EXCLUSIVE: the exclusive re-run records its own latency
        return r;
    }
    // True iff `id` is a registered, HOST-resident column (so a shared-lock read touches only immutable bytes).
    bool host_resident(uint64_t id) const {
        auto it = catalog_.find(id);
        return it != catalog_.end() && it->second->tier() == MemorySpace::HOST;
    }
    ReadOutcome execute_query_shared_impl(const MatrixQuery& q, std::vector<uint64_t>& out) {
        out.clear();
        if (!host_resident(q.value_col) || null_masks_.count(q.value_col)) return ReadOutcome::NEEDS_EXCLUSIVE;
        const MatrixType vty = column_type(q.value_col);
        const void* vp = catalog_.at(q.value_col)->host_ptr();
        const size_t n = column_rows(q.value_col);

        // scalar cross-column WHERE: filter on a different u32 column
        if (!q.grouped && q.has_filter && q.filter_col != 0 && q.filter_col != q.value_col) {
            if (!host_resident(q.filter_col) || column_type(q.filter_col) != MatrixType::U32) return ReadOutcome::NEEDS_EXCLUSIVE;
            if (column_rows(q.filter_col) != n) return ReadOutcome::NEEDS_EXCLUSIVE;
            const uint32_t* f = reinterpret_cast<const uint32_t*>(catalog_.at(q.filter_col)->host_ptr());
            const MatrixPredicate p{q.cmp, q.threshold, q.upper};
            if (vty == MatrixType::I64)      out.assign(1, static_cast<uint64_t>(matrix_cpu_reduce_filtered_by_i64(f, p, reinterpret_cast<const int64_t*>(vp), n, q.agg)));
            else if (vty == MatrixType::F64) out.assign(1, matrix_bit_cast<uint64_t>(matrix_cpu_reduce_filtered_by_f64(f, p, reinterpret_cast<const double*>(vp), n, q.agg)));
            else                             out.assign(1, matrix_cpu_reduce_filtered_by(f, p, reinterpret_cast<const uint32_t*>(vp), n, q.agg));
            return ReadOutcome::SERVED;
        }
        if (q.grouped && q.has_filter && q.filter_col != 0 && q.filter_col != q.value_col)
            return ReadOutcome::NEEDS_EXCLUSIVE;   // grouped cross-column: defer to the exclusive path

        // grouped (key must be a registered, HOST, u32 column distinct from value, valid group count, aligned)
        if (q.grouped) {
            if (!host_resident(q.key_col) || q.key_col == q.value_col || q.num_groups == 0
                || q.num_groups > max_query_groups_ || column_type(q.key_col) != MatrixType::U32
                || column_rows(q.key_col) != n) return ReadOutcome::NEEDS_EXCLUSIVE;
            const uint32_t* keys = reinterpret_cast<const uint32_t*>(catalog_.at(q.key_col)->host_ptr());
            const uint32_t G = q.num_groups;
            out.assign(G, 0);
            if (vty == MatrixType::I64) {
                std::vector<int64_t> tmp(G);
                if (q.has_filter) matrix_cpu_group_reduce_i64_pred(keys, reinterpret_cast<const int64_t*>(vp), n, G, q.agg, MatrixPredicateI64{q.cmp, q.lo_i64, q.hi_i64}, tmp.data());
                else              matrix_cpu_group_reduce_i64(keys, reinterpret_cast<const int64_t*>(vp), n, G, q.agg, tmp.data());
                for (uint32_t g = 0; g < G; ++g) out[g] = static_cast<uint64_t>(tmp[g]);
            } else if (vty == MatrixType::F64) {
                std::vector<double> tmp(G);
                if (q.has_filter) matrix_cpu_group_reduce_f64_pred(keys, reinterpret_cast<const double*>(vp), n, G, q.agg, MatrixPredicateF64{q.cmp, q.lo_f64, q.hi_f64}, tmp.data());
                else              matrix_cpu_group_reduce_f64(keys, reinterpret_cast<const double*>(vp), n, G, q.agg, tmp.data());
                for (uint32_t g = 0; g < G; ++g) out[g] = matrix_bit_cast<uint64_t>(tmp[g]);
            } else {
                if (q.has_filter) matrix_cpu_group_reduce_pred(keys, reinterpret_cast<const uint32_t*>(vp), n, G, q.agg, MatrixPredicate{q.cmp, q.threshold, q.upper}, out.data());
                else              matrix_cpu_group_reduce(keys, reinterpret_cast<const uint32_t*>(vp), n, G, q.agg, out.data());
            }
            return ReadOutcome::SERVED;
        }

        // scalar (same-column or unfiltered)
        if (vty == MatrixType::I64) {
            const MatrixPredicateI64 p{q.cmp, q.lo_i64, q.hi_i64}; const int64_t* v = reinterpret_cast<const int64_t*>(vp);
            out.assign(1, static_cast<uint64_t>(q.has_filter ? matrix_cpu_reduce_pred_i64(v, n, p, q.agg) : matrix_cpu_reduce_all_i64(v, n, q.agg)));
        } else if (vty == MatrixType::F64) {
            const MatrixPredicateF64 p{q.cmp, q.lo_f64, q.hi_f64}; const double* v = reinterpret_cast<const double*>(vp);
            out.assign(1, matrix_bit_cast<uint64_t>(q.has_filter ? matrix_cpu_reduce_pred_f64(v, n, p, q.agg) : matrix_cpu_reduce_all_f64(v, n, q.agg)));
        } else {
            const MatrixPredicate p{q.cmp, q.threshold, q.upper}; const uint32_t* v = reinterpret_cast<const uint32_t*>(vp);
            out.assign(1, q.has_filter ? matrix_cpu_reduce_pred(v, n, p, q.agg) : matrix_cpu_reduce_all(v, n, q.agg));
        }
        return ReadOutcome::SERVED;
    }

    // Top-N groups by aggregate value (ORDER BY agg DESC LIMIT k): run a grouped query, then return the k
    // (group_id, value) pairs with the largest aggregate. The staple "top 10 X by Y" analytical query.
    // ponytail: sorts by the RAW uint64 aggregate — exact for U32-valued groups and COUNT (non-negative);
    // for int64/double SUM/MIN/MAX the result bits aren't value-order, so use this for U32/COUNT grouping.
    std::vector<std::pair<uint64_t, uint64_t>> top_groups(const MatrixQuery& q, size_t k) {
        std::vector<uint64_t> out;
        if (!q.grouped || execute_query(q, out) != MatrixQueryStatus::OK) return {};
        std::vector<std::pair<uint64_t, uint64_t>> pairs;
        pairs.reserve(out.size());
        for (uint64_t g = 0; g < out.size(); ++g) pairs.emplace_back(g, out[g]);
        const size_t kk = std::min(k, pairs.size());
        std::partial_sort(pairs.begin(), pairs.begin() + static_cast<std::ptrdiff_t>(kk), pairs.end(),
                          [](const auto& a, const auto& b) { return a.second > b.second; });   // value desc
        pairs.resize(kk);
        return pairs;
    }

    // HAVING: the groups whose aggregate satisfies `cmp` against threshold (BETWEEN uses [threshold, upper]) —
    // e.g. "regions where SUM(amount) > 1000". Runs the grouped query, then filters the (group, value) pairs.
    // ponytail: compares the RAW uint64 aggregate (a uint64 comparator, since a grouped SUM can exceed u32),
    // so it's exact for U32-valued groups and COUNT; I64/F64 bit-patterns aren't value-order (use for U32/COUNT),
    // matching top_groups.
    std::vector<std::pair<uint64_t, uint64_t>> having(const MatrixQuery& q, MatrixCmp cmp,
                                                      uint64_t threshold, uint64_t upper = 0) {
        std::vector<uint64_t> out;
        if (!q.grouped || execute_query(q, out) != MatrixQueryStatus::OK) return {};
        std::vector<std::pair<uint64_t, uint64_t>> pairs;
        for (uint64_t g = 0; g < out.size(); ++g)
            if (cmp_u64(cmp, out[g], threshold, upper)) pairs.emplace_back(g, out[g]);
        return pairs;
    }

    // AVG(value_col) = SUM/COUNT as double(s), derived from the two existing aggregates — so it inherits
    // their per-type handling (U32/I64/F64) and NULL-skipping for free, with no new reducer op threaded
    // through every typed/grouped/filtered/nullable path. A scalar query yields one element; a grouped
    // query yields one per group. A zero-count set/group yields NaN (the average of nothing). NULL-aware on
    // both paths: SUM and COUNT both skip NULLs (scalar always; grouped since the grouped reducers take the mask).
    std::vector<double> average(const MatrixQuery& q) {
        MatrixQuery sq = q; sq.agg = AGG_SUM;   std::vector<uint64_t> sv;
        MatrixQuery cq = q; cq.agg = AGG_COUNT; std::vector<uint64_t> cv;
        if (execute_query(sq, sv) != MatrixQueryStatus::OK || execute_query(cq, cv) != MatrixQueryStatus::OK
            || sv.size() != cv.size())
            return {};
        const MatrixType ty = column_type(q.value_col);
        std::vector<double> out(sv.size());
        for (size_t i = 0; i < sv.size(); ++i) {
            const double sum = (ty == MatrixType::F64) ? matrix_bit_cast<double>(sv[i])
                             : (ty == MatrixType::I64) ? static_cast<double>(static_cast<int64_t>(sv[i]))
                                                       : static_cast<double>(sv[i]);            // U32
            // COUNT is encoded like the column: F64 columns return it as double-bits (execute_query bit_casts
            // the whole F64 result), U32/I64 return a plain integer count.
            const double count = (ty == MatrixType::F64) ? matrix_bit_cast<double>(cv[i])
                                                         : static_cast<double>(cv[i]);
            out[i] = (count != 0.0) ? sum / count : std::numeric_limits<double>::quiet_NaN();
        }
        return out;
    }

    // COUNT(DISTINCT col): number of distinct non-NULL values in a column. Borrow-and-return like the
    // scalar aggregates; null-aware (skips masked rows, via value_nulls). Typed over U32/I64/F64.
    // ponytail: an exact hash set over every value — O(distinct) memory. A HyperLogLog sketch is the
    // upgrade path when the column is huge and an estimate is acceptable. F64 NaN edge: each NaN counts as
    // distinct (NaN != NaN), since the set is over double values — documented, not special-cased.
    uint64_t count_distinct(uint64_t col_id) {
        TieredColumn& col = *catalog_.at(col_id);
        tier_mgr_.record_access(col_id, col.size_bytes());
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const MatrixType ty = column_type(col_id);
        const size_t elem = (ty == MatrixType::U32) ? sizeof(uint32_t) : sizeof(uint64_t);
        const size_t n = col.size_bytes() / elem;
        const uint8_t* nulls = value_nulls(col_id, n);
        uint64_t result;
        if (ty == MatrixType::I64)
            result = distinct_count(reinterpret_cast<const int64_t*>(col.host_ptr()), n, nulls);
        else if (ty == MatrixType::F64)
            result = distinct_count(reinterpret_cast<const double*>(col.host_ptr()), n, nulls);
        else
            result = distinct_count(reinterpret_cast<const uint32_t*>(col.host_ptr()), n, nulls);
        if (home != MemorySpace::HOST) col.migrate_to(home);
        maybe_rebalance();
        return result;
    }

    // Parse + run an AVG query string ("SELECT AVG(col) [WHERE ...] [GROUP BY k]") -> the average(s) as
    // double(s). AVG isn't a reducer op (see average()), so rather than teach the parser a phantom agg we
    // rewrite the AVG token to SUM, reuse the full parser (WHERE/GROUP BY/etc.), then derive SUM/COUNT.
    // The tokenizer round-trips (space-join re-tokenizes identically), so this needs no parser change.
    // Empty on parse error or a non-AVG query.
    std::vector<double> avg_query(const std::string& sql) {
        std::vector<std::string> tk = matrixparse_tokenize(sql);
        if (tk.size() < 2) return {};
        std::string a = tk[1];
        for (char& c : a) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c)));
        if (a != "AVG") return {};                                   // only the AVG string form lives here
        tk[1] = "SUM";                                               // rewrite the aggregate keyword
        std::string rewritten;
        for (size_t i = 0; i < tk.size(); ++i) { if (i) rewritten += ' '; rewritten += tk[i]; }
        MatrixQuery q;
        if (parse_query(rewritten, q) != MatrixQueryStatus::OK) return {};
        return average(q);
    }

    // Parse + run a MULTI-aggregate query ("SELECT COUNT(a), SUM(b), MIN(c) [WHERE ...] [GROUP BY k]") ->
    // one result column per aggregate (scalar: size 1; grouped: size num_groups), each encoded like
    // execute_query for that aggregate's value type. Splits the comma-separated SELECT list and delegates
    // each "SELECT agg(col) <shared tail>" to the full parser+executor, so it inherits WHERE (incl.
    // cross-column / string filters), GROUP BY, and per-type handling for free. Empty {} on any parse/exec
    // error. ponytail: COUNT/SUM/MIN/MAX only (AVG via avg_query); the SELECT list is literal-free, so the
    // first WHERE/GROUP/ORDER keyword is the tail boundary and the list splits cleanly on commas.
    std::vector<std::vector<uint64_t>> query_multi(const std::string& sql) {
        size_t b = 0; while (b < sql.size() && std::isspace(static_cast<unsigned char>(sql[b]))) ++b;
        std::string up = sql.substr(b);
        for (char& c : up) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c)));
        if (up.rfind("SELECT", 0) != 0) return {};
        size_t tail = std::string::npos;                            // first of WHERE/GROUP/ORDER (the tail)
        for (const char* kw : {" WHERE", " GROUP", " ORDER"}) {
            const size_t p = up.find(kw, 6);
            if (p != std::string::npos) tail = std::min(tail, b + p);
        }
        const size_t listEnd = (tail == std::string::npos) ? sql.size() : tail;
        const std::string list = sql.substr(b + 6, listEnd - (b + 6));   // the comma-separated agg list
        const std::string rest = (tail == std::string::npos) ? std::string{} : sql.substr(tail); // " WHERE ..." etc.
        std::vector<std::vector<uint64_t>> results;
        for (size_t i = 0; i <= list.size(); ) {
            const size_t j = list.find(',', i);
            std::string term = list.substr(i, (j == std::string::npos ? list.size() : j) - i);
            size_t t0 = 0, t1 = term.size();                        // trim
            while (t0 < t1 && std::isspace(static_cast<unsigned char>(term[t0]))) ++t0;
            while (t1 > t0 && std::isspace(static_cast<unsigned char>(term[t1 - 1]))) --t1;
            term = term.substr(t0, t1 - t0);
            if (!term.empty()) {
                MatrixQuery q; std::vector<uint64_t> r;
                if (parse_query("SELECT " + term + rest, q) != MatrixQueryStatus::OK) return {};
                if (execute_query(q, r) != MatrixQueryStatus::OK) return {};
                results.push_back(std::move(r));
            }
            if (j == std::string::npos) break;
            i = j + 1;
        }
        return results;
    }

    // Parse + run a PROJECTION query ("SELECT col [WHERE fcol op val [AND val]] [LIMIT n]") -> the values of
    // `col` for the matching rows (encoded like gather). Distinct from the aggregate forms: a bare column,
    // no AGG(...). The WHERE predicate reuses parse_query on a synthetic "SELECT COUNT(fcol) WHERE ..." so it
    // inherits numeric + string-dict / ordered / BETWEEN filters; v1 filters on a u32 column. Empty {} if it
    // isn't a well-formed projection. ponytail: O(n) row scan behind the filter; a sorted index is the upgrade.
    std::vector<uint64_t> project_query(const std::string& sql) {
        const std::vector<std::string> tk = matrixparse_tokenize(sql);
        auto up = [](std::string s){ for (char& c : s) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c))); return s; };
        if (tk.size() < 2 || up(tk[0]) != "SELECT") return {};
        if (tk.size() >= 3 && tk[2] == "(") return {};               // an aggregate, not a projection
        const uint64_t col_id = column_id(tk[1]);
        if (col_id == 0) return {};
        uint64_t limit = 0; size_t whereEnd = tk.size();             // [2, whereEnd) is the optional WHERE region
        for (size_t i = 2; i < tk.size(); ++i) {
            if (up(tk[i]) == "LIMIT") {
                if (i + 2 != tk.size()) return {};                   // LIMIT must be the tail: LIMIT n
                uint64_t nlim = 0; auto [p, ec] = std::from_chars(tk[i + 1].data(), tk[i + 1].data() + tk[i + 1].size(), nlim);
                if (ec != std::errc{} || p != tk[i + 1].data() + tk[i + 1].size()) return {};
                limit = nlim; whereEnd = i; break;
            }
        }
        bool has_filter = false; uint64_t fcol_id = 0; MatrixPredicate pred{};
        if (whereEnd > 2) {
            if (up(tk[2]) != "WHERE" || tk.size() < 4) return {};
            fcol_id = column_id(tk[3]);
            if (fcol_id == 0 || column_type(fcol_id) != MatrixType::U32) return {};   // v1: u32 filter (incl. dict strings)
            std::string agg = "SELECT COUNT ( " + tk[3] + " )";       // reuse parse_query's WHERE parsing
            for (size_t i = 2; i < whereEnd; ++i) { agg += ' '; agg += tk[i]; }
            MatrixQuery q;
            if (parse_query(agg, q) != MatrixQueryStatus::OK || !q.has_filter) return {};
            if (column_rows(fcol_id) != column_rows(col_id)) return {};   // misaligned filter / projection columns
            has_filter = true; pred = MatrixPredicate{q.cmp, q.threshold, q.upper};
        }
        return project(col_id, has_filter, fcol_id, pred, limit);
    }

    // Parse + run a top-N grouped query string ("SELECT SUM(x) GROUP BY k ORDER BY SUM DESC LIMIT n").
    // Returns the (group, value) pairs by value desc; empty on parse error or a query without GROUP BY + LIMIT.
    std::vector<std::pair<uint64_t, uint64_t>> top_query(const std::string& sql) {
        MatrixQuery q;
        if (parse_query(sql, q) != MatrixQueryStatus::OK || !q.grouped || q.limit == 0) return {};
        return top_groups(q, q.limit);
    }

    // Parse + run a HAVING query string ("SELECT SUM(x) GROUP BY k HAVING SUM <cmp> v") -> the (group,value)
    // pairs whose aggregate satisfies the comparison. Splits the HAVING tail off, parses the head (the
    // grouped query) via the full parser — the tokenizer round-trips, so the space-joined head re-parses
    // identically — then runs having(). Empty on parse error or a query without GROUP BY + HAVING.
    // ponytail: the string form takes a single comparison (GT/GE/LT/LE/EQ/NE); BETWEEN-in-HAVING is
    // reachable only via the having() method (rare in practice).
    std::vector<std::pair<uint64_t, uint64_t>> having_query(const std::string& sql) {
        std::vector<std::string> tk = matrixparse_tokenize(sql);
        auto up = [](std::string s){ for (char& c : s) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c))); return s; };
        size_t hi = tk.size();
        for (size_t i = 0; i < tk.size(); ++i) if (up(tk[i]) == "HAVING") { hi = i; break; }
        if (hi == tk.size() || hi + 4 != tk.size()) return {};            // need exactly: HAVING <key> <cmp> <value>
        std::string head;
        for (size_t i = 0; i < hi; ++i) { if (i) head += ' '; head += tk[i]; }
        MatrixQuery q;
        if (parse_query(head, q) != MatrixQueryStatus::OK || !q.grouped) return {};
        // sort key must name the SELECT aggregate or the value column (same rule as ORDER BY)
        static const char* AGGN[] = { "COUNT", "SUM", "MIN", "MAX" };
        if (up(tk[hi + 1]) != AGGN[q.agg] && column_id(tk[hi + 1]) != q.value_col) return {};
        MatrixCmp cmp;
        const std::string op = tk[hi + 2];
        if      (op == ">")  cmp = MatrixCmp::GT;  else if (op == ">=") cmp = MatrixCmp::GE;
        else if (op == "<")  cmp = MatrixCmp::LT;  else if (op == "<=") cmp = MatrixCmp::LE;
        else if (op == "=")  cmp = MatrixCmp::EQ;  else if (op == "!=") cmp = MatrixCmp::NE;
        else return {};
        const std::string& v = tk[hi + 3];
        uint64_t threshold = 0; auto [p, ec] = std::from_chars(v.data(), v.data() + v.size(), threshold);
        if (ec != std::errc{} || p != v.data() + v.size()) return {};
        return having(q, cmp, threshold);
    }

    // Parse + run a COUNT(DISTINCT col) query string -> the distinct non-NULL value count. Completes the
    // aggregate-string surface (every supported aggregate is now expressible as SQL). Returns false on a
    // malformed / non-distinct query or an unknown column (out untouched), true with the count otherwise.
    bool distinct_query(const std::string& sql, uint64_t& out) {
        std::vector<std::string> tk = matrixparse_tokenize(sql);
        auto up = [](std::string s){ for (char& c : s) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c))); return s; };
        if (tk.size() != 6) return false;                                 // SELECT COUNT ( DISTINCT col )
        if (up(tk[0]) != "SELECT" || up(tk[1]) != "COUNT" || tk[2] != "(" || up(tk[3]) != "DISTINCT" || tk[5] != ")")
            return false;
        const uint64_t cid = column_id(tk[4]);
        if (cid == 0) return false;                                       // unknown column
        out = count_distinct(cid);
        return true;
    }

    // Parse a scalar query string into `out` (see DM-4 grammar). Returns OK, ERR_UNKNOWN_COLUMN (bad name),
    // or ERR_PARSE (any malformed form). Untrusted input — never crashes; on ANY non-OK status `out` is
    // reset to a default MatrixQuery (so a caller that ignores the status can't execute partial garbage).
    MatrixQueryStatus parse_query(const std::string& sql, MatrixQuery& out) {
        const MatrixQueryStatus st = parse_query_impl(sql, out);
        if (st != MatrixQueryStatus::OK) out = MatrixQuery{};   // no partial state survives a parse error
        return st;
    }

    // The parse pipeline (see grammar). `out` is reset at entry; the public parse_query wrapper above also
    // clears it on any error exit, so partial fields set before a mid-parse failure never leak to a caller.
    MatrixQueryStatus parse_query_impl(const std::string& sql, MatrixQuery& out) {
        out = MatrixQuery{};
        const std::vector<std::string> tk = matrixparse_tokenize(sql);
        size_t k = 0;
        auto up   = [](std::string s){ for (char& c : s) c = static_cast<char>(std::toupper(static_cast<unsigned char>(c))); return s; };
        auto next = [&]{ return k < tk.size() ? tk[k++] : std::string{}; };
        if (up(next()) != "SELECT") return MatrixQueryStatus::ERR_PARSE;
        const std::string aggs = up(next());
        MatrixAggOp agg;
        if (aggs == "COUNT") agg = AGG_COUNT; else if (aggs == "SUM") agg = AGG_SUM;
        else if (aggs == "MIN") agg = AGG_MIN; else if (aggs == "MAX") agg = AGG_MAX;
        else return MatrixQueryStatus::ERR_PARSE;
        if (next() != "(") return MatrixQueryStatus::ERR_PARSE;
        const std::string col = next();
        if (next() != ")") return MatrixQueryStatus::ERR_PARSE;
        const uint64_t vid = column_id(col);
        if (vid == 0) return MatrixQueryStatus::ERR_UNKNOWN_COLUMN;
        out.value_col = vid; out.agg = agg;
        auto peek = [&]{ return k < tk.size() ? up(tk[k]) : std::string{}; };   // uppercased lookahead, no consume
        // optional WHERE <valuecol> <op> <val> [AND <val>]
        if (peek() == "WHERE") {
            next();                                                            // consume WHERE
            // The filter column may differ from the SELECT column (cross-column WHERE) — but only a u32
            // column can be the filter (v1: dict-encoded strings + u32 dims). Same column keeps any type.
            const uint64_t fcol = column_id(next());
            if (fcol == 0) return MatrixQueryStatus::ERR_UNKNOWN_COLUMN;
            if (fcol != vid && column_type(fcol) != MatrixType::U32) return MatrixQueryStatus::ERR_PARSE;
            const uint64_t pcol = fcol;                       // the column the predicate reads
            out.filter_col = (fcol == vid) ? 0 : fcol;        // 0 = filter the value column (existing behavior)
            const std::string ops = up(next());
            const MatrixType ty = column_type(pcol);
            out.has_filter = true;
            if (string_dicts_.find(pcol) != string_dicts_.end()) {
                // Dict-encoded string column: the dict is sorted, so a code's order == its string's order.
                // EQ/NE use the exact code (string_encode -> a no-row code if absent: '=' matches nothing,
                // '!=' all). Ordered ops + BETWEEN translate the (optionally quoted) literal to a code rank via
                // binary search on the sorted dict, so they mean lexicographic string comparison.
                auto unquote = [](std::string s){ return (s.size() >= 2 && (s.front()=='\'' || s.front()=='"') && s.back()==s.front()) ? s.substr(1, s.size()-2) : s; };
                const std::vector<std::string>& dict = string_dicts_.at(pcol);   // sorted
                auto lb = [&](const std::string& x){ return static_cast<uint32_t>(std::lower_bound(dict.begin(), dict.end(), x) - dict.begin()); }; // first code with dict[c] >= x
                auto ub = [&](const std::string& x){ return static_cast<uint32_t>(std::upper_bound(dict.begin(), dict.end(), x) - dict.begin()); }; // first code with dict[c] >  x
                if (ops == "BETWEEN") {
                    const std::string lo = unquote(next());
                    if (up(next()) != "AND") return MatrixQueryStatus::ERR_PARSE;
                    const std::string hi = unquote(next());
                    const uint32_t u = ub(hi);
                    out.cmp = MatrixCmp::BETWEEN; out.threshold = lb(lo); out.upper = (u == 0 ? 0u : u - 1); // codes [lb(lo), ub(hi)-1] == strings [lo,hi]
                } else if (ops == "=" || ops == "!=") {
                    out.cmp = (ops == "=") ? MatrixCmp::EQ : MatrixCmp::NE;
                    const std::string lit = next(); if (lit.empty()) return MatrixQueryStatus::ERR_PARSE;
                    out.threshold = string_encode(pcol, unquote(lit));
                } else {
                    const std::string x = unquote(next()); if (x.empty()) return MatrixQueryStatus::ERR_PARSE;
                    if      (ops == ">")  { out.cmp = MatrixCmp::GE; out.threshold = ub(x); }   // s >  x: codes >= upper_bound(x)
                    else if (ops == ">=") { out.cmp = MatrixCmp::GE; out.threshold = lb(x); }   // s >= x: codes >= lower_bound(x)
                    else if (ops == "<")  { out.cmp = MatrixCmp::LT; out.threshold = lb(x); }   // s <  x: codes <  lower_bound(x)
                    else if (ops == "<=") { out.cmp = MatrixCmp::LT; out.threshold = ub(x); }   // s <= x: codes <  upper_bound(x)
                    else return MatrixQueryStatus::ERR_PARSE;
                }
            } else if (ops == "BETWEEN") {
                out.cmp = MatrixCmp::BETWEEN;
                if (!set_bound(ty, out, true, next())) return MatrixQueryStatus::ERR_PARSE;
                if (up(next()) != "AND")               return MatrixQueryStatus::ERR_PARSE;
                if (!set_bound(ty, out, false, next())) return MatrixQueryStatus::ERR_PARSE;
            } else {
                if      (ops == ">")  out.cmp = MatrixCmp::GT;  else if (ops == ">=") out.cmp = MatrixCmp::GE;
                else if (ops == "<")  out.cmp = MatrixCmp::LT;  else if (ops == "<=") out.cmp = MatrixCmp::LE;
                else if (ops == "=")  out.cmp = MatrixCmp::EQ;  else if (ops == "!=") out.cmp = MatrixCmp::NE;
                else return MatrixQueryStatus::ERR_PARSE;
                if (!set_bound(ty, out, true, next())) return MatrixQueryStatus::ERR_PARSE;
            }
        }
        // optional GROUP BY <keycol> (uint32 key). num_groups is derived from the key column (max+1).
        if (peek() == "GROUP") {
            next();                                                            // consume GROUP
            if (up(next()) != "BY") return MatrixQueryStatus::ERR_PARSE;
            const uint64_t kid = column_id(next());
            if (kid == 0) return MatrixQueryStatus::ERR_UNKNOWN_COLUMN;
            if (column_type(kid) != MatrixType::U32) return MatrixQueryStatus::ERR_PARSE;  // grouped key must be u32
            out.grouped = true; out.key_col = kid; out.num_groups = derive_num_groups_u32(kid);
        }
        // optional ORDER BY <agg|valuecol> DESC LIMIT <n> -> top-N groups. DESC only (top_groups is descending)
        // and requires GROUP BY (top-N is over groups). Sets out.limit; top_query applies it.
        if (peek() == "ORDER") {
            next();                                                            // consume ORDER
            if (up(next()) != "BY") return MatrixQueryStatus::ERR_PARSE;
            if (!out.grouped)       return MatrixQueryStatus::ERR_PARSE;       // top-N is meaningless without groups
            const std::string sortraw = next();                               // sort key: the agg name or the value col
            if (up(sortraw) != aggs && column_id(sortraw) != vid) return MatrixQueryStatus::ERR_PARSE;
            if (up(next()) != "DESC")  return MatrixQueryStatus::ERR_PARSE;    // only DESC (top_groups sorts desc)
            if (up(next()) != "LIMIT") return MatrixQueryStatus::ERR_PARSE;
            const std::string lim = next();
            uint64_t n = 0; auto [p, ec] = std::from_chars(lim.data(), lim.data() + lim.size(), n);
            if (ec != std::errc{} || p != lim.data() + lim.size() || n == 0) return MatrixQueryStatus::ERR_PARSE;
            out.limit = n;
        }
        if (k != tk.size()) return MatrixQueryStatus::ERR_PARSE;     // trailing junk
        return MatrixQueryStatus::OK;
    }

private:
    // Parse the numeric literal `v` into the bound field matching the column type (lo=true -> primary/lower).
    // Returns false on junk / overflow / empty. int64 via from_chars; double via strtod; u32 via from_chars.
    bool set_bound(MatrixType ty, MatrixQuery& q, bool lo, const std::string& v) {
        if (v.empty()) return false;
        if (ty == MatrixType::F64) {
            errno = 0; char* e = nullptr; const double d = std::strtod(v.c_str(), &e);
            if (e != v.c_str() + v.size() || errno == ERANGE) return false;
            (lo ? q.lo_f64 : q.hi_f64) = d; return true;
        }
        if (ty == MatrixType::I64) {
            int64_t x = 0; auto [p, ec] = std::from_chars(v.data(), v.data() + v.size(), x);
            if (ec != std::errc{} || p != v.data() + v.size()) return false;
            (lo ? q.lo_i64 : q.hi_i64) = x; return true;
        }
        uint32_t x = 0; auto [p, ec] = std::from_chars(v.data(), v.data() + v.size(), x);
        if (ec != std::errc{} || p != v.data() + v.size()) return false;
        (lo ? q.threshold : q.upper) = x; return true;
    }

    // num_groups for a parsed GROUP BY = (max key in the u32 key column) + 1 — the dense-group count.
    // ponytail: this reads/borrows the key column's DATA at PARSE time (a max-scan), unlike pure-metadata
    // parsing — an explicit `INTO n` or a planner stats-lookup would avoid it. A sparse/huge key makes
    // num_groups large and execute_query then rejects it (ERR_TOO_MANY_GROUPS); empty -> 0 (ERR_INVALID_GROUP).
    uint32_t derive_num_groups_u32(uint64_t kid) {
        TieredColumn& col = *catalog_.at(kid);
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(col.host_ptr());
        const size_t n = col.size_bytes() / sizeof(uint32_t);
        const uint64_t mx = matrix_cpu_reduce_all(keys, n, AGG_MAX);   // max key (0 if n==0)
        if (home != MemorySpace::HOST) col.migrate_to(home);
        if (n == 0) return 0u;
        // mx+1 must stay in range: a key of UINT32_MAX would need 2^32 groups, which no query
        // could ever pass max_query_groups_ for anyway — reject explicitly instead of silently
        // wrapping to 0 (0 also means ERR_INVALID_GROUP downstream, so a wrap used to "work" by
        // accident; this makes the rejection reason correct instead of coincidental).
        if (mx >= std::numeric_limits<uint32_t>::max()) return std::numeric_limits<uint32_t>::max();
        return static_cast<uint32_t>(mx + 1);
    }

    // Cross-column scalar WHERE: aggregate value_col over the rows where the u32 filter_col satisfies the
    // predicate. Borrows both columns to HOST, dispatches by the value type, returns the encoded result
    // (same codec as the per-type scalar scans). ponytail: two-pass borrow + one fused reduce; a GPU
    // cross-column kernel + a grouped variant are the documented follow-ups.
    uint64_t scalar_cross_filter(const MatrixQuery& q) {
        TieredColumn& fc = *catalog_.at(q.filter_col);
        TieredColumn& vc = *catalog_.at(q.value_col);
        tier_mgr_.record_access(q.filter_col, fc.size_bytes());
        tier_mgr_.record_access(q.value_col, vc.size_bytes());
        const MemorySpace fh = fc.tier(); if (fh != MemorySpace::HOST) { ++cold_borrows_; fc.migrate_to(MemorySpace::HOST); }
        const MemorySpace vh = vc.tier(); if (vh != MemorySpace::HOST) { ++cold_borrows_; vc.migrate_to(MemorySpace::HOST); }
        const uint32_t* f = reinterpret_cast<const uint32_t*>(fc.host_ptr());
        const size_t n = fc.size_bytes() / sizeof(uint32_t);
        const MatrixPredicate p{q.cmp, q.threshold, q.upper};
        const MatrixType vty = column_type(q.value_col);
        uint64_t enc;
        if (vty == MatrixType::I64)
            enc = static_cast<uint64_t>(matrix_cpu_reduce_filtered_by_i64(f, p, reinterpret_cast<const int64_t*>(vc.host_ptr()), n, q.agg));
        else if (vty == MatrixType::F64)
            enc = matrix_bit_cast<uint64_t>(matrix_cpu_reduce_filtered_by_f64(f, p, reinterpret_cast<const double*>(vc.host_ptr()), n, q.agg));
        else
            enc = matrix_cpu_reduce_filtered_by(f, p, reinterpret_cast<const uint32_t*>(vc.host_ptr()), n, q.agg);
        if (vh != MemorySpace::HOST) vc.migrate_to(vh);
        if (fh != MemorySpace::HOST) fc.migrate_to(fh);
        maybe_rebalance();
        return enc;
    }

    // Grouped cross-column WHERE: GROUP BY key_col, aggregate value_col, over rows where the u32 filter_col
    // satisfies the predicate. Borrows the distinct columns (key may == filter) to HOST, dispatches by the
    // value type, writes num_groups encoded results. ponytail: CPU path; a GPU grouped cross-column kernel
    // is the follow-up (the same-column grouped path already runs on the GPU).
    void grouped_cross_filter(const MatrixQuery& q, std::vector<uint64_t>& out) {
        std::vector<std::pair<uint64_t, MemorySpace>> borrowed;
        auto borrow = [&](uint64_t id) {
            for (auto& b : borrowed) if (b.first == id) return;   // distinct columns only (key may == filter)
            TieredColumn& c = *catalog_.at(id);
            const MemorySpace home = c.tier();
            borrowed.push_back({id, home});
            tier_mgr_.record_access(id, c.size_bytes());
            if (home != MemorySpace::HOST) { ++cold_borrows_; c.migrate_to(MemorySpace::HOST); }
        };
        borrow(q.value_col); borrow(q.filter_col); borrow(q.key_col);   // all HOST-resident after this
        const uint32_t* keys = reinterpret_cast<const uint32_t*>(catalog_.at(q.key_col)->host_ptr());
        const uint32_t* f    = reinterpret_cast<const uint32_t*>(catalog_.at(q.filter_col)->host_ptr());
        TieredColumn& vc = *catalog_.at(q.value_col);
        const size_t n = catalog_.at(q.key_col)->size_bytes() / sizeof(uint32_t);
        const MatrixPredicate p{q.cmp, q.threshold, q.upper};
        const uint32_t G = q.num_groups;
        out.assign(G, 0);
        const MatrixType vty = column_type(q.value_col);
        if (vty == MatrixType::I64) {
            std::vector<int64_t> tmp(G);
            matrix_cpu_group_reduce_filtered_by_i64(keys, f, p, reinterpret_cast<const int64_t*>(vc.host_ptr()), n, G, q.agg, tmp.data());
            for (uint32_t g = 0; g < G; ++g) out[g] = static_cast<uint64_t>(tmp[g]);
        } else if (vty == MatrixType::F64) {
            std::vector<double> tmp(G);
            matrix_cpu_group_reduce_filtered_by_f64(keys, f, p, reinterpret_cast<const double*>(vc.host_ptr()), n, G, q.agg, tmp.data());
            for (uint32_t g = 0; g < G; ++g) out[g] = matrix_bit_cast<uint64_t>(tmp[g]);
        } else {
            matrix_cpu_group_reduce_filtered_by(keys, f, p, reinterpret_cast<const uint32_t*>(vc.host_ptr()), n, G, q.agg, out.data());
        }
        for (auto it = borrowed.rbegin(); it != borrowed.rend(); ++it)   // return each distinct borrow to its home
            if (it->second != MemorySpace::HOST) catalog_.at(it->first)->migrate_to(it->second);
        maybe_rebalance();
    }

    MatrixQueryStatus execute_query_impl(const MatrixQuery& q, std::vector<uint64_t>& out) {
        out.clear();
        if (!catalog_has(q.value_col)) return MatrixQueryStatus::ERR_UNKNOWN_COLUMN;
        // Cross-column scalar WHERE (v1): filter on a different u32 column than the aggregate. Grouped
        // cross-column + value NULL-awareness on this path are the documented next increments.
        if (!q.grouped && q.has_filter && q.filter_col != 0 && q.filter_col != q.value_col) {
            if (!catalog_has(q.filter_col) || column_type(q.filter_col) != MatrixType::U32) return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE;
            if (column_rows(q.filter_col) != column_rows(q.value_col)) return MatrixQueryStatus::ERR_INVALID_GROUP;   // misaligned columns
            out.assign(1, scalar_cross_filter(q));
            return MatrixQueryStatus::OK;
        }
        // Grouped cross-column WHERE (v1): GROUP BY a u32 key, aggregate value_col, filter on a different
        // u32 column. Same validation as the typed grouped paths (key u32, distinct, bounded, aligned).
        if (q.grouped && q.has_filter && q.filter_col != 0 && q.filter_col != q.value_col) {
            if (!catalog_has(q.key_col) || q.key_col == q.value_col || q.num_groups == 0) return MatrixQueryStatus::ERR_INVALID_GROUP;
            if (column_type(q.key_col) != MatrixType::U32) return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE;
            if (!catalog_has(q.filter_col) || column_type(q.filter_col) != MatrixType::U32) return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE;
            if (q.num_groups > max_query_groups_) return MatrixQueryStatus::ERR_TOO_MANY_GROUPS;
            if (column_rows(q.key_col) != column_rows(q.value_col) || column_rows(q.filter_col) != column_rows(q.value_col))
                return MatrixQueryStatus::ERR_INVALID_GROUP;   // misaligned columns
            grouped_cross_filter(q, out);
            return MatrixQueryStatus::OK;
        }
        if (column_type(q.value_col) == MatrixType::I64) {
            if (q.grouped) {
                // Key-type check FIRST: an int64 GROUP BY key is unsupported (DM-3d) regardless of whether
                // it aliases the value column — i64val+i64key must report ERR_UNSUPPORTED_TYPE (spec §1),
                // not the ERR_INVALID_GROUP that the key==value/row-count guard below would otherwise give.
                if (catalog_has(q.key_col) && column_type(q.key_col) != MatrixType::U32)
                    return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE; // int64 key = DM-3d
                if (!catalog_has(q.key_col) || q.key_col == q.value_col || q.num_groups == 0
                    || column_rows(q.key_col) != column_rows(q.value_col))
                    return MatrixQueryStatus::ERR_INVALID_GROUP;
                if (q.num_groups > max_query_groups_) return MatrixQueryStatus::ERR_TOO_MANY_GROUPS;
                grouped_aggregate_i64(q.key_col, q.value_col, q.num_groups, q.agg,
                                      MatrixPredicateI64{q.cmp, q.lo_i64, q.hi_i64}, q.has_filter, out);
                return MatrixQueryStatus::OK;
            }
            if (null_masks_.count(q.value_col)) { out.assign(1, scalar_aggregate_nullable(q)); return MatrixQueryStatus::OK; }
            out.assign(1, static_cast<uint64_t>(
                scan_tiered_column_i64(q.value_col, MatrixPredicateI64{q.cmp, q.lo_i64, q.hi_i64}, q.agg, q.has_filter)));
            return MatrixQueryStatus::OK;
        }
        if (column_type(q.value_col) == MatrixType::F64) {
            if (q.grouped) {
                if (catalog_has(q.key_col) && column_type(q.key_col) != MatrixType::U32)
                    return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE;                  // double key = later
                if (!catalog_has(q.key_col) || q.key_col == q.value_col || q.num_groups == 0
                    || column_rows(q.key_col) != column_rows(q.value_col))
                    return MatrixQueryStatus::ERR_INVALID_GROUP;
                if (q.num_groups > max_query_groups_) return MatrixQueryStatus::ERR_TOO_MANY_GROUPS;
                grouped_aggregate_f64(q.key_col, q.value_col, q.num_groups, q.agg,
                                      MatrixPredicateF64{q.cmp, q.lo_f64, q.hi_f64}, q.has_filter, out);
                return MatrixQueryStatus::OK;
            }
            if (null_masks_.count(q.value_col)) { out.assign(1, scalar_aggregate_nullable(q)); return MatrixQueryStatus::OK; }
            out.assign(1, matrix_bit_cast<uint64_t>(
                scan_tiered_column_f64(q.value_col, MatrixPredicateF64{q.cmp, q.lo_f64, q.hi_f64}, q.agg, q.has_filter)));
            return MatrixQueryStatus::OK;
        }
        if (q.grouped) {
            if (!catalog_has(q.key_col) || q.key_col == q.value_col || q.num_groups == 0
                || catalog_.at(q.key_col)->size_bytes() != catalog_.at(q.value_col)->size_bytes())
                return MatrixQueryStatus::ERR_INVALID_GROUP;
            // A typed (int64) GROUP BY key would be reinterpreted as uint32 by grouped_aggregate; an
            // 8N-byte int64 key of N rows even passes the byte-length guard above (== a 2N-row u32 value).
            // Reject it — typed-key grouping lands in DM-3b. (value_col is already known U32 here.)
            if (column_type(q.key_col) != MatrixType::U32) return MatrixQueryStatus::ERR_UNSUPPORTED_TYPE;
            if (q.num_groups > max_query_groups_) return MatrixQueryStatus::ERR_TOO_MANY_GROUPS;
            if (q.has_filter) grouped_aggregate_pred(q.key_col, q.value_col, q.num_groups, q.agg, MatrixPredicate{q.cmp, q.threshold, q.upper}, out);
            else              grouped_aggregate(q.key_col, q.value_col, q.num_groups, q.agg, out);
        } else {
            // Null-aware path: a column with a null mask skips NULL rows; the filter (if any) is applied too (DM-3j).
            if (null_masks_.count(q.value_col))
                out.assign(1, scalar_aggregate_nullable(q));
            else
                out.assign(1, scan_tiered_column(q.value_col, MatrixPredicate{q.cmp, q.threshold, q.upper}, q.agg, q.has_filter));
        }
        return MatrixQueryStatus::OK;
    }

public:
    ~CPUMockEngine() override {
        // Make the fixed-capacity overflow seam loud (not silent) even in release builds:
        // if any write was dropped because the KVStore filled, report it. Inc 3's SSD
        // spill removes the drop entirely; until then this is the visible failure signal.
        if (store_overflows_ > 0) {
            // Dropped writes = data loss → an ERROR-level diagnostic (prints at the default WARN threshold).
            Log::emit(LogLevel::ERROR, std::to_string(store_overflows_)
                      + " point-op writes dropped — KVStore full (Inc 3 adds SSD spill).");
        }
        std::cerr << "CPUMockEngine shutdown cleanly." << std::endl;
    }

    void execute_batch(DatabaseQuery* batch_array, size_t count) override {
        if (count == 0) return;
        if (count > MATRIX_BATCH_MAX) count = MATRIX_BATCH_MAX;

        // Bin by page (the step the dual-trigger batcher will eventually fold in).
        matrix_bin_by_page(batch_array, count, binned_.data(), offsets_.data());

        // One logical owner per page: process only this page's queries against its
        // contiguous store slice. Pages are independent — this is the parallel unit.
        // Scans arrive on a separate path (execute_scan), so this sees only point ops.
        for (size_t page = 0; page < MATRIX_PAGE_COUNT; ++page) {
            const uint32_t begin = offsets_[page];
            const uint32_t end = offsets_[page + 1];
            for (uint32_t j = begin; j < end; ++j) {
                DatabaseQuery& q = binned_[j];
                if (q.opcode == OP_READ) {
                    uint64_t v = 0;
                    kv_.get(q.query_id, v); // miss leaves v=0 (matches old zero-init store)
                    q.transaction_id = v;
                    ++reads_;
                } else if (q.opcode == OP_WRITE) {
                    // Durability invariant: append to the WAL FIRST (fsync per policy) so a
                    // write is only counted committed once it is durable. The in-memory kv_
                    // is volatile and rebuilt from the WAL on recovery.
                    if (cold_store_) cold_store_->append_put(q.query_id, q.query_id);
                    apply_committed_write(q.query_id, q.query_id);
                }
            }
        }

        // ponytail: read results land in binned_ (reordered), not scattered back to
        // batch_array. Callers here assert on counters + store contents, not on each
        // query's transaction_id, so the scatter-back is YAGNI. Add it if a caller
        // needs per-query read results in original order.
    }

    void execute_scan(DatabaseQuery& q) override {
        // id==0 -> the legacy fixed resident column; id>0 -> a tiered catalog column. The agg op
        // (default AGG_COUNT) selects the reduction; AGG_COUNT preserves the original count result.
        const uint32_t threshold = matrix_get_scan_threshold(q);
        const uint64_t col_id = matrix_get_scan_column_id(q);
        const MatrixAggOp op = matrix_get_scan_agg_op(q);
        const auto st0 = std::chrono::steady_clock::now();
        uint64_t c = 0;
        if (col_id == 0) {
            c = matrix_cpu_reduce(scan_column_.data(), MATRIX_SCAN_COLUMN_SIZE, threshold, op);
        } else {
            c = scan_tiered_column(col_id, MatrixPredicate{MatrixCmp::GT, threshold, 0}, op);
        }
        scan_time_s_ += std::chrono::duration<double>(
            std::chrono::steady_clock::now() - st0).count();
        q.transaction_id = c;
        ++scans_;
        scan_result_sum_ += c;
    }

    uint64_t reads() const override { return reads_; }
    uint64_t writes() const override { return writes_; }
    uint64_t commits() const override { return commits_; }
    uint64_t scans() const override { return scans_; }
    uint64_t scan_result_sum() const override { return scan_result_sum_; }
    double scan_time_s() const override { return scan_time_s_; }
    // CPU has no launch/sync layer, so kernel time == the timed scan loop (zero overhead).
    double scan_kernel_time_s() const override { return scan_time_s_; }

    uint64_t store_checksum() const override {
        return kv_.checksum();
    }

    double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) override {
        std::vector<uint64_t> data(n);
        for (size_t i = 0; i < n; ++i) data[i] = i; // resident; fill not timed
        const auto t0 = std::chrono::steady_clock::now();
        uint64_t count = 0;
        for (size_t i = 0; i < n; ++i) count += (data[i] > threshold);
        const auto t1 = std::chrono::steady_clock::now();
        out_count = count;
        return std::chrono::duration<double>(t1 - t0).count();
    }

    double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) override {
        std::vector<uint32_t> data(n);
        for (size_t i = 0; i < n; ++i) data[i] = static_cast<uint32_t>(i);
        const auto t0 = std::chrono::steady_clock::now();
        uint64_t count = 0;
        for (size_t i = 0; i < n; ++i) count += (data[i] > threshold);
        const auto t1 = std::chrono::steady_clock::now();
        out_count = count;
        return std::chrono::duration<double>(t1 - t0).count();
    }

    double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // ponytail: uint4 vectorized loads are a GPU concept; the CPU compiler already
        // auto-vectorizes the scalar loop, so this just delegates. Keeps the interface
        // uniform without faking a "CPU vectorized" path.
        return benchmark_scan_u32(n, threshold, out_count);
    }

    double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // ponytail: register-blocking is a GPU latency-hiding lever; on CPU it's the same
        // auto-vectorized loop. Delegate.
        return benchmark_scan_u32(n, threshold, out_count);
    }

private:
    // True iff `id` names a real catalog column (id 0 is the legacy fixed column, never a query target).
    bool catalog_has(uint64_t id) const { return id != 0 && catalog_.count(id) != 0; }

    // Append `n` elements of raw bytes to catalog column `id`, growing it (borrow COLD->HOST, append,
    // return the borrow, update the brain's accounting). Private — typed wrappers enforce the element type.
    void append_raw(uint64_t id, const unsigned char* bytes, size_t byte_count) {
        TieredColumn& col = *catalog_.at(id);
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        col.append_bytes(bytes, byte_count);
        if (home != MemorySpace::HOST) col.migrate_to(home);     // return the borrow (grown buffer)
        tier_mgr_.update_bytes(id, col.size_bytes());
    }

    // Scan one catalog column for value>threshold. A cold column is borrowed to HOST for the
    // scan then returned to its home tier, so the engine's residency always matches the brain's
    // accounting (no side-channel migration the budget can't see). Every REBALANCE_EVERY scans,
    // run the brain + executor: promote hot columns (DEVICE inert here), demote the coldest
    // HOST columns to SSD under the budget.
    // Null-aware scalar aggregate over a U32/I64/F64 column (DM-3j): borrow-and-return like
    // scan_tiered_column, but skip NULL rows via the column's null mask. When q.has_filter, the WHERE
    // predicate is applied too (the pred reducers null-check), so a NULL row is excluded whether or not it
    // would match. Returns the result as a uint64 (U32 zero-extended; I64/F64 as the bit pattern).
    uint64_t scalar_aggregate_nullable(const MatrixQuery& q) {
        const uint64_t col_id = q.value_col; const MatrixAggOp op = q.agg;
        TieredColumn& col = *catalog_.at(col_id);
        tier_mgr_.record_access(col_id, col.size_bytes());
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const uint8_t* nulls = null_masks_.at(col_id).data();
        const MatrixType ty = column_type(col_id);
        uint64_t result;
        if (ty == MatrixType::I64) {
            const auto* v = reinterpret_cast<const int64_t*>(col.host_ptr());
            const size_t nn = col.size_bytes() / sizeof(int64_t);
            const int64_t r = q.has_filter
                ? matrix_cpu_reduce_pred_i64(v, nn, MatrixPredicateI64{q.cmp, q.lo_i64, q.hi_i64}, op, nulls)
                : matrix_cpu_reduce_all_i64_nullable(v, nn, op, nulls);
            result = static_cast<uint64_t>(r);
        } else if (ty == MatrixType::F64) {
            const auto* v = reinterpret_cast<const double*>(col.host_ptr());
            const size_t nn = col.size_bytes() / sizeof(double);
            const double r = q.has_filter
                ? matrix_cpu_reduce_pred_f64(v, nn, MatrixPredicateF64{q.cmp, q.lo_f64, q.hi_f64}, op, nulls)
                : matrix_cpu_reduce_all_f64_nullable(v, nn, op, nulls);
            result = matrix_bit_cast<uint64_t>(r);
        } else {
            const auto* v = reinterpret_cast<const uint32_t*>(col.host_ptr());
            const size_t nn = col.size_bytes() / sizeof(uint32_t);
            result = q.has_filter
                ? matrix_cpu_reduce_pred(v, nn, MatrixPredicate{q.cmp, q.threshold, q.upper}, op, nulls)
                : matrix_cpu_reduce_all_nullable(v, nn, op, nulls);
        }
        if (home != MemorySpace::HOST) col.migrate_to(home);
        maybe_rebalance();
        return result;
    }

    uint64_t scan_tiered_column(uint64_t col_id, MatrixPredicate pred, MatrixAggOp op, bool has_filter = true) {
        auto it = catalog_.find(col_id);
        if (it == catalog_.end()) {
            assert(false && "scan of unregistered column id"); // debug: catch the caller bug
            std::cerr << "CPUMockEngine: scan of unregistered column id " << col_id
                      << " — empty result\n";                 // release: diagnosable, no null-deref
            return 0;
        }
        TieredColumn& col = *it->second;
        tier_mgr_.record_access(col_id, col.size_bytes());          // heat signal
#if defined(MATRIX_USE_CUDA)
        if (col.tier() == MemorySpace::DEVICE) {                    // GPU-3: reduce in VRAM, no borrow-down
            const size_t nvals = col.size_bytes() / sizeof(uint32_t);
            const uint64_t result = matrix_gpu_reduce_dev_u32(col.device_ptr(), nvals, pred, op, has_filter);
            maybe_rebalance();
            return result;
        }
#endif

        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); } // pull SSD->RAM to scan
        const uint32_t* vals = reinterpret_cast<const uint32_t*>(col.host_ptr());
        const size_t nvals = col.size_bytes() / sizeof(uint32_t);
        const uint64_t result = has_filter ? matrix_cpu_reduce_pred(vals, nvals, pred, op)
                                           : matrix_cpu_reduce_all(vals, nvals, op);
        // ponytail: returning the borrow rewrites the COLD file each cold scan; skip-if-unchanged
        // (or a TierManager note_residency) is the upgrade path if cold-scan churn ever matters.
        if (home != MemorySpace::HOST) col.migrate_to(home);        // return the borrow

        maybe_rebalance();
        return result;
    }

    // Every REBALANCE_EVERY scans, run the brain + executor: promote hot (DEVICE inert here), demote the
    // coldest HOST columns to SSD under the budget. Shared by the u32 and int64 scan paths.
    void maybe_rebalance() {
        if (++scans_since_rebalance_ >= rebalance_every_) {
            std::unordered_map<uint64_t, TieredColumn*> ptrs;
            for (auto& kv : catalog_) ptrs[kv.first] = kv.second.get();
            migrations_ += executor_.apply(tier_mgr_.rebalance(), ptrs);
            ++rebalances_;
            scans_since_rebalance_ = 0;
        }
    }

    // int64 scalar scan over a catalog column (DM-3a unfiltered; DM-3b adds the filtered path). Same
    // borrow-to-HOST-and-return as scan_tiered_column: record heat, pull a COLD column to RAM, reinterpret
    // the raw bytes as int64, reduce by op (filtered when has_filter, else over all), return the borrow to
    // its home tier, and drive the shared rebalance cadence (int64 scans now count too — DM-3a follow-up).
    int64_t scan_tiered_column_i64(uint64_t col_id, MatrixPredicateI64 pred, MatrixAggOp op, bool has_filter = false) {
        TieredColumn& col = *catalog_.at(col_id);
        tier_mgr_.record_access(col_id, col.size_bytes());
#if defined(MATRIX_USE_CUDA)
        if (col.tier() == MemorySpace::DEVICE) {                    // GPU-3: reduce in VRAM, no borrow-down
            const size_t nvals = col.size_bytes() / sizeof(int64_t);
            const int64_t result = matrix_gpu_reduce_dev_i64(col.device_ptr(), nvals, pred, op, has_filter);
            maybe_rebalance();
            return result;
        }
#endif
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const int64_t* vals = reinterpret_cast<const int64_t*>(col.host_ptr());
        const size_t nvals = col.size_bytes() / sizeof(int64_t);
        const int64_t result = has_filter ? matrix_cpu_reduce_pred_i64(vals, nvals, pred, op)
                                          : matrix_cpu_reduce_all_i64(vals, nvals, op);
        if (home != MemorySpace::HOST) col.migrate_to(home);
        maybe_rebalance();
        return result;
    }

    // double scalar scan over a catalog column (DM-3e). Same borrow-to-HOST-and-return as
    // scan_tiered_column_i64: record heat, pull a COLD column to RAM, reinterpret the raw bytes as
    // double, reduce by op (filtered when has_filter, else over all), return the borrow to its home
    // tier, and drive the shared rebalance cadence.
    double scan_tiered_column_f64(uint64_t col_id, MatrixPredicateF64 pred, MatrixAggOp op, bool has_filter = false) {
        TieredColumn& col = *catalog_.at(col_id);
        tier_mgr_.record_access(col_id, col.size_bytes());
#if defined(MATRIX_USE_CUDA)
        if (col.tier() == MemorySpace::DEVICE) {                    // GPU-3: reduce in VRAM, no borrow-down
            const size_t nvals = col.size_bytes() / sizeof(double);
            const double result = matrix_gpu_reduce_dev_f64(col.device_ptr(), nvals, pred, op, has_filter);
            maybe_rebalance();
            return result;
        }
#endif
        const MemorySpace home = col.tier();
        if (home != MemorySpace::HOST) { ++cold_borrows_; col.migrate_to(MemorySpace::HOST); }
        const double* vals = reinterpret_cast<const double*>(col.host_ptr());
        const size_t nvals = col.size_bytes() / sizeof(double);
        const double result = has_filter ? matrix_cpu_reduce_pred_f64(vals, nvals, pred, op)
                                         : matrix_cpu_reduce_all_f64(vals, nvals, op);
        if (home != MemorySpace::HOST) col.migrate_to(home);
        maybe_rebalance();
        return result;
    }

    // Point-op store: a real hash table (gap DM-1). Distinct keys never overwrite; a full
    // table is an explicit error, not silent loss. Sized with headroom over the demo's
    // distinct write-keys; real capacity / SSD-spill is gap DM-9 / Inc 3 (the seam).
    // MATRIX_STORE_SLOTS (not a bare literal): DM-1b fixed a real bug where this and the GPU
    // engine's own point-op store capacity had silently drifted to two different values — sharing
    // one symbol makes that impossible to repeat.
    KVStore kv_{MATRIX_STORE_SLOTS}; // 65536 slots
    // Ordered secondary index (DM-7): key -> value, mirrors kv_ (maintained on commit, rebuilt on recovery).
    // Enables log-time range scans (kv_range_sorted) vs kv_range's O(n) full scan.
    // ponytail: a std::map mirror — doubles point-op key memory + a map insert per commit; a packed
    // B-tree/ART would be denser, and the index is rebuilt from kv_ on restart (not separately persisted).
    std::map<uint64_t, uint64_t> key_index_;
    std::unique_ptr<ColdStore> cold_store_; // null = durability off (default); set via WAL path
    std::string checkpoint_path_;     // <wal_path>.ckpt — last point-op compaction snapshot
    uint64_t checkpoints_ = 0;        // DU-4: number of WAL compactions performed
    // --- live tiering (INT-1): a catalog of analytical columns the brain auto-tiers ---
    static constexpr uint64_t REBALANCE_EVERY = 4;     // default: rebalance every N tiered scans
    uint64_t rebalance_every_ = REBALANCE_EVERY;       // OB-4: runtime-tunable rebalance cadence (default = the constant)
    static constexpr uint32_t MATRIX_CATALOG_MAGIC = 0x4D434132u; // 'MCA2' — typed+named catalog snapshot v2 (DM-2b)
    static constexpr uint32_t MATRIX_CKPT_MAGIC = 0x4D434B50u; // 'MCKP' — point-op checkpoint file
    static constexpr uint32_t MAX_QUERY_GROUPS = 1u << 28; // default grouped-query num_groups ceiling (out alloc guard)
    uint32_t max_query_groups_ = MAX_QUERY_GROUPS;     // RM-2: runtime-tunable admission cap (default = the constant)
    TierManager tier_mgr_;                              // decides migrations (heat-driven)
    MigrationExecutor executor_;                        // moves the bytes per decision
    std::unordered_map<uint64_t, std::unique_ptr<TieredColumn>> catalog_; // id -> column
    std::unordered_map<uint64_t, MatrixType> col_types_; // id -> element type (absent ⇒ U32); DM-3a
    std::unordered_map<uint64_t, std::vector<std::string>> string_columns_; // DM-3i: separate string-column store
    std::unordered_map<uint64_t, std::vector<std::string>> string_dicts_;   // dict-encoded: code -> string (decode)
    std::unordered_map<uint64_t, std::unordered_map<std::string, uint32_t>> string_encoders_; // string -> code (encode/filter)
    std::unordered_map<uint64_t, std::vector<uint8_t>> null_masks_;          // DM-3j: id -> per-row NULL flag (1=null)
    std::unordered_map<uint64_t, std::string> column_names_;   // id -> optional name
    std::unordered_map<std::string, uint64_t> name_to_id_;     // name -> id (resolve)
    std::unordered_map<std::string, std::vector<uint64_t>> tables_;   // table name -> ordered column ids (DM-2c)
    uint64_t scans_since_rebalance_ = 0;
    uint64_t cold_borrows_ = 0;    // observability: COLD->HOST borrows
    uint64_t rebalances_ = 0;      // observability: rebalance passes
    uint64_t migrations_ = 0;      // observability: migration decisions executed
    // Record one query's latency into the (atomic) read-path stats. Shared by execute_query and
    // execute_query_shared, so it is safe under concurrent readers (relaxed atomics; single-threaded
    // values are unchanged). OB-2b: bucket by floor(log2(ns+1)) for percentile estimation.
    void record_query_latency(uint64_t ns) {
        query_count_.fetch_add(1, std::memory_order_relaxed);
        total_query_ns_.fetch_add(ns, std::memory_order_relaxed);
        for (uint64_t cur = max_query_ns_.load(std::memory_order_relaxed);
             ns > cur && !max_query_ns_.compare_exchange_weak(cur, ns, std::memory_order_relaxed); ) {}
        uint64_t x = ns + 1, b = 0; while (x > 1 && b < LAT_BUCKETS - 1) { x >>= 1; ++b; }
        latency_hist_[b].fetch_add(1, std::memory_order_relaxed);
    }

    // Read-path stats: bumped by every execute_query / execute_query_shared, so atomic for concurrent
    // readers (relaxed — heuristics; values are identical single-threaded). All other counters are
    // exclusive-path-only and stay plain.
    std::atomic<uint64_t> query_count_{0};     // observability: execute_query calls served (OK and ERR)
    std::atomic<uint64_t> total_query_ns_{0};  // observability: summed execute_query wall-time (ns)
    std::atomic<uint64_t> max_query_ns_{0};    // observability: slowest single execute_query (ns)
    static constexpr int LAT_BUCKETS = 40;                  // log2 latency buckets (OB-2b); 2^39 ns ≈ 9 min
    std::array<std::atomic<uint64_t>, LAT_BUCKETS> latency_hist_{};   // per-query latency histogram (bucket = floor(log2(ns+1)))
    std::vector<DatabaseQuery> binned_;                // scratch: batch sorted by page
    std::array<uint32_t, MATRIX_PAGE_COUNT + 1> offsets_{}; // CSR page offsets
    std::vector<uint32_t> scan_column_;                // resident analytical column
    uint64_t reads_ = 0;
    uint64_t writes_ = 0;
    uint64_t commits_ = 0;
    uint64_t scans_ = 0;
    uint64_t scan_result_sum_ = 0;
    uint64_t store_overflows_ = 0; // writes dropped because the fixed-capacity KVStore was full (Inc 3 adds SSD spill)
    double scan_time_s_ = 0.0;

    // Apply one durable write to the point-op store with the standard overflow accounting.
    // mock projection: value == key. Fixed-capacity seam: a full table is counted as an
    // overflow (always live, even under NDEBUG) so a dropped write is never silent. Inc 3
    // replaces this with SSD spill; the assert makes it fail loud in debug builds too.
    void apply_committed_write(uint64_t key, uint64_t value) {
        ++writes_;
        if (kv_.put(key, value)) { ++commits_; key_index_[key] = value; }   // mirror into the ordered secondary index
        else { ++store_overflows_; assert(false && "KVStore full — point-op store capacity exceeded (Inc 3 adds spill)"); }
    }

    // --- Atomic transactions (WAL group commit) ---
    std::vector<std::pair<uint64_t, uint64_t>> txn_buf_; // pending writes in the open transaction
    bool in_txn_ = false;
    uint64_t transactions_committed_ = 0;
    uint64_t transactions_rolled_back_ = 0;
};


In [ ]:
%%writefile compute_cuda.cuh
#pragma once

// CUDA backend — page-ownership model (Component 4: Parallel Engine).
// Compile on a GPU host (Google Colab) with:
//     nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA main.cpp -o matrixdb_proto
//
// One CUDA BLOCK owns one page. The batch is binned by page on the host (CSR offsets),
// so block p processes only page p's contiguous queries against page p's store slice.
// Different blocks touch disjoint store slots ⇒ no cross-block conflict, no store
// atomics, no delta log. Each page has a single owner; pages are independent
// (shared-nothing). Threads within a block stride over the page's queries.

#include "compute.hpp"
#include <cuda_runtime.h>
#include <vector>
#include <chrono>
#include <cstdio>
#include <cstdlib>
#include <limits>
#include <cstdint>

#define CUDA_CHECK(call)                                                            \
    do {                                                                           \
        cudaError_t _err = (call);                                                 \
        if (_err != cudaSuccess) {                                                 \
            std::fprintf(stderr, "CUDA error %s at %s:%d -> %s\n",                 \
                         #call, __FILE__, __LINE__, cudaGetErrorString(_err));     \
            std::abort();                                                          \
        }                                                                          \
    } while (0)

// One block per page. blockIdx.x == page id; the block's threads cooperatively process
// that page's queries [offsets[page], offsets[page+1]). Writes to a slot within a page
// are owned by this block alone, so no atomics on the store are needed. Same-slot writes
// within the page race between threads -> last-writer-wins, matching the CPU mock's
// deterministic in-order result only when keys are unique.
//
// DM-1b (PRODUCTION_READINESS.md), fixed for the actual demonstrated case, pending hardware
// verification: `store[slot]` below is a flat MATRIX_STORE_SLOTS-size array indexed by `key & MASK`
// with no probing -- unlike kv_store.hpp's KVStore (open-addressing), two DIFFERENT keys that
// collide on the same slot silently overwrite each other here. This USED to be guaranteed for the
// standard benchmark workload (MATRIX_STORE_SLOTS was 4096 while BATCH_MAX/KVStore's capacity was
// 65536) -- fixed by matching MATRIX_STORE_SLOTS to BATCH_MAX (see types.hpp's own note), which
// makes the sequential-key benchmark workload a perfect bijection: zero collisions by construction.
// test_gpu_pointop_collision.cu proves this on real hardware (build+run on Colab -- no nvcc in the
// sandbox this was fixed in). Narrower residual caveat, NOT fixed (still real, still needs a GPU to
// address if it ever matters): a batch with more than STORE_SLOTS truly-distinct keys, or a
// deliberately duplicate key within one page, still collides/races the same way -- this kernel was
// confirmed unreachable from anything exercised in this repo before this fix (main.cpp hardcodes
// point-ops to the CPU engine unconditionally; no other test called execute_batch), so the residual
// case was judged not worth a general-purpose GPU hash table for code nothing currently calls.
__global__ void matrix_page_kernel(const DatabaseQuery* binned, const uint32_t* offsets,
                                   uint64_t* store,
                                   unsigned long long* reads,
                                   unsigned long long* writes) {
    const size_t page = blockIdx.x;
    if (page >= MATRIX_PAGE_COUNT) return;

    const uint32_t begin = offsets[page];
    const uint32_t end = offsets[page + 1];

    unsigned r = 0, w = 0;
    for (uint32_t j = begin + threadIdx.x; j < end; j += blockDim.x) {
        const DatabaseQuery q = binned[j];
        const size_t slot = q.query_id & MATRIX_STORE_MASK;
        if (q.opcode == OP_READ) {
            volatile uint64_t v = store[slot]; (void)v; // touch the store (read path)
            ++r;
        } else if (q.opcode == OP_WRITE) {
            store[slot] = q.query_id; // mock projection: value == key
            ++w;
        }
    }
    if (r) atomicAdd(reads, (unsigned long long)r);   // counters only — not on the store
    if (w) atomicAdd(writes, (unsigned long long)w);
}

// Filter-count scan: count values > threshold. Grid-stride over resident VRAM data,
// block-local reduction, one atomicAdd per block. This is the GPU's home turf —
// streaming bandwidth over data too large for CPU cache.
__global__ void matrix_scan_kernel(const uint64_t* data, size_t n,
                                   uint64_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        if (data[i] > threshold) ++local;
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// uint32 column variant — half the bytes/value. Same grid-stride filter-count.
__global__ void matrix_scan_kernel_u32(const uint32_t* data, size_t n,
                                       uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        if (data[i] > threshold) ++local;
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// AGG-2 (GPU-1): SUM / MIN / MAX of values matching `value > threshold`, same grid-stride +
// block-local + one-atomic-per-block shape as the COUNT kernel, differing only in accumulator + atomic.
// Each is correct iff its result equals matrix_cpu_reduce(host, n, threshold, op) over the same bytes
// (the cross-backend invariant — test_gpu_agg.cu is the merge gate). Empty-match sentinels match the CPU:
// SUM 0, MIN UINT64_MAX, MAX 0. `out` must be pre-initialized to the sentinel before launch.
__global__ void matrix_sum_kernel_u32(const uint32_t* data, size_t n,
                                      uint32_t threshold, unsigned long long* out) {
    __shared__ unsigned long long blk;
    if (threadIdx.x == 0) blk = 0;
    __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (data[i] > threshold) local += data[i];
    atomicAdd(&blk, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_min_kernel_u32(const uint32_t* data, size_t n,
                                      uint32_t threshold, unsigned long long* out) {
    __shared__ unsigned long long blk;
    if (threadIdx.x == 0) blk = 0xFFFFFFFFFFFFFFFFull;   // UINT64_MAX (CPU empty-MIN sentinel)
    __syncthreads();
    unsigned long long local = 0xFFFFFFFFFFFFFFFFull;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (data[i] > threshold && data[i] < local) local = data[i];
    atomicMin(&blk, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicMin(out, blk);
}
__global__ void matrix_max_kernel_u32(const uint32_t* data, size_t n,
                                      uint32_t threshold, unsigned long long* out) {
    __shared__ unsigned long long blk;
    if (threadIdx.x == 0) blk = 0;                       // 0 (CPU empty-MAX sentinel)
    __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (data[i] > threshold && data[i] > local) local = data[i];
    atomicMax(&blk, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicMax(out, blk);
}

// === GPU-4 (predicates) + GPU-2 (grouped) — ADDITIVE; the GPU-1 threshold kernels above are untouched. ===
// Device predicate mirroring matrix_pred_match (compute.hpp) so a GPU WHERE matches the CPU's exactly.
__device__ __forceinline__ bool matrix_pred_match_dev(uint32_t v, MatrixCmp c, uint32_t a, uint32_t b) {
    switch (c) {
        case MatrixCmp::GE:      return v >= a;
        case MatrixCmp::LT:      return v <  a;
        case MatrixCmp::LE:      return v <= a;
        case MatrixCmp::EQ:      return v == a;
        case MatrixCmp::NE:      return v != a;
        case MatrixCmp::BETWEEN: return v >= a && v <= b;
        case MatrixCmp::GT:
        default:                 return v >  a;
    }
}
// GPU-4: predicate-filtered scalar reductions — the general sibling of the GPU-1 threshold kernels.
// Each == matrix_cpu_reduce_pred(host, n, {cmp,a,b}, op). Sentinels: COUNT/SUM 0, MIN UINT64_MAX, MAX 0
// (caller pre-inits `out` to the same: cudaMemset 0x00 for COUNT/SUM/MAX, 0xFF for MIN).
__global__ void matrix_count_kernel_pred_u32(const uint32_t* data, size_t n, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_dev(data[i], c, a, b)) ++local;
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_sum_kernel_pred_u32(const uint32_t* data, size_t n, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_dev(data[i], c, a, b)) local += data[i];
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_min_kernel_pred_u32(const uint32_t* data, size_t n, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0xFFFFFFFFFFFFFFFFull; __syncthreads();
    unsigned long long local = 0xFFFFFFFFFFFFFFFFull;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_dev(data[i], c, a, b) && data[i] < local) local = data[i];
    atomicMin(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMin(out, blk);
}
__global__ void matrix_max_kernel_pred_u32(const uint32_t* data, size_t n, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_dev(data[i], c, a, b) && data[i] > local) local = data[i];
    atomicMax(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMax(out, blk);
}
// GPU-2: grouped reduction — one atomic per row into its dense group slot out[k] (k = keys[i], k < num_groups;
// out-of-range keys ignored, matching the CPU). Each == matrix_cpu_group_reduce(keys, vals, n, num_groups, op).
// Caller pre-inits out[num_groups] per op via cudaMemset: COUNT/SUM/MAX -> 0x00, MIN -> 0xFF (UINT64_MAX).
// ponytail: simple global-atomic per row (correct first); a per-block shared-memory privatized histogram is
// the contention-reduction follow-up for high group counts.
__global__ void matrix_group_count_kernel(const uint32_t* keys, size_t n, uint32_t num_groups, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups) atomicAdd(&out[k], 1ull);
    }
}
__global__ void matrix_group_sum_kernel(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups) atomicAdd(&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_min_kernel(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups) atomicMin(&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_max_kernel(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups) atomicMax(&out[k], (unsigned long long)vals[i]);
    }
}

// Filtered grouped kernels (GROUP BY ... WHERE pred(value)) — same one-atomic-per-row shape as the
// unfiltered ones above, gated by the verified matrix_pred_match_dev on the VALUE (matches the CPU's
// matrix_cpu_group_reduce_pred). COUNT reads vals too (it must apply the predicate). Sentinels identical
// to the unfiltered path (COUNT/SUM/MAX 0, MIN ULLONG_MAX).
__global__ void matrix_group_count_kernel_pred(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_dev(vals[i], c, a, b)) atomicAdd(&out[k], 1ull);
    }
}
__global__ void matrix_group_sum_kernel_pred(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_dev(vals[i], c, a, b)) atomicAdd(&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_min_kernel_pred(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_dev(vals[i], c, a, b)) atomicMin(&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_max_kernel_pred(const uint32_t* keys, const uint32_t* vals, size_t n, uint32_t num_groups, MatrixCmp c, uint32_t a, uint32_t b, unsigned long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_dev(vals[i], c, a, b)) atomicMax(&out[k], (unsigned long long)vals[i]);
    }
}

// === GPU-5 (typed int64 + double) — ADDITIVE; the u32 kernels above are untouched. ===
// Device predicates mirroring matrix_pred_match_i64 / matrix_pred_match_f64 (compute.hpp).
__device__ __forceinline__ bool matrix_pred_match_i64_dev(long long v, MatrixCmp c, long long a, long long b) {
    switch (c) {
        case MatrixCmp::GE:      return v >= a;
        case MatrixCmp::LT:      return v <  a;
        case MatrixCmp::LE:      return v <= a;
        case MatrixCmp::EQ:      return v == a;
        case MatrixCmp::NE:      return v != a;
        case MatrixCmp::BETWEEN: return v >= a && v <= b;
        case MatrixCmp::GT:
        default:                 return v >  a;
    }
}
__device__ __forceinline__ bool matrix_pred_match_f64_dev(double v, MatrixCmp c, double a, double b) {
    switch (c) {
        case MatrixCmp::GE:      return v >= a;
        case MatrixCmp::LT:      return v <  a;
        case MatrixCmp::LE:      return v <= a;
        case MatrixCmp::EQ:      return v == a;
        case MatrixCmp::NE:      return v != a;
        case MatrixCmp::BETWEEN: return v >= a && v <= b;
        case MatrixCmp::GT:
        default:                 return v >  a;
    }
}
// double has no native atomicMin/Max — CAS loop on the bit pattern (the standard idiom). Works on
// shared or global. NaN: matches the CPU's IEEE behavior (a NaN val never updates, since cur<=NaN is false).
__device__ __forceinline__ double atomicMinDouble(double* addr, double val) {
    unsigned long long* p = (unsigned long long*)addr;
    unsigned long long old = *p, assumed;
    do { assumed = old;
         if (__longlong_as_double(assumed) <= val) break;
         old = atomicCAS(p, assumed, __double_as_longlong(val));
    } while (assumed != old);
    return __longlong_as_double(old);
}
__device__ __forceinline__ double atomicMaxDouble(double* addr, double val) {
    unsigned long long* p = (unsigned long long*)addr;
    unsigned long long old = *p, assumed;
    do { assumed = old;
         if (__longlong_as_double(assumed) >= val) break;
         old = atomicCAS(p, assumed, __double_as_longlong(val));
    } while (assumed != old);
    return __longlong_as_double(old);
}
// Native atomicAdd(double*) exists on arch >= 6.0 and in the host pass; nvcc without an explicit -arch
// can default the device pass below 6.0, where the overload is absent. Provide the CAS fallback ONLY in
// that device pass (defined(__CUDA_ARCH__) && < 600) — NOT the host pass, where the native one is already
// declared (defining it there is the redefinition Colab caught).
#if defined(__CUDA_ARCH__) && __CUDA_ARCH__ < 600
__device__ __forceinline__ double atomicAdd(double* address, double val) {
    unsigned long long* p = (unsigned long long*)address;
    unsigned long long old = *p, assumed;
    do { assumed = old;
         old = atomicCAS(p, assumed, __double_as_longlong(val + __longlong_as_double(assumed)));
    } while (assumed != old);
    return __longlong_as_double(old);
}
#endif
#define MATRIX_DEV_POSINF __longlong_as_double((long long)0x7FF0000000000000ULL)
#define MATRIX_DEV_NEGINF __longlong_as_double((long long)0xFFF0000000000000ULL)

// int64 predicate-filtered reductions. out is `long long*`; sentinels match matrix_cpu_reduce_pred_i64
// (COUNT/SUM 0, MIN INT64_MAX, MAX INT64_MIN — caller cudaMemcpy's the init). SUM accumulates as unsigned
// (two's-complement add is bit-identical to signed; same overflow-wrap as the CPU).
__global__ void matrix_count_kernel_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_i64_dev(d[i], c, a, b)) ++local;
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd((unsigned long long*)out, blk);
}
__global__ void matrix_sum_kernel_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, long long* out) {
    __shared__ unsigned long long blk; if (threadIdx.x == 0) blk = 0; __syncthreads();
    long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_i64_dev(d[i], c, a, b)) local += d[i];
    atomicAdd(&blk, (unsigned long long)local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd((unsigned long long*)out, blk);
}
__global__ void matrix_min_kernel_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, long long* out) {
    __shared__ long long blk; if (threadIdx.x == 0) blk = INT64_MAX; __syncthreads();
    long long local = INT64_MAX;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_i64_dev(d[i], c, a, b) && d[i] < local) local = d[i];
    atomicMin(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMin(out, blk);
}
__global__ void matrix_max_kernel_pred_i64(const long long* d, size_t n, MatrixCmp c, long long a, long long b, long long* out) {
    __shared__ long long blk; if (threadIdx.x == 0) blk = INT64_MIN; __syncthreads();
    long long local = INT64_MIN;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_i64_dev(d[i], c, a, b) && d[i] > local) local = d[i];
    atomicMax(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMax(out, blk);
}
// double predicate-filtered reductions. out is `double*`; sentinels match matrix_cpu_reduce_pred_f64
// (COUNT/SUM 0.0, MIN +inf, MAX -inf — caller cudaMemcpy's the init). atomicAdd(double*) needs arch>=6.0
// (T4 is 7.5); MIN/MAX use the CAS-loop helpers above.
__global__ void matrix_count_kernel_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, double* out) {
    __shared__ double blk; if (threadIdx.x == 0) blk = 0.0; __syncthreads();
    double local = 0.0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_f64_dev(d[i], c, a, b)) local += 1.0;
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_sum_kernel_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, double* out) {
    __shared__ double blk; if (threadIdx.x == 0) blk = 0.0; __syncthreads();
    double local = 0.0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_f64_dev(d[i], c, a, b)) local += d[i];
    atomicAdd(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicAdd(out, blk);
}
__global__ void matrix_min_kernel_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, double* out) {
    __shared__ double blk; if (threadIdx.x == 0) blk = MATRIX_DEV_POSINF; __syncthreads();
    double local = MATRIX_DEV_POSINF;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_f64_dev(d[i], c, a, b) && d[i] < local) local = d[i];
    atomicMinDouble(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMinDouble(out, blk);
}
__global__ void matrix_max_kernel_pred_f64(const double* d, size_t n, MatrixCmp c, double a, double b, double* out) {
    __shared__ double blk; if (threadIdx.x == 0) blk = MATRIX_DEV_NEGINF; __syncthreads();
    double local = MATRIX_DEV_NEGINF;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n; i += stride)
        if (matrix_pred_match_f64_dev(d[i], c, a, b) && d[i] > local) local = d[i];
    atomicMaxDouble(&blk, local); __syncthreads();
    if (threadIdx.x == 0) atomicMaxDouble(out, blk);
}
// Vectorized uint32 scan: each thread loads 4 values per instruction via uint4
// (LDG.128 = 16 bytes/load). This is the standard memory-bound fix across DL/DB/HPC
// kernels — more bytes in flight per thread => deeper memory-level parallelism =>
// closer to peak DRAM bandwidth than the scalar 4-byte load. Requires n % 4 == 0 and
// 16-byte-aligned data (cudaMalloc guarantees alignment; n is a power of two here).
__global__ void matrix_scan_kernel_u32x4(const uint4* data4, size_t n4,
                                         uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x * blockDim.x + threadIdx.x; i < n4; i += stride) {
        const uint4 v = data4[i]; // one 16-byte load -> 4 comparisons
        local += (v.x > threshold) + (v.y > threshold)
               + (v.z > threshold) + (v.w > threshold);
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

// Items-per-thread uint32 scan (register blocking) — the lever CUB's BlockReduce uses
// (128 threads x 4 items each). Each thread issues ITEMS independent loads into a
// register array BEFORE comparing any, so multiple loads are in flight per thread =>
// DRAM latency is hidden => bandwidth saturates. Unlike uint4 this needs no special
// type/alignment and doesn't inflate per-load register width. Strided access keeps the
// warp's ITEMS loads coalesced (lane L reads base+L, base+L+blockDim, ...).
template <int ITEMS>
__global__ void matrix_scan_kernel_ipt(const uint32_t* data, size_t n,
                                       uint32_t threshold, unsigned long long* count) {
    __shared__ unsigned long long block_count;
    if (threadIdx.x == 0) block_count = 0;
    __syncthreads();

    unsigned long long local = 0;
    const size_t base_stride = (size_t)gridDim.x * blockDim.x;
    const size_t chunk = base_stride * ITEMS;
    // Striped layout: thread's global id is the base; item k is base + k*base_stride.
    // (base init must NOT include *ITEMS — that mixes the blocked convention with the
    // striped strides below and leaves most indices unvisited. Verified by
    // test_scan_coverage.cpp.)
    for (size_t base = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
         base < n; base += chunk) {
        uint32_t reg[ITEMS];
        #pragma unroll
        for (int k = 0; k < ITEMS; ++k) {
            const size_t idx = base + (size_t)k * base_stride;
            reg[k] = (idx < n) ? data[idx] : 0u; // load all ITEMS first (in flight)
        }
        #pragma unroll
        for (int k = 0; k < ITEMS; ++k) {
            const size_t idx = base + (size_t)k * base_stride;
            if (idx < n) local += (reg[k] > threshold); // then compare
        }
    }
    atomicAdd(&block_count, local);
    __syncthreads();
    if (threadIdx.x == 0) atomicAdd(count, block_count);
}

/**
 * @brief GPU-3 device-resident scalar reduce. Run the (already cross-checked) predicate kernels over a
 * catalog column's VRAM bytes and return the SAME bit-encoding as matrix_cpu_reduce_pred / _all, so
 * CPUMockEngine::execute_query's DEVICE path == its CPU path. Forward-declared in compute_mock.cpp (which
 * is included before this header in the nvcc TU) and called from scan_tiered_column* when tier()==DEVICE.
 * Unfiltered (!has_filter) synthesizes an always-true predicate (GE min) to mirror matrix_cpu_reduce_all.
 * ponytail: an always-true pred drops NaN rows that reduce_all would count — fine for real numeric data
 * (no NaN in the cross-check); add dedicated unfiltered kernels only if NaN COUNT ever matters.
 */
inline uint64_t matrix_gpu_reduce_dev_u32(const void* d_data, size_t n,
                                          MatrixPredicate pred, MatrixAggOp op, bool has_filter) {
    const MatrixCmp c = has_filter ? pred.cmp : MatrixCmp::GE;
    const uint32_t  a = has_filter ? pred.a   : 0u;            // GE 0 == every u32
    const uint32_t  b = has_filter ? pred.b   : 0u;
    const uint32_t* d = reinterpret_cast<const uint32_t*>(d_data);
    unsigned long long* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, sizeof(unsigned long long)));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      { CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(*d_out))); matrix_sum_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out); }
    else if (op == AGG_MIN) { CUDA_CHECK(cudaMemset(d_out, 0xFF, sizeof(*d_out))); matrix_min_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out); }
    else if (op == AGG_MAX) { CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(*d_out))); matrix_max_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out); }
    else                    { CUDA_CHECK(cudaMemset(d_out, 0x00, sizeof(*d_out))); matrix_count_kernel_pred_u32<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out); }
    CUDA_CHECK(cudaGetLastError());
    unsigned long long h = 0; CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(h), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}
inline int64_t matrix_gpu_reduce_dev_i64(const void* d_data, size_t n,
                                         MatrixPredicateI64 pred, MatrixAggOp op, bool has_filter) {
    const MatrixCmp  c = has_filter ? pred.cmp : MatrixCmp::GE;
    const long long  a = has_filter ? pred.a   : INT64_MIN;    // GE INT64_MIN == every i64
    const long long  b = has_filter ? pred.b   : 0;
    const long long* d = reinterpret_cast<const long long*>(d_data);
    long long* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, sizeof(long long)));
    long long init = (op == AGG_MIN) ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0);   // sentinels aren't memset-able
    CUDA_CHECK(cudaMemcpy(d_out, &init, sizeof(init), cudaMemcpyHostToDevice));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_sum_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_min_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_max_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else                    matrix_count_kernel_pred_i64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    long long h = 0; CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(h), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return static_cast<int64_t>(h);
}
inline double matrix_gpu_reduce_dev_f64(const void* d_data, size_t n,
                                        MatrixPredicateF64 pred, MatrixAggOp op, bool has_filter) {
    const double inf = std::numeric_limits<double>::infinity();
    const MatrixCmp c = has_filter ? pred.cmp : MatrixCmp::GE;
    const double    a = has_filter ? pred.a   : -inf;          // GE -inf == every non-NaN f64
    const double    b = has_filter ? pred.b   : 0.0;
    const double*   d = reinterpret_cast<const double*>(d_data);
    double* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, sizeof(double)));
    double init = (op == AGG_MIN) ? inf : (op == AGG_MAX ? -inf : 0.0);
    CUDA_CHECK(cudaMemcpy(d_out, &init, sizeof(init), cudaMemcpyHostToDevice));
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_sum_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_min_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_max_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    else                    matrix_count_kernel_pred_f64<<<BLOCKS, TPB>>>(d, n, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    double h = 0; CUDA_CHECK(cudaMemcpy(&h, d_out, sizeof(h), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
    return h;
}

/**
 * @brief GPU-3 (grouped) device-resident GROUP BY. Run the grouped kernels over key+value VRAM bytes and
 * write num_groups results into out_host with the SAME per-group encoding/sentinels as
 * matrix_cpu_group_reduce(_pred), so CPUMockEngine's grouped DEVICE path is bit-identical to its CPU path.
 * Forward-declared in compute_mock.cpp; u32 key+value only (matches the verified kernels), no null mask.
 */
inline void matrix_gpu_group_reduce_dev(const void* keys_dev, const void* vals_dev, size_t n,
                                        uint32_t num_groups, MatrixAggOp op, MatrixPredicate pred,
                                        bool has_filter, uint64_t* out_host) {
    const uint32_t* dk = reinterpret_cast<const uint32_t*>(keys_dev);
    const uint32_t* dv = reinterpret_cast<const uint32_t*>(vals_dev);
    unsigned long long* d_out = nullptr;
    CUDA_CHECK(cudaMalloc(&d_out, (size_t)num_groups * sizeof(unsigned long long)));
    CUDA_CHECK(cudaMemset(d_out, (op == AGG_MIN) ? 0xFF : 0x00, (size_t)num_groups * sizeof(unsigned long long))); // MIN -> ULLONG_MAX
    constexpr int TPB = 256, BLOCKS = 1024;
    if (has_filter) {
        const MatrixCmp c = pred.cmp; const uint32_t a = pred.a, b = pred.b;
        if (op == AGG_SUM)      matrix_group_sum_kernel_pred<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
        else if (op == AGG_MIN) matrix_group_min_kernel_pred<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
        else if (op == AGG_MAX) matrix_group_max_kernel_pred<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
        else                    matrix_group_count_kernel_pred<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    } else {
        if (op == AGG_SUM)      matrix_group_sum_kernel<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, d_out);
        else if (op == AGG_MIN) matrix_group_min_kernel<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, d_out);
        else if (op == AGG_MAX) matrix_group_max_kernel<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, d_out);
        else                    matrix_group_count_kernel<<<BLOCKS, TPB>>>(dk, n, num_groups, d_out);   // unfiltered COUNT: keys only
    }
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(out_host, d_out, (size_t)num_groups * sizeof(unsigned long long), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
}

// === Typed (int64/double) grouped kernels — GROUP BY a u32 key over an i64/f64 value, predicate-gated on
// the VALUE (unfiltered = an all-match predicate synthesized by the wrapper). Per-group atomics reuse the
// verified GPU-5 typed atomics; out[] carries the same int64/double bit-pattern per group as the CPU
// matrix_cpu_group_reduce_{i64,f64}. ===
__global__ void matrix_group_count_kernel_i64(const uint32_t* keys, const long long* vals, size_t n, uint32_t num_groups, MatrixCmp c, long long a, long long b, long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_i64_dev(vals[i], c, a, b)) atomicAdd((unsigned long long*)&out[k], 1ull);
    }
}
__global__ void matrix_group_sum_kernel_i64(const uint32_t* keys, const long long* vals, size_t n, uint32_t num_groups, MatrixCmp c, long long a, long long b, long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_i64_dev(vals[i], c, a, b)) atomicAdd((unsigned long long*)&out[k], (unsigned long long)vals[i]);
    }
}
__global__ void matrix_group_min_kernel_i64(const uint32_t* keys, const long long* vals, size_t n, uint32_t num_groups, MatrixCmp c, long long a, long long b, long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_i64_dev(vals[i], c, a, b)) atomicMin(&out[k], vals[i]);
    }
}
__global__ void matrix_group_max_kernel_i64(const uint32_t* keys, const long long* vals, size_t n, uint32_t num_groups, MatrixCmp c, long long a, long long b, long long* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_i64_dev(vals[i], c, a, b)) atomicMax(&out[k], vals[i]);
    }
}
__global__ void matrix_group_count_kernel_f64(const uint32_t* keys, const double* vals, size_t n, uint32_t num_groups, MatrixCmp c, double a, double b, double* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_f64_dev(vals[i], c, a, b)) atomicAdd(&out[k], 1.0);
    }
}
__global__ void matrix_group_sum_kernel_f64(const uint32_t* keys, const double* vals, size_t n, uint32_t num_groups, MatrixCmp c, double a, double b, double* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_f64_dev(vals[i], c, a, b)) atomicAdd(&out[k], vals[i]);
    }
}
__global__ void matrix_group_min_kernel_f64(const uint32_t* keys, const double* vals, size_t n, uint32_t num_groups, MatrixCmp c, double a, double b, double* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_f64_dev(vals[i], c, a, b)) atomicMinDouble(&out[k], vals[i]);
    }
}
__global__ void matrix_group_max_kernel_f64(const uint32_t* keys, const double* vals, size_t n, uint32_t num_groups, MatrixCmp c, double a, double b, double* out) {
    const size_t stride = (size_t)gridDim.x * blockDim.x;
    for (size_t i = (size_t)blockIdx.x*blockDim.x+threadIdx.x; i < n; i += stride) {
        const uint32_t k = keys[i]; if (k < num_groups && matrix_pred_match_f64_dev(vals[i], c, a, b)) atomicMaxDouble(&out[k], vals[i]);
    }
}

// Typed grouped wrappers: same per-group encoding as matrix_cpu_group_reduce_{i64,f64} (int64/double bits).
// Sentinels (MIN INT64_MAX/+inf, MAX INT64_MIN/-inf, SUM/COUNT 0) aren't byte-fillable, so init via memcpy.
// Unfiltered synthesizes an all-match predicate (GE min) to mirror the unfiltered CPU reducer.
inline void matrix_gpu_group_reduce_dev_i64(const void* keys_dev, const void* vals_dev, size_t n,
                                            uint32_t num_groups, MatrixAggOp op, MatrixPredicateI64 pred,
                                            bool has_filter, uint64_t* out_host) {
    const uint32_t*  dk = reinterpret_cast<const uint32_t*>(keys_dev);
    const long long* dv = reinterpret_cast<const long long*>(vals_dev);
    long long* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, (size_t)num_groups * sizeof(long long)));
    std::vector<long long> init(num_groups, (op == AGG_MIN) ? INT64_MAX : (op == AGG_MAX ? INT64_MIN : 0));
    CUDA_CHECK(cudaMemcpy(d_out, init.data(), (size_t)num_groups * sizeof(long long), cudaMemcpyHostToDevice));
    const MatrixCmp  c = has_filter ? pred.cmp : MatrixCmp::GE;
    const long long  a = has_filter ? pred.a   : INT64_MIN;
    const long long  b = has_filter ? pred.b   : 0;
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_group_sum_kernel_i64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_group_min_kernel_i64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_group_max_kernel_i64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else                    matrix_group_count_kernel_i64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(out_host, d_out, (size_t)num_groups * sizeof(long long), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
}
inline void matrix_gpu_group_reduce_dev_f64(const void* keys_dev, const void* vals_dev, size_t n,
                                            uint32_t num_groups, MatrixAggOp op, MatrixPredicateF64 pred,
                                            bool has_filter, uint64_t* out_host) {
    const uint32_t* dk = reinterpret_cast<const uint32_t*>(keys_dev);
    const double*   dv = reinterpret_cast<const double*>(vals_dev);
    const double inf = std::numeric_limits<double>::infinity();
    double* d_out = nullptr; CUDA_CHECK(cudaMalloc(&d_out, (size_t)num_groups * sizeof(double)));
    std::vector<double> init(num_groups, (op == AGG_MIN) ? inf : (op == AGG_MAX ? -inf : 0.0));
    CUDA_CHECK(cudaMemcpy(d_out, init.data(), (size_t)num_groups * sizeof(double), cudaMemcpyHostToDevice));
    const MatrixCmp c = has_filter ? pred.cmp : MatrixCmp::GE;
    const double    a = has_filter ? pred.a   : -inf;
    const double    b = has_filter ? pred.b   : 0.0;
    constexpr int TPB = 256, BLOCKS = 1024;
    if (op == AGG_SUM)      matrix_group_sum_kernel_f64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else if (op == AGG_MIN) matrix_group_min_kernel_f64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else if (op == AGG_MAX) matrix_group_max_kernel_f64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    else                    matrix_group_count_kernel_f64<<<BLOCKS, TPB>>>(dk, dv, n, num_groups, c, a, b, d_out);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(out_host, d_out, (size_t)num_groups * sizeof(double), cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_out));
}

/**
 * @brief Real CUDA GPU engine, page-ownership model. Device-resident store persists
 * across batches (it is the database). Same ComputeInterface + correctness contract
 * as CPUMockEngine.
 */
class CUDAGPUEngine : public ComputeInterface {
public:
    explicit CUDAGPUEngine(size_t /*worker_count*/ = 0)
        : h_binned_(MATRIX_BATCH_MAX) {
        int device_count = 0;
        CUDA_CHECK(cudaGetDeviceCount(&device_count));
        if (device_count == 0) {
            std::fprintf(stderr, "CUDAGPUEngine: no CUDA device found.\n");
            std::abort();
        }
        cudaDeviceProp prop{};
        CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));

        CUDA_CHECK(cudaMalloc(&d_store_, MATRIX_STORE_SLOTS * sizeof(uint64_t)));
        CUDA_CHECK(cudaMemset(d_store_, 0, MATRIX_STORE_SLOTS * sizeof(uint64_t)));
        CUDA_CHECK(cudaMalloc(&d_binned_, MATRIX_BATCH_MAX * sizeof(DatabaseQuery)));
        CUDA_CHECK(cudaMalloc(&d_offsets_, (MATRIX_PAGE_COUNT + 1) * sizeof(uint32_t)));
        CUDA_CHECK(cudaMalloc(&d_reads_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMalloc(&d_writes_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_reads_, 0, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_writes_, 0, sizeof(unsigned long long)));

        // Resident analytical column (value[i]=i), filled once, scanned in place by
        // OP_SCAN. This is the GPU-DB's actual data — never shipped per query.
        CUDA_CHECK(cudaMalloc(&d_scan_col_, MATRIX_SCAN_COLUMN_SIZE * sizeof(uint32_t)));
        {
            std::vector<uint32_t> h(MATRIX_SCAN_COLUMN_SIZE);
            for (size_t i = 0; i < MATRIX_SCAN_COLUMN_SIZE; ++i) h[i] = static_cast<uint32_t>(i);
            CUDA_CHECK(cudaMemcpy(d_scan_col_, h.data(),
                                  MATRIX_SCAN_COLUMN_SIZE * sizeof(uint32_t),
                                  cudaMemcpyHostToDevice));
        }
        CUDA_CHECK(cudaMalloc(&d_scan_count_, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_scan_count_, 0, sizeof(unsigned long long)));

        offsets_.resize(MATRIX_PAGE_COUNT + 1);

        // ponytail: single stream. Hyper-Q multi-stream is the throughput upgrade.
        CUDA_CHECK(cudaStreamCreate(&stream_));
        // Dedicated stream for scans so they overlap point-op work and don't serialize
        // behind it (the HTAP separation, GPU side).
        CUDA_CHECK(cudaStreamCreate(&scan_stream_));
        // Reused events to time just the scan kernel (per-scan create/destroy would
        // itself add overhead and pollute the measurement).
        CUDA_CHECK(cudaEventCreate(&scan_k0_));
        CUDA_CHECK(cudaEventCreate(&scan_k1_));
        std::printf("CUDAGPUEngine initialized on '%s' (%d SMs, page-ownership, %zu pages).\n",
                    prop.name, prop.multiProcessorCount, MATRIX_PAGE_COUNT);
    }

    ~CUDAGPUEngine() override {
        cudaFree(d_store_);
        cudaFree(d_binned_);
        cudaFree(d_offsets_);
        cudaFree(d_reads_);
        cudaFree(d_writes_);
        cudaFree(d_scan_col_);
        cudaFree(d_scan_count_);
        cudaEventDestroy(scan_k0_);
        cudaEventDestroy(scan_k1_);
        cudaStreamDestroy(stream_);
        cudaStreamDestroy(scan_stream_);
        std::printf("CUDAGPUEngine released device resources.\n");
    }

    void execute_batch(DatabaseQuery* batch_array, size_t count) override {
        if (count == 0) return;
        if (count > MATRIX_BATCH_MAX) count = MATRIX_BATCH_MAX;

        // Bin by page on the host (folds into the dual-trigger batcher later).
        matrix_bin_by_page(batch_array, count, h_binned_.data(), offsets_.data());

        CUDA_CHECK(cudaMemcpyAsync(d_binned_, h_binned_.data(), count * sizeof(DatabaseQuery),
                                   cudaMemcpyHostToDevice, stream_));
        CUDA_CHECK(cudaMemcpyAsync(d_offsets_, offsets_.data(),
                                   (MATRIX_PAGE_COUNT + 1) * sizeof(uint32_t),
                                   cudaMemcpyHostToDevice, stream_));

        // One block per page; 128 threads/block stride over the page's queries.
        // Point ops only — scans arrive via execute_scan on their own stream.
        constexpr int TPB = 128;
        matrix_page_kernel<<<MATRIX_PAGE_COUNT, TPB, 0, stream_>>>(
            d_binned_, d_offsets_, d_store_, d_reads_, d_writes_);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaStreamSynchronize(stream_));
    }

    void execute_scan(DatabaseQuery& q) override {
        // u32x4 filter-count over the resident column, on the dedicated scan stream so
        // it overlaps point-op submission on stream_ and never head-of-line-blocks them.
        // AGG-2 (GPU-1): dispatch on matrix_get_scan_agg_op(q). COUNT keeps the byte-identical u32x4
        // path (the 83886070 oracle); SUM/MIN/MAX init `out` to the matching CPU sentinel then launch the
        // scalar reduction kernel. Each is verified GPU==matrix_cpu_reduce by test_gpu_agg.cu.
        const auto st0 = std::chrono::steady_clock::now();
        const uint32_t threshold = matrix_get_scan_threshold(q);
        const MatrixAggOp op = matrix_get_scan_agg_op(q);
        constexpr int SCAN_TPB = 256;
        const int blocks = 1024;
        CUDA_CHECK(cudaEventRecord(scan_k0_, scan_stream_));
        if (op == AGG_SUM) {
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0, sizeof(unsigned long long), scan_stream_));  // SUM sentinel 0
            matrix_sum_kernel_u32<<<blocks, SCAN_TPB, 0, scan_stream_>>>(d_scan_col_, MATRIX_SCAN_COLUMN_SIZE, threshold, d_scan_count_);
        } else if (op == AGG_MIN) {
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0xFF, sizeof(unsigned long long), scan_stream_)); // MIN sentinel UINT64_MAX
            matrix_min_kernel_u32<<<blocks, SCAN_TPB, 0, scan_stream_>>>(d_scan_col_, MATRIX_SCAN_COLUMN_SIZE, threshold, d_scan_count_);
        } else if (op == AGG_MAX) {
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0, sizeof(unsigned long long), scan_stream_));    // MAX sentinel 0
            matrix_max_kernel_u32<<<blocks, SCAN_TPB, 0, scan_stream_>>>(d_scan_col_, MATRIX_SCAN_COLUMN_SIZE, threshold, d_scan_count_);
        } else {                                                                                       // AGG_COUNT (default) — unchanged
            CUDA_CHECK(cudaMemsetAsync(d_scan_count_, 0, sizeof(unsigned long long), scan_stream_));
            const uint4* col4 = reinterpret_cast<const uint4*>(d_scan_col_);
            matrix_scan_kernel_u32x4<<<blocks, SCAN_TPB, 0, scan_stream_>>>(
                col4, MATRIX_SCAN_COLUMN_SIZE / 4, threshold, d_scan_count_);
        }
        CUDA_CHECK(cudaEventRecord(scan_k1_, scan_stream_));
        CUDA_CHECK(cudaGetLastError());
        unsigned long long c = 0;
        CUDA_CHECK(cudaMemcpyAsync(&c, d_scan_count_, sizeof(c),
                                   cudaMemcpyDeviceToHost, scan_stream_));
        CUDA_CHECK(cudaStreamSynchronize(scan_stream_));
        scan_time_s_ += std::chrono::duration<double>(
            std::chrono::steady_clock::now() - st0).count();
        float k_ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&k_ms, scan_k0_, scan_k1_));
        scan_kernel_time_s_ += k_ms / 1e3;
        q.transaction_id = c;
        ++scans_;
        scan_result_sum_ += c;
    }

    uint64_t reads() const override { return read_counter(d_reads_); }
    uint64_t writes() const override { return read_counter(d_writes_); }
    uint64_t commits() const override { return read_counter(d_writes_); } // every write commits (owned slot)
    uint64_t scans() const override { return scans_; }
    uint64_t scan_result_sum() const override { return scan_result_sum_; }
    double scan_time_s() const override { return scan_time_s_; }
    double scan_kernel_time_s() const override { return scan_kernel_time_s_; }

    uint64_t store_checksum() const override {
        // ponytail: copy the whole store back (32KB) and sum on host. Once, off the
        // hot path — a device reduction would be premature for a correctness check.
        std::vector<uint64_t> h(MATRIX_STORE_SLOTS);
        CUDA_CHECK(cudaMemcpy(h.data(), d_store_, MATRIX_STORE_SLOTS * sizeof(uint64_t),
                              cudaMemcpyDeviceToHost));
        uint64_t sum = 0;
        for (uint64_t v : h) sum += v;
        return sum;
    }

    double benchmark_scan(size_t n, uint64_t threshold, uint64_t& out_count) override {
        return timed_scan<uint64_t>(n, threshold, out_count,
            [](const uint64_t* d, size_t nn, uint64_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel<<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

    double benchmark_scan_u32(size_t n, uint32_t threshold, uint64_t& out_count) override {
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel_u32<<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

    double benchmark_scan_u32x4(size_t n, uint32_t threshold, uint64_t& out_count) override {
        // Same resident-column setup, but the kernel reads 4 u32 per uint4 load.
        // n is a power of two (>= 65536) so n % 4 == 0; cudaMalloc is 256-byte aligned.
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                const uint4* d4 = reinterpret_cast<const uint4*>(d);
                matrix_scan_kernel_u32x4<<<blocks, tpb, 0, s>>>(d4, nn / 4, thr, c);
            });
    }

    double benchmark_scan_ipt(size_t n, uint32_t threshold, uint64_t& out_count) override {
        return timed_scan<uint32_t>(n, threshold, out_count,
            [](const uint32_t* d, size_t nn, uint32_t thr, unsigned long long* c,
               int blocks, int tpb, cudaStream_t s) {
                matrix_scan_kernel_ipt<8><<<blocks, tpb, 0, s>>>(d, nn, thr, c);
            });
    }

private:
    // Shared scan harness: alloc resident column, fill (untimed), time kernel via CUDA
    // events, return seconds. Templated on column type so u32/u64 share one code path.
    template <typename T, typename Launch>
    double timed_scan(size_t n, T threshold, uint64_t& out_count, Launch launch) {
        T* d_data = nullptr;
        unsigned long long* d_count = nullptr;
        CUDA_CHECK(cudaMalloc(&d_data, n * sizeof(T)));
        CUDA_CHECK(cudaMalloc(&d_count, sizeof(unsigned long long)));
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(unsigned long long)));
        {
            std::vector<T> h(n);
            for (size_t i = 0; i < n; ++i) h[i] = static_cast<T>(i);
            CUDA_CHECK(cudaMemcpy(d_data, h.data(), n * sizeof(T), cudaMemcpyHostToDevice));
        }

        constexpr int TPB = 256;
        const int blocks = 1024; // saturate the 40 SMs; grid-stride handles any n
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));
        CUDA_CHECK(cudaEventRecord(start, stream_));
        launch(d_data, n, threshold, d_count, blocks, TPB, stream_);
        CUDA_CHECK(cudaEventRecord(stop, stream_));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float ms = 0.0f;
        CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));

        unsigned long long h_count = 0;
        CUDA_CHECK(cudaMemcpy(&h_count, d_count, sizeof(h_count), cudaMemcpyDeviceToHost));
        out_count = h_count;

        cudaEventDestroy(start);
        cudaEventDestroy(stop);
        cudaFree(d_data);
        cudaFree(d_count);
        return ms / 1e3; // seconds
    }

    static uint64_t read_counter(const unsigned long long* d_ptr) {
        unsigned long long h = 0;
        CUDA_CHECK(cudaMemcpy(&h, d_ptr, sizeof(h), cudaMemcpyDeviceToHost));
        return static_cast<uint64_t>(h);
    }

    uint64_t* d_store_ = nullptr;
    DatabaseQuery* d_binned_ = nullptr;
    uint32_t* d_offsets_ = nullptr;
    unsigned long long* d_reads_ = nullptr;
    unsigned long long* d_writes_ = nullptr;
    uint32_t* d_scan_col_ = nullptr;            // resident analytical column
    unsigned long long* d_scan_count_ = nullptr; // scratch for one scan's count
    uint64_t scans_ = 0;                          // host-side scan accounting
    uint64_t scan_result_sum_ = 0;
    double scan_time_s_ = 0.0;
    double scan_kernel_time_s_ = 0.0;            // cudaEvent-measured pure kernel time
    cudaEvent_t scan_k0_{}, scan_k1_{};          // reused scan-kernel timing events
    std::vector<DatabaseQuery> h_binned_;
    std::vector<uint32_t> offsets_;
    cudaStream_t stream_{};
    cudaStream_t scan_stream_{};                  // dedicated scan path (HTAP)
};


In [ ]:
%%writefile spike_server.cpp
// spike/spike_server.cpp — throwaway network+GPU round-trip spike server (NOT production code).
// Answers one question: does a real TCP+serialization hop still leave the GPU's measured
// in-process scan win (FINDINGS.md 3.5, ~12-19x) multi-fold once a client talks to it over the
// network? Serves ONE connection at a time, reusing matrixdbd's own framing helpers
// (matrixsrv_detail::recv_all/send_all) but a minimal opcode protocol instead of the full
// MatrixRequest wire format — this file is never wired into matrixdbd or any tested path.
//
// Wire format:
//   request:  [u32 len][payload]   len==0 -> HEALTH (server echoes an empty response)
//                                  len==5 -> SCAN: payload[0]=opcode(1), payload[1..5)=u32 threshold (LE)
//   response: [u32 len][payload]   HEALTH -> len==0
//                                  SCAN    -> len==16: [0..8)=double seconds (LE), [8..16)=uint64 count (LE)
//
// Build (CPU):  clang++ -std=c++20 -O2 spike_server.cpp -o spike_server_cpu
// Build (GPU):  nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA \
//               spike_server.cpp -o spike_server_gpu
#include "server_tcp.hpp"      // matrixsrv_detail::recv_all/send_all, matrix_set_recv_timeout/send_timeout
#if defined(MATRIX_USE_CUDA)
    #include "compute_cuda.cuh"
#endif
#include <chrono>
#include <csignal>
#include <cstdlib>
#include <cstring>
#include <iostream>
#include <memory>
#include <netinet/tcp.h>        // TCP_NODELAY

int main(int argc, char** argv) {
    if (argc < 2) { std::cerr << "usage: spike_server <port>\n"; return 2; }
    const uint16_t port = static_cast<uint16_t>(std::strtoul(argv[1], nullptr, 10));
    std::signal(SIGPIPE, SIG_IGN);   // a peer hanging up mid-send must not kill the server

#if defined(MATRIX_USE_CUDA)
    std::unique_ptr<ComputeInterface> engine = std::make_unique<CUDAGPUEngine>(4);
    std::cerr << "spike_server: CUDA engine, port " << port << "\n";
#else
    std::unique_ptr<ComputeInterface> engine = std::make_unique<CPUMockEngine>(4);
    std::cerr << "spike_server: CPU engine, port " << port << "\n";
#endif

    const int srv = ::socket(AF_INET, SOCK_STREAM, 0);
    if (srv < 0) { std::cerr << "spike_server: socket() failed\n"; return 1; }
    int yes = 1; ::setsockopt(srv, SOL_SOCKET, SO_REUSEADDR, &yes, sizeof yes);
    sockaddr_in addr{}; addr.sin_family = AF_INET; addr.sin_addr.s_addr = INADDR_ANY; addr.sin_port = htons(port);
    if (::bind(srv, reinterpret_cast<sockaddr*>(&addr), sizeof addr) != 0) {
        std::cerr << "spike_server: bind() failed (note: blocked in a sandboxed build env — run on a real host)\n";
        return 1;
    }
    if (::listen(srv, 16) != 0) { std::cerr << "spike_server: listen() failed\n"; return 1; }
    std::cerr << "spike_server: listening\n";

    for (;;) {
        const int fd = ::accept(srv, nullptr, nullptr);
        if (fd < 0) continue;
        matrix_set_recv_timeout(fd, 30000);
        matrix_set_send_timeout(fd, 30000);
        // Every SCAN response is two send_all() calls (length prefix, then payload). Without
        // TCP_NODELAY here, Nagle holds the second send waiting to coalesce, and the client's
        // delayed-ACK timer (~40ms) is what releases it — a stall that dwarfs the actual scan
        // time and drowns the CPU-vs-GPU signal this spike exists to measure.
        int nodelay = 1; ::setsockopt(fd, IPPROTO_TCP, TCP_NODELAY, &nodelay, sizeof nodelay);
        for (;;) {
            uint32_t len = 0;
            if (!matrixsrv_detail::recv_all(fd, &len, sizeof len)) break;
            if (len == 0) {
                const uint32_t rlen = 0;
                if (!matrixsrv_detail::send_all(fd, &rlen, sizeof rlen)) break;
                continue;
            }
            if (len != 5) break;   // only a 5-byte SCAN request is supported
            uint8_t req[5];
            if (!matrixsrv_detail::recv_all(fd, req, sizeof req)) break;
            if (req[0] != 1) break;   // opcode 1 == SCAN
            uint32_t threshold;
            std::memcpy(&threshold, req + 1, sizeof threshold);

            DatabaseQuery q{};
            matrix_set_scan_threshold(q, threshold);
            const auto t0 = std::chrono::steady_clock::now();
            engine->execute_scan(q);
            const double seconds = std::chrono::duration<double>(std::chrono::steady_clock::now() - t0).count();
            const uint64_t count = q.transaction_id;

            uint8_t resp[16];
            std::memcpy(resp, &seconds, sizeof seconds);
            std::memcpy(resp + 8, &count, sizeof count);
            const uint32_t rlen = sizeof resp;
            if (!matrixsrv_detail::send_all(fd, &rlen, sizeof rlen)) break;
            if (!matrixsrv_detail::send_all(fd, resp, sizeof resp)) break;
        }
        ::close(fd);
    }
}


In [ ]:
%%writefile SpikeClient.java
// spike/SpikeClient.java — throwaway Java client for the network+GPU round-trip spike.
// Speaks the minimal wire protocol implemented by spike_server.cpp: length-prefixed frames,
// a zero-length HEALTH-equivalent request, and a 5-byte SCAN request (opcode=1, u32 threshold).
// NOT production code — no Spring Boot, no dependencies, just java.net.Socket.
//
// Usage: java SpikeClient <host> <port> <iterations>
// Prints CSV to stdout: kind,iteration,round_trip_ns,server_seconds,count
import java.io.DataInputStream;
import java.io.DataOutputStream;
import java.io.IOException;
import java.net.Socket;
import java.nio.ByteBuffer;
import java.nio.ByteOrder;

public class SpikeClient {
    private static final int WARMUP = 5;
    private static final int SCAN_THRESHOLD = 8_388_608; // half of MATRIX_SCAN_COLUMN_SIZE (16,777,216)

    public static void main(String[] args) throws IOException {
        final String host = args.length > 0 ? args[0] : "127.0.0.1";
        final int port = args.length > 1 ? Integer.parseInt(args[1]) : 7070;
        final int iterations = args.length > 2 ? Integer.parseInt(args[2]) : 50;

        try (Socket sock = new Socket(host, port)) {
            sock.setTcpNoDelay(true);
            final DataOutputStream out = new DataOutputStream(sock.getOutputStream());
            final DataInputStream in = new DataInputStream(sock.getInputStream());

            System.out.println("kind,iteration,round_trip_ns,server_seconds,count");

            for (int i = -WARMUP; i < iterations; i++) {
                final long t0 = System.nanoTime();
                writeFrame(out, new byte[0]);
                readFrame(in);
                final long t1 = System.nanoTime();
                if (i >= 0) System.out.println("health," + i + "," + (t1 - t0) + ",,");
            }

            final ByteBuffer req = ByteBuffer.allocate(5).order(ByteOrder.LITTLE_ENDIAN);
            req.put(0, (byte) 1);
            req.putInt(1, SCAN_THRESHOLD);
            for (int i = -WARMUP; i < iterations; i++) {
                final long t0 = System.nanoTime();
                writeFrame(out, req.array());
                final byte[] resp = readFrame(in);
                final long t1 = System.nanoTime();
                if (i >= 0) {
                    final ByteBuffer rb = ByteBuffer.wrap(resp).order(ByteOrder.LITTLE_ENDIAN);
                    final double serverSeconds = Double.longBitsToDouble(rb.getLong(0));
                    final long count = rb.getLong(8);
                    System.out.println("scan," + i + "," + (t1 - t0) + "," + serverSeconds + "," + count);
                }
            }
        }
    }

    private static void writeFrame(DataOutputStream out, byte[] payload) throws IOException {
        final ByteBuffer lenBuf = ByteBuffer.allocate(4).order(ByteOrder.LITTLE_ENDIAN).putInt(0, payload.length);
        out.write(lenBuf.array());
        out.write(payload);
        out.flush();
    }

    private static byte[] readFrame(DataInputStream in) throws IOException {
        final byte[] lenBytes = new byte[4];
        in.readFully(lenBytes);
        final int len = ByteBuffer.wrap(lenBytes).order(ByteOrder.LITTLE_ENDIAN).getInt();
        final byte[] payload = new byte[len];
        if (len > 0) in.readFully(payload);
        return payload;
    }
}


## 3. Install a JDK for the spike client

In [ ]:
!apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless

## 4. Build spike_server (CPU) and compile the Java client

In [ ]:
!clang++ -std=c++20 -O2 spike_server.cpp -o spike_server_cpu 2>/dev/null || g++ -std=c++20 -O2 spike_server.cpp -o spike_server_cpu

In [ ]:
!javac SpikeClient.java

## 5. CPU-via-network run

Starts spike_server_cpu in the background, runs the Java client sweep against it, then stops it. Output is CSV: `kind,iteration,round_trip_ns,server_seconds,count`.

In [ ]:
%%bash
./spike_server_cpu 7070 &
SERVER_PID=$!
sleep 1
java SpikeClient 127.0.0.1 7070 50 > cpu_results.csv
kill $SERVER_PID
wait 2>/dev/null
cat cpu_results.csv

## 6. Build spike_server (GPU, real CUDAGPUEngine)

In [ ]:
!nvcc -std=c++17 -O3 -x cu -D_GNU_SOURCE -Xcompiler -pthread -DMATRIX_USE_CUDA spike_server.cpp -o spike_server_gpu

## 7. GPU-via-network run

In [ ]:
%%bash
./spike_server_gpu 7071 &
SERVER_PID=$!
sleep 1
java SpikeClient 127.0.0.1 7071 50 > gpu_results.csv
kill $SERVER_PID
wait 2>/dev/null
cat gpu_results.csv

## 8. Compare against the decision rule

Medians of `round_trip_ns` per `kind`. `scan` = the actual query cost over the network; `health` = the fixed network+serialization tax (zero payload). Compare against `docs/superpowers/specs/2026-06-30-network-gpu-spike-design.md`'s PASS/FAIL rule: PASS needs via-network GPU speedup >= ~10x AND health/scan tax fraction < 10%.

Also reports `server_seconds` — the compute-only time each server timed around its own `execute_scan()` call, embedded in the response. It's immune to client-side JVM/OS scheduling noise, so comparing it to `round_trip_ns` tells apart two very different things: compute that's genuinely slower on this host, vs. round-trip inflation from something *around* the compute (client scheduling contention, GC, etc). Since `CPUMockEngine::execute_scan` is a single-core hot loop while `CUDAGPUEngine::execute_scan` mostly just waits on the GPU chip, a CPU-hungry JVM client sharing this box's vCPUs would contend directly with the CPU engine's scan but barely touch the GPU engine's — watch for that asymmetry here before trusting the speedup number.

In [ ]:
import csv, statistics

def medians(path):
    scan, health, scan_server = [], [], []
    with open(path) as fh:
        for row in csv.DictReader(fh):
            if row['kind'] == 'scan':
                scan.append(int(row['round_trip_ns']))
                scan_server.append(float(row['server_seconds']))
            elif row['kind'] == 'health':
                health.append(int(row['round_trip_ns']))
    return statistics.median(scan), statistics.median(health), statistics.median(scan_server)

cpu_scan_ns, cpu_health_ns, cpu_server_s = medians('cpu_results.csv')
gpu_scan_ns, gpu_health_ns, gpu_server_s = medians('gpu_results.csv')
cpu_server_ns, gpu_server_ns = cpu_server_s * 1e9, gpu_server_s * 1e9
print(f'CPU: scan round_trip median={cpu_scan_ns:.0f}ns  server_seconds median={cpu_server_ns:.0f}ns  health(fixed tax) median={cpu_health_ns:.0f}ns  tax_fraction={cpu_health_ns/cpu_scan_ns:.1%}')
print(f'GPU: scan round_trip median={gpu_scan_ns:.0f}ns  server_seconds median={gpu_server_ns:.0f}ns  health(fixed tax) median={gpu_health_ns:.0f}ns  tax_fraction={gpu_health_ns/gpu_scan_ns:.1%}')
print(f'via-network GPU speedup over CPU (round_trip_ns): {cpu_scan_ns / gpu_scan_ns:.1f}x')
print(f'compute-only GPU speedup over CPU (server_seconds): {cpu_server_ns / gpu_server_ns:.1f}x')
print(f'CPU round_trip - server_seconds gap (client/scheduling overhead, not network tax): {cpu_scan_ns - cpu_server_ns:.0f}ns')
print(f'GPU round_trip - server_seconds gap (client/scheduling overhead, not network tax): {gpu_scan_ns - gpu_server_ns:.0f}ns')